In [0]:
import sys
from datetime import datetime, timezone

print("Projeto: Oil & Gas Production Lakehouse — ANP")
print(f"Execução UTC: {datetime.now(timezone.utc).isoformat()}")
print(f"Python: {sys.version.split()[0]}")

display(
    spark.sql("""
        SELECT
            current_catalog() AS catalogo_atual,
            current_schema() AS schema_atual,
            current_timezone() AS fuso_da_sessao,
            version() AS versao_spark
    """)
)

display(spark.sql("SHOW CATALOGS"))

Projeto: Oil & Gas Production Lakehouse — ANP
Execução UTC: 2026-10-16T10:03:41.481131+00:00
Python: 3.12.3


catalogo_atual,schema_atual,fuso_da_sessao,versao_spark
workspace,default,Etc/UTC,4.2.0 0000000000000000000000000000000000000000


catalog
samples
system
workspace


In [0]:
import requests
from html.parser import HTMLParser
from urllib.parse import urlparse

ANP_SOURCE_URL = (
    "https://www.gov.br/anp/pt-br/centrais-de-conteudo/"
    "dados-abertos/fase-de-desenvolvimento-e-producao"
)

class PageTitleParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.inside_title = False
        self.parts = []

    def handle_starttag(self, tag, attrs):
        if tag == "title":
            self.inside_title = True

    def handle_endtag(self, tag):
        if tag == "title":
            self.inside_title = False

    def handle_data(self, data):
        if self.inside_title:
            self.parts.append(data)

response = requests.get(
    ANP_SOURCE_URL,
    timeout=(10, 30),
)
response.raise_for_status()

content_type = response.headers.get("Content-Type", "")
final_host = urlparse(response.url).hostname

if final_host != "www.gov.br":
    raise RuntimeError(f"Redirecionamento inesperado: {response.url}")

if "text/html" not in content_type.lower():
    raise RuntimeError(f"Conteúdo inesperado: {content_type}")

anp_source_html = response.text

title_parser = PageTitleParser()
title_parser.feed(anp_source_html)
page_title = " ".join("".join(title_parser.parts).split())

print(f"HTTP: {response.status_code}")
print(f"URL final: {response.url}")
print(f"Tipo de conteúdo: {content_type}")
print(f"Tamanho recebido: {len(response.content):,} bytes")
print(f"Título: {page_title}")

HTTP: 200
URL final: https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/fase-de-desenvolvimento-e-producao
Tipo de conteúdo: text/html;charset=utf-8
Tamanho recebido: 380,827 bytes
Título: Fase de Desenvolvimento e Produção — Agência Nacional do Petróleo, Gás Natural e Biocombustíveis


In [0]:
from html.parser import HTMLParser
from urllib.parse import urljoin, urlsplit, unquote
from pathlib import PurePosixPath

if not globals().get("anp_source_html"):
    raise RuntimeError("Execute a célula 2 antes desta célula.")

class FileLinkParser(HTMLParser):
    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.links = []
        self.current_link = None

    def handle_starttag(self, tag, attrs):
        attributes = dict(attrs)

        if tag == "a":
            href = attributes.get("href")
            self.current_link = (
                {"href": href, "text": []} if href else None
            )

        elif tag == "img" and self.current_link is not None:
            self.current_link["text"].append(
                attributes.get("alt", "")
            )

    def handle_data(self, data):
        if self.current_link is not None:
            self.current_link["text"].append(data)

    def handle_endtag(self, tag):
        if tag == "a" and self.current_link is not None:
            self.links.append(self.current_link)
            self.current_link = None

link_parser = FileLinkParser()
link_parser.feed(anp_source_html)

supported_extensions = {".csv", ".zip", ".xlsx", ".xls", ".pdf"}
source_inventory = []
seen_links = set()

for link in link_parser.links:
    absolute_url = urljoin(response.url, link["href"])
    parsed_url = urlsplit(absolute_url)

    if parsed_url.scheme not in {"http", "https"}:
        continue

    decoded_path = unquote(parsed_url.path)
    extension = PurePosixPath(decoded_path.rstrip("/")).suffix.lower()

    if extension not in supported_extensions:
        if decoded_path.rstrip("/").endswith(("/view", "/download")):
            parent_path = decoded_path.rstrip("/").rsplit("/", 1)[0]
            extension = PurePosixPath(parent_path).suffix.lower()

    if extension not in supported_extensions:
        continue

    label = " ".join(" ".join(link["text"]).split())
    unique_key = (absolute_url, label)

    if unique_key in seen_links:
        continue

    seen_links.add(unique_key)

    source_inventory.append({
        "id": len(source_inventory) + 1,
        "rotulo": label or "(sem rótulo)",
        "formato_indicado": extension.lstrip("."),
        "url": absolute_url,
    })

if not source_inventory:
    raise RuntimeError(
        "Nenhum link de arquivo identificado. "
        "Precisaremos inspecionar a estrutura da página."
    )

inventory_df = spark.createDataFrame(
    source_inventory,
    schema=(
        "id INT, rotulo STRING, "
        "formato_indicado STRING, url STRING"
    ),
)

print(f"Links candidatos encontrados: {len(source_inventory)}")
print("Os formatos foram inferidos pelas URLs, ainda sem validar os arquivos.")

display(
    inventory_df.groupBy("formato_indicado")
    .count()
    .orderBy("formato_indicado")
)

display(inventory_df.orderBy("id"))

Links candidatos encontrados: 307
Os formatos foram inferidos pelas URLs, ainda sem validar os arquivos.


formato_indicado,count
csv,302
pdf,4
xlsx,1


id,rotulo,formato_indicado,url
1,Relatório – Resolução CNPE no 5/2022,pdf,https://www.gov.br/anp/pt-br/assuntos/tecnologia-meio-ambiente/arquivos/relatorio-resolucao-cnpe05-2022.pdf
2,Agendas Antigas,xlsx,https://www.gov.br/anp/pt-br/acesso-a-informacao/institucional/agenda-dirigentes/agendadirigentesate2020.xlsx
3,Relatórios de Órgãos de Controle,pdf,https://www.gov.br/anp/pt-br/acesso-a-informacao/arquivos-lgpd/relatorio-tcu-lgpd.pdf
4,Termo de Uso e Política de Privacidade,pdf,https://www.gov.br/anp/pt-br/acesso-a-informacao/termo-uso-politica-privacidade-lgpd.pdf
5,Programa de Governança em Privacidade,pdf,https://www.gov.br/anp/pt-br/acesso-a-informacao/acoes-e-programas/governanca-privacidade.pdf
6,2026,csv,https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/arquivos-fase-de-desenvolvimento-e-producao/arquivos-producao-de-petroleo-e-gas-natural-nacional/pm/producao-mar-2026.csv
7,2025,csv,https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/arquivos-fase-de-desenvolvimento-e-producao/arquivos-producao-de-petroleo-e-gas-natural-nacional/pm/producao-mar-2025.csv
8,2024,csv,https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/arquivos-fase-de-desenvolvimento-e-producao/arquivos-producao-de-petroleo-e-gas-natural-nacional/pm/producao_por_poco_2024.csv
9,2023,csv,https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/arquivos-fase-de-desenvolvimento-e-producao/arquivos-producao-de-petroleo-e-gas-natural-nacional/pm/producao-mar-2023.csv
10,2022,csv,https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/arquivos-fase-de-desenvolvimento-e-producao/arquivos-producao-de-petroleo-e-gas-natural-nacional/pm/producao-mar-2022.csv


In [0]:
import unicodedata
from urllib.parse import urljoin, urlsplit

def normalize_search_text(value):
    return (
        unicodedata.normalize("NFKD", value)
        .encode("ascii", "ignore")
        .decode("ascii")
        .lower()
    )

if "link_parser" not in globals():
    raise RuntimeError("Execute a célula 3 antes desta célula.")

metadata_inventory = []
seen_metadata = set()

for link in link_parser.links:
    label = " ".join(" ".join(link["text"]).split())
    absolute_url = urljoin(ANP_SOURCE_URL, link["href"])
    parsed_url = urlsplit(absolute_url)

    if parsed_url.scheme not in {"http", "https"}:
        continue

    searchable_text = normalize_search_text(
        f"{label} {absolute_url}"
    )

    if "metadad" not in searchable_text:
        continue

    unique_key = (absolute_url, label)

    if unique_key in seen_metadata:
        continue

    seen_metadata.add(unique_key)

    metadata_inventory.append({
        "id": len(metadata_inventory) + 1,
        "rotulo": label or "(sem rótulo)",
        "url": absolute_url,
    })

if not metadata_inventory:
    raise RuntimeError(
        "Nenhum link de metadados localizado. "
        "Será necessário inspecionar a estrutura da página."
    )

metadata_inventory_df = spark.createDataFrame(
    metadata_inventory,
    schema="id INT, rotulo STRING, url STRING",
)

print(f"Links candidatos de metadados: {len(metadata_inventory)}")
display(metadata_inventory_df.orderBy("id"))

Links candidatos de metadados: 17


id,rotulo,url
1,Metadados - Terra,https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/arquivos-fase-de-desenvolvimento-e-producao/arquivos-producao-de-petroleo-e-gas-natural-nacional/metadados-terra.docx
2,Metadados - Mar,https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/arquivos-fase-de-desenvolvimento-e-producao/arquivos-producao-de-petroleo-e-gas-natural-nacional/metadados-mar.docx
3,Metadados,https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/arquivos-fase-de-desenvolvimento-e-producao/producao-por-zona/metadados-producao-zona.docx
4,Metadados,https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/arquivos-fase-de-desenvolvimento-e-producao/lpo/metadadosplataformasoperacao.docx
5,Metadados - campos,https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/arquivos-fase-de-desenvolvimento-e-producao/informacoes-sobre-campos/metadados-campos.docx
6,Metadados - campos devolvidos,https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/arquivos-fase-de-desenvolvimento-e-producao/informacoes-sobre-campos/metadados-campos-devolvidos.docx
7,Metadados - concessionários,https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/arquivos-fase-de-desenvolvimento-e-producao/informacoes-sobre-campos/metadados-campos-concessionarios.docx
8,Metadados - lista de coordenadas de vértices de áreas na fase de produção,https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/arquivos-fase-de-desenvolvimento-e-producao/informacoes-sobre-campos/metadadoscoordenadasvertices.docx
9,Metadados - Intervenções em poços,https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/arquivos-fase-de-desenvolvimento-e-producao/informacoes-sobre-pocos/metadado-intervencao-em-pocos.docx
10,Metadados - Situação de poços,https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/arquivos-fase-de-desenvolvimento-e-producao/informacoes-sobre-pocos/metadado-situacao-de-poco.docx


In [0]:
from io import BytesIO
from zipfile import ZipFile, is_zipfile
import xml.etree.ElementTree as ET
import hashlib
import requests

if "metadata_inventory" not in globals():
    raise RuntimeError("Execute a célula 4 antes desta célula.")

selected_metadata = {
    "producao_terra": "metadados-terra.docx",
    "producao_mar": "metadados-mar.docx",
    "intervencoes": "metadado-intervencao-em-pocos.docx",
    "situacao_pocos": "metadado-situacao-de-poco.docx",
}

WORD_NS = {"w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main"}

def extract_docx_text(raw_bytes):
    buffer = BytesIO(raw_bytes)

    if not is_zipfile(buffer):
        raise ValueError("O conteúdo recebido não é um DOCX válido.")

    with ZipFile(buffer) as document:
        xml_bytes = document.read("word/document.xml")

    root = ET.fromstring(xml_bytes)
    body = root.find("w:body", WORD_NS)

    if body is None:
        raise ValueError("Documento sem corpo de texto.")

    def element_text(element):
        return " ".join(
            "".join(element.itertext()).split()
        )

    lines = []

    for element in body:
        if element.tag == f"{{{WORD_NS['w']}}}p":
            text = element_text(element)
            if text:
                lines.append(text)

        elif element.tag == f"{{{WORD_NS['w']}}}tbl":
            for row in element.findall("w:tr", WORD_NS):
                cells = [
                    element_text(cell)
                    for cell in row.findall("w:tc", WORD_NS)
                ]
                lines.append(" | ".join(cells))

    return "\n".join(lines)

metadata_documents = {}

for dataset, filename in selected_metadata.items():
    matches = [
        item for item in metadata_inventory
        if urlsplit(item["url"]).path.endswith("/" + filename)
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"{dataset}: esperado um link, encontrados {len(matches)}."
        )

    source_url = matches[0]["url"]

    with requests.get(source_url, timeout=(10, 30)) as document_response:
        document_response.raise_for_status()
        raw_bytes = document_response.content

    extracted_text = extract_docx_text(raw_bytes)

    if not extracted_text.strip():
        raise RuntimeError(f"{dataset}: nenhum texto extraído.")

    metadata_documents[dataset] = {
        "source_url": source_url,
        "sha256": hashlib.sha256(raw_bytes).hexdigest(),
        "raw_bytes": raw_bytes,
        "text": extracted_text,
    }

    print(f"\n{'=' * 70}")
    print(f"CONJUNTO: {dataset}")
    print(f"URL: {source_url}")
    print(f"Tamanho: {len(raw_bytes):,} bytes")
    print(f"SHA-256: {metadata_documents[dataset]['sha256']}")
    print(f"{'=' * 70}\n")
    print(extracted_text)

print(f"\nDocumentos lidos: {len(metadata_documents)}")


CONJUNTO: producao_terra
URL: https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/arquivos-fase-de-desenvolvimento-e-producao/arquivos-producao-de-petroleo-e-gas-natural-nacional/metadados-terra.docx
Tamanho: 27,602 bytes
SHA-256: 57bbc8d05c3f147fb491c584c2f7824e7bacff64917c418e053b2eb4a790f529

METADADOS: Produção poço - TERRA
SEÇÃO 1: INFORMAÇÃO
CAMPO | VALOR
TÍTULO | Produção poço – TERRA.
DESCRIÇÃO | Relação de poços produtores terrestres no território nacional.
ORIGEM DO DADO | https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/fase-de-desenvolvimento-e-producao
FONTE DO DADO | ANP/SDP (Superintendência de Desenvolvimento e Produção)
FORMATO DO ARQUIVO | CSV
PERIODICIDADE DE ATUALIZAÇÃO | Mensal
NOTAS | Os dados que compõem este conjunto de informações são extraídos dos sistemas Sistema de Informações Gerenciais de Exploração e Produção (SIGEP) e Do Poço ao Posto (DPP). Os relatórios são elaborados por meio da ferramenta Microsoft Power BI.
CONTA

In [0]:
import csv
import json
from io import StringIO
from itertools import islice
from urllib.parse import urlsplit

if "source_inventory" not in globals():
    raise RuntimeError("Execute a célula 3 antes desta célula.")

pilot_files = {
    "producao_mar": "producao-mar-2025.csv",
    "producao_terra": "producao_por_poco_terra_2025_1_trim.csv",
    "intervencoes": "intervencao_em_pocos_2025-2026.csv",
    "situacao_pocos": "situacao-pocos-1939-2026.csv",
}

SAMPLE_LIMIT_BYTES = 128 * 1024
csv_samples = {}

for dataset, filename in pilot_files.items():
    matches = [
        item for item in source_inventory
        if urlsplit(item["url"]).path.endswith("/" + filename)
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"{dataset}: esperado um link, encontrados {len(matches)}."
        )

    source_url = matches[0]["url"]
    chunks = []
    received_bytes = 0

    with requests.get(
        source_url,
        stream=True,
        timeout=(10, 30),
    ) as csv_response:
        csv_response.raise_for_status()

        content_type = csv_response.headers.get("Content-Type", "")
        declared_size = csv_response.headers.get("Content-Length")

        if "text/html" in content_type.lower():
            raise RuntimeError(
                f"{dataset}: recebemos HTML em vez do CSV."
            )

        for chunk in csv_response.iter_content(chunk_size=16 * 1024):
            if not chunk:
                continue

            remaining = SAMPLE_LIMIT_BYTES - received_bytes
            selected_chunk = chunk[:remaining]
            chunks.append(selected_chunk)
            received_bytes += len(selected_chunk)

            if received_bytes >= SAMPLE_LIMIT_BYTES:
                break

    raw_sample = b"".join(chunks)


    last_newline = raw_sample.rfind(b"\n")

    if last_newline < 0:
        raise RuntimeError(f"{dataset}: amostra sem quebra de linha.")

    parseable_sample = raw_sample[:last_newline + 1]

    try:
        sample_text = parseable_sample.decode("utf-8-sig")
        encoding_used = "utf-8-sig"
    except UnicodeDecodeError:
        sample_text = parseable_sample.decode("cp1252")
        encoding_used = "cp1252 — candidato, a confirmar"

    if sample_text.lstrip().lower().startswith(("<!doctype", "<html")):
        raise RuntimeError(f"{dataset}: conteúdo aparenta ser HTML.")

    dialect = csv.Sniffer().sniff(
        sample_text[:32_768],
        delimiters=";,\t|",
    )

    reader = csv.reader(
        StringIO(sample_text, newline=""),
        dialect=dialect,
        strict=True,
    )

    header = next(reader)
    sample_rows = list(islice(reader, 3))

    if len(header) < 2 or not sample_rows:
        raise RuntimeError(f"{dataset}: estrutura insuficiente.")

    if any(len(row) != len(header) for row in sample_rows):
        raise RuntimeError(
            f"{dataset}: amostra com quantidade de campos inconsistente."
        )

    csv_samples[dataset] = {
        "source_url": source_url,
        "raw_sample": raw_sample,
        "encoding_candidate": encoding_used,
        "delimiter": dialect.delimiter,
        "header": header,
        "rows": sample_rows,
    }

    print(f"\n{'=' * 70}")
    print(f"CONJUNTO: {dataset}")
    print(f"URL: {source_url}")
    print(f"Content-Type: {content_type}")
    print(f"Tamanho declarado pelo servidor: {declared_size}")
    print(f"Bytes lidos nesta amostra: {received_bytes:,}")
    print(f"Codificação: {encoding_used}")
    print(f"Separador detectado: {dialect.delimiter!r}")
    print(f"Quantidade de colunas: {len(header)}")

    print("\nCOLUNAS:")
    for position, column in enumerate(header, start=1):
        print(f"{position:02d}. {column!r}")

    print("\nPRIMEIROS REGISTROS — mesma ordem das colunas:")
    for row in sample_rows:
        print(json.dumps(row, ensure_ascii=False))

print(f"\nAmostras inspecionadas: {len(csv_samples)}")


CONJUNTO: producao_mar
URL: https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/arquivos-fase-de-desenvolvimento-e-producao/arquivos-producao-de-petroleo-e-gas-natural-nacional/pm/producao-mar-2025.csv
Content-Type: text/csv
Tamanho declarado pelo servidor: 1996681
Bytes lidos nesta amostra: 131,072
Codificação: utf-8-sig
Separador detectado: ','
Quantidade de colunas: 21

COLUNAS:
01. '[Ano]'
02. '[Mês/Ano]'
03. '[Estado]'
04. '[Bacia]'
05. '[Campo]'
06. '[Poço]'
07. '[Ambiente]'
08. '[Instalação]'
09. '[Produção de Óleo (m³)]'
10. '[Produção de Condensado (m³)]'
11. '[Produção de Gás Associado (Mm³)]'
12. '[Produção de Gás Não Associado (Mm³)]'
13. '[Produção de Água (m³)]'
14. '[Injeção de Gás (Mm³)]'
15. '[Injeção de Água para Recuperação Secundária (m³)]'
16. '[Injeção de Água para Descarte (m³)]'
17. '[Injeção de Gás Carbônico (Mm³)]'
18. '[Injeção de Nitrogênio (Mm³)]'
19. '[Injeção de Vapor de Água (t)]'
20. '[Injeção de Polímeros (m³)]'
21. '[Injeção de O

In [0]:
PROJECT_CATALOG = "workspace"

PROJECT_SCHEMAS = {
    "bronze": "oil_gas_bronze",
    "silver": "oil_gas_silver",
    "gold": "oil_gas_gold",
    "ops": "oil_gas_ops",
}

schema_comments = {
    "bronze": "Dados ingeridos e arquivos originais da ANP.",
    "silver": "Dados padronizados e validados do projeto.",
    "gold": "Data warehouse dimensional e indicadores analíticos.",
    "ops": "Controle de ingestão, auditoria e qualidade de dados.",
}

for layer, schema_name in PROJECT_SCHEMAS.items():
    spark.sql(f"""
        CREATE SCHEMA IF NOT EXISTS
            `{PROJECT_CATALOG}`.`{schema_name}`
        COMMENT '{schema_comments[layer]}'
    """)

    print(f"Schema disponível: {PROJECT_CATALOG}.{schema_name}")

RAW_VOLUME_NAME = (
    f"{PROJECT_CATALOG}.{PROJECT_SCHEMAS['bronze']}.raw_files"
)

spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS {RAW_VOLUME_NAME}
    COMMENT 'Arquivos originais e metadados coletados da ANP.'
""")

RAW_VOLUME_PATH = (
    f"/Volumes/{PROJECT_CATALOG}/"
    f"{PROJECT_SCHEMAS['bronze']}/raw_files"
)

print(f"\nVolume: {RAW_VOLUME_NAME}")
print(f"Caminho: {RAW_VOLUME_PATH}")

print("\nSchemas encontrados:")
display(
    spark.sql(f"""
        SHOW SCHEMAS IN `{PROJECT_CATALOG}`
        LIKE 'oil_gas_*'
    """)
)

print("\nVolumes encontrados na Bronze:")
display(
    spark.sql(f"""
        SHOW VOLUMES IN
            `{PROJECT_CATALOG}`.`{PROJECT_SCHEMAS['bronze']}`
    """)
)

Schema disponível: workspace.oil_gas_bronze
Schema disponível: workspace.oil_gas_silver
Schema disponível: workspace.oil_gas_gold
Schema disponível: workspace.oil_gas_ops

Volume: workspace.oil_gas_bronze.raw_files
Caminho: /Volumes/workspace/oil_gas_bronze/raw_files

Schemas encontrados:


databaseName
oil_gas_bronze
oil_gas_gold
oil_gas_ops
oil_gas_silver



Volumes encontrados na Bronze:


database,volume_name
oil_gas_bronze,raw_files


In [0]:
import os
import csv
import uuid
import hashlib
import requests

from io import BytesIO, StringIO
from datetime import datetime, timezone
from urllib.parse import urlsplit

required_datasets = [
    "producao_mar",
    "producao_terra",
    "intervencoes",
    "situacao_pocos",
]

if "RAW_VOLUME_PATH" not in globals():
    raise RuntimeError("Execute a célula 7 antes desta célula.")

if "csv_samples" not in globals():
    raise RuntimeError("Execute a célula 6 antes desta célula.")

for dataset in required_datasets:
    if dataset not in csv_samples:
        raise RuntimeError(f"Amostra ausente: {dataset}")

INGESTION_RUN_ID = str(uuid.uuid4())
MAX_FILE_BYTES = 32 * 1024 * 1024
AUDIT_TABLE = "workspace.oil_gas_ops.ingestion_files"

audit_schema = """
    run_id STRING,
    dataset STRING,
    source_url STRING,
    collected_at TIMESTAMP,
    file_path STRING,
    file_sha256 STRING,
    size_bytes LONG,
    status STRING,
    error_message STRING
"""

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {AUDIT_TABLE} (
        run_id STRING,
        dataset STRING,
        source_url STRING,
        collected_at TIMESTAMP,
        file_path STRING,
        file_sha256 STRING,
        size_bytes BIGINT,
        status STRING,
        error_message STRING
    )
    USING DELTA
    COMMENT 'Auditoria de coleta dos arquivos originais da ANP.'
""")

def calculate_stored_hash(file_path):
    digest = hashlib.sha256()

    with open(file_path, "rb") as stored_file:
        for block in iter(lambda: stored_file.read(1024 * 1024), b""):
            digest.update(block)

    return digest.hexdigest()

print(f"Execução: {INGESTION_RUN_ID}")

for dataset in required_datasets:
    sample = csv_samples[dataset]
    source_url = sample["source_url"]

    file_path = None
    file_hash = None
    size_bytes = None

    try:
        with requests.get(
            source_url,
            stream=True,
            timeout=(10, 60),
        ) as download_response:
            download_response.raise_for_status()

            final_host = urlsplit(download_response.url).hostname
            if final_host != "www.gov.br":
                raise RuntimeError("Redirecionamento fora de www.gov.br.")

            content_type = download_response.headers.get(
                "Content-Type", ""
            )
            if "text/html" in content_type.lower():
                raise RuntimeError("Recebido HTML em vez de CSV.")

            buffer = BytesIO()

            for chunk in download_response.iter_content(
                chunk_size=64 * 1024
            ):
                if not chunk:
                    continue

                if buffer.tell() + len(chunk) > MAX_FILE_BYTES:
                    raise RuntimeError(
                        "Arquivo excedeu o limite de 32 MiB desta coleta."
                    )

                buffer.write(chunk)

            raw_bytes = buffer.getvalue()

        collected_at = datetime.now(timezone.utc)
        size_bytes = len(raw_bytes)

        if size_bytes == 0:
            raise RuntimeError("Arquivo vazio.")

 
        text = raw_bytes.decode("utf-8-sig")

        if text.lstrip().lower().startswith(("<!doctype", "<html")):
            raise RuntimeError("Conteúdo aparenta ser HTML.")

        reader = csv.reader(
            StringIO(text, newline=""),
            delimiter=sample["delimiter"],
            strict=True,
        )
        header = next(reader, None)

        if header != sample["header"]:
            raise RuntimeError(
                "Cabeçalho diferente da amostra. Revisar o layout."
            )

        file_hash = hashlib.sha256(raw_bytes).hexdigest()
        filename = os.path.basename(urlsplit(source_url).path)

        destination_dir = (
            f"{RAW_VOLUME_PATH}/datasets/{dataset}/"
            f"sha256={file_hash}"
        )
        os.makedirs(destination_dir, exist_ok=True)

        file_path = f"{destination_dir}/{filename}"

        if os.path.exists(file_path):
            if calculate_stored_hash(file_path) != file_hash:
                raise RuntimeError(
                    "Arquivo existente com hash incompatível."
                )
            status = "REUSED"

        else:
            with open(file_path, "xb") as destination:
                destination.write(raw_bytes)

            if calculate_stored_hash(file_path) != file_hash:
                raise RuntimeError(
                    "Falha na verificação do arquivo gravado."
                )
            status = "STORED"

    except Exception as error:
        failure_record = [(
            INGESTION_RUN_ID,
            dataset,
            source_url,
            datetime.now(timezone.utc),
            file_path,
            file_hash,
            size_bytes,
            "FAILED",
            str(error)[:2000],
        )]

        spark.createDataFrame(
            failure_record, schema=audit_schema
        ).write.format("delta").mode("append").saveAsTable(AUDIT_TABLE)

        raise RuntimeError(
            f"Coleta interrompida em {dataset}: {error}"
        ) from error

    success_record = [(
        INGESTION_RUN_ID,
        dataset,
        source_url,
        collected_at,
        file_path,
        file_hash,
        size_bytes,
        status,
        None,
    )]

    spark.createDataFrame(
        success_record, schema=audit_schema
    ).write.format("delta").mode("append").saveAsTable(AUDIT_TABLE)

    print(f"{dataset}: {status} | {size_bytes:,} bytes")

    del raw_bytes, text, reader, buffer

from pyspark.sql import functions as F

print("\nAuditoria desta execução:")
display(
    spark.table(AUDIT_TABLE)
    .filter(F.col("run_id") == INGESTION_RUN_ID)
    .select(
        "dataset",
        "status",
        "size_bytes",
        "file_sha256",
        "file_path",
        "error_message",
    )
    .orderBy("dataset")
)

Execução: 1f0f7d61-7bae-4d59-af8a-316ab87ab334
producao_mar: STORED | 1,996,681 bytes
producao_terra: STORED | 4,025,387 bytes
intervencoes: STORED | 1,108,877 bytes
situacao_pocos: STORED | 6,420,612 bytes

Auditoria desta execução:


dataset,status,size_bytes,file_sha256,file_path,error_message
intervencoes,STORED,1108877,0994f2ec9a1dea96b7f861bbb0c8a93cb25dfe6b431164d73d51f26ce9133361,/Volumes/workspace/oil_gas_bronze/raw_files/datasets/intervencoes/sha256=0994f2ec9a1dea96b7f861bbb0c8a93cb25dfe6b431164d73d51f26ce9133361/intervencao_em_pocos_2025-2026.csv,null
producao_mar,STORED,1996681,c78d1f56ade7ead05a19afe23f980d5c7ef2097616334a5802d149eddd858677,/Volumes/workspace/oil_gas_bronze/raw_files/datasets/producao_mar/sha256=c78d1f56ade7ead05a19afe23f980d5c7ef2097616334a5802d149eddd858677/producao-mar-2025.csv,null
producao_terra,STORED,4025387,363786979ea678c9be58638335255aafa05b0807fac63ec9286309846b83e2a0,/Volumes/workspace/oil_gas_bronze/raw_files/datasets/producao_terra/sha256=363786979ea678c9be58638335255aafa05b0807fac63ec9286309846b83e2a0/producao_por_poco_terra_2025_1_trim.csv,null
situacao_pocos,STORED,6420612,babba11ca5128d0e657efd01888af37ea6d57e4ed0aea387a71919eb70c3c197,/Volumes/workspace/oil_gas_bronze/raw_files/datasets/situacao_pocos/sha256=babba11ca5128d0e657efd01888af37ea6d57e4ed0aea387a71919eb70c3c197/situacao-pocos-1939-2026.csv,null


In [0]:
import csv
from pyspark.sql import functions as F

if "INGESTION_RUN_ID" not in globals():
    raise RuntimeError("Execute a célula 8 antes desta célula.")

audit_rows = (
    spark.table(AUDIT_TABLE)
    .filter(
        (F.col("run_id") == INGESTION_RUN_ID)
        & F.col("status").isin("STORED", "REUSED")
    )
    .collect()
)

sources_by_dataset = {}

for item in audit_rows:
    dataset = item["dataset"]

    if dataset in sources_by_dataset:
        raise RuntimeError(
            f"Mais de um arquivo auditado para {dataset} nesta execução."
        )

    sources_by_dataset[dataset] = item.asDict()

if set(sources_by_dataset) != set(required_datasets):
    raise RuntimeError("A auditoria não contém os quatro conjuntos esperados.")

validated_sources = {}
structural_validation_results = []

for dataset in required_datasets:
    source = sources_by_dataset[dataset]
    expected_header = csv_samples[dataset]["header"]

    row_count = 0
    malformed_count = 0
    blank_records = 0
    missing_well_id = 0
    malformed_examples = []

    with open(
        source["file_path"],
        "r",
        encoding="utf-8-sig",
        newline="",
    ) as csv_file:
        reader = csv.reader(
            csv_file,
            delimiter=csv_samples[dataset]["delimiter"],
            strict=True,
        )

        header = next(reader, None)

        if header != expected_header:
            raise RuntimeError(f"{dataset}: cabeçalho divergente.")

        if len(header) != len(set(header)):
            raise RuntimeError(f"{dataset}: nomes de colunas duplicados.")

        well_column = (
            "[Poço]"
            if dataset.startswith("producao_")
            else "Nome_poço_anp"
        )
        well_position = header.index(well_column)

        for record_number, row in enumerate(reader, start=1):
            if not row:
                blank_records += 1
                continue

            row_count += 1

            if len(row) != len(header):
                malformed_count += 1

                if len(malformed_examples) < 5:
                    malformed_examples.append({
                        "registro": record_number,
                        "linha_fisica_final": reader.line_num,
                        "campos_encontrados": len(row),
                        "campos_esperados": len(header),
                    })

                continue

          
            if not row[well_position].strip():
                missing_well_id += 1

    result = (
        dataset,
        len(header),
        row_count,
        blank_records,
        malformed_count,
        missing_well_id,
    )
    structural_validation_results.append(result)

    if malformed_examples:
        print(f"\nProblemas estruturais em {dataset}:")
        for example in malformed_examples:
            print(example)

    if malformed_count == 0 and row_count > 0:
        validated_sources[dataset] = {
            **source,
            "header": header,
            "delimiter": csv_samples[dataset]["delimiter"],
            "expected_row_count": row_count,
        }

validation_df = spark.createDataFrame(
    structural_validation_results,
    schema="""
        dataset STRING,
        quantidade_colunas INT,
        registros_nao_vazios LONG,
        registros_em_branco LONG,
        registros_com_estrutura_invalida LONG,
        registros_validos_sem_id_poco LONG
    """,
)

display(validation_df.orderBy("dataset"))

if len(validated_sources) != len(required_datasets):
    raise RuntimeError(
        "Validação estrutural não aprovada para todos os arquivos. "
        "Confira a tabela e os exemplos acima."
    )

print(
    "\nValidação estrutural aprovada para os quatro arquivos. "
    "Identificadores ausentes permanecem como pendências de qualidade."
)

dataset,quantidade_colunas,registros_nao_vazios,registros_em_branco,registros_com_estrutura_invalida,registros_validos_sem_id_poco
intervencoes,12,8065,0,0,1
producao_mar,21,16690,0,0,0
producao_terra,21,29802,0,0,0
situacao_pocos,24,27316,0,0,112



Validação estrutural aprovada para os quatro arquivos. Identificadores ausentes permanecem como pendências de qualidade.


In [0]:
import re
import uuid
import unicodedata

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

if (
    "validated_sources" not in globals()
    or set(validated_sources) != set(required_datasets)
):
    raise RuntimeError(
        "Execute e aprove a validação estrutural da célula 9."
    )

BRONZE_LOAD_RUN_ID = str(uuid.uuid4())
bronze_load_results = []
bronze_column_mappings = {}

def normalize_column_name(name):
    normalized = (
        unicodedata.normalize("NFKD", name)
        .encode("ascii", "ignore")
        .decode("ascii")
        .lower()
    )

    normalized = re.sub(r"[^a-z0-9]+", "_", normalized).strip("_")

    if not normalized:
        raise ValueError(f"Nome de coluna inválido: {name!r}")

    if normalized[0].isdigit():
        normalized = "col_" + normalized

    return normalized

def quoted_column(name):
    return F.col("`" + name.replace("`", "``") + "`")

print(f"Execução Bronze: {BRONZE_LOAD_RUN_ID}")

for dataset in required_datasets:
    source = validated_sources[dataset]
    source_header = source["header"]
    expected_count = source["expected_row_count"]
    file_hash = source["file_sha256"]

    table_name = (
        f"{PROJECT_CATALOG}."
        f"{PROJECT_SCHEMAS['bronze']}."
        f"{dataset}"
    )

    normalized_header = [
        normalize_column_name(name)
        for name in source_header
    ]

    if len(normalized_header) != len(set(normalized_header)):
        raise RuntimeError(
            f"{dataset}: colisão de nomes após normalização."
        )

    bronze_column_mappings[dataset] = dict(
        zip(source_header, normalized_header)
    )

    existing_count = 0

    if spark.catalog.tableExists(table_name):
        existing_count = (
            spark.table(table_name)
            .filter(F.col("_source_file_sha256") == file_hash)
            .count()
        )

    if existing_count > 0:
        if existing_count != expected_count:
            raise RuntimeError(
                f"{dataset}: conteúdo já carregado com contagem "
                f"divergente: {existing_count} versus {expected_count}."
            )

        status = "SKIPPED_ALREADY_LOADED"
        actual_count = existing_count

    else:
        source_schema = StructType([
            StructField(name, StringType(), True)
            for name in source_header
        ])

        source_df = (
            spark.read
            .format("csv")
            .schema(source_schema)
            .option("header", "true")
            .option("enforceSchema", "false")
            .option("encoding", "UTF-8")
            .option("sep", source["delimiter"])
            .option("quote", '"')
            .option("escape", '"')
            .option("multiLine", "true")
            .option("mode", "FAILFAST")
            .option("ignoreLeadingWhiteSpace", "false")
            .option("ignoreTrailingWhiteSpace", "false")
            .load(source["file_path"])
        )

        bronze_df = source_df.select(
            *[
                quoted_column(original).alias(normalized)
                for original, normalized
                in zip(source_header, normalized_header)
            ],
            F.lit(source["source_url"]).alias("_source_url"),
            F.lit(source["file_path"]).alias("_source_file_path"),
            F.lit(file_hash).alias("_source_file_sha256"),
            F.lit(source["collected_at"]).alias("_source_collected_at"),
            F.lit(source["run_id"]).alias("_ingestion_run_id"),
            F.lit(BRONZE_LOAD_RUN_ID).alias("_bronze_load_run_id"),
            F.current_timestamp().alias("_bronze_loaded_at"),
        )

        (
            bronze_df.write
            .format("delta")
            .mode("append")
            .saveAsTable(table_name)
        )

        actual_count = (
            spark.table(table_name)
            .filter(F.col("_source_file_sha256") == file_hash)
            .count()
        )

        if actual_count != expected_count:
            raise RuntimeError(
                f"{dataset}: contagem Bronze divergente: "
                f"{actual_count} versus {expected_count}. "
                "Interrompa aqui para investigarmos."
            )

        status = "LOADED"

    bronze_load_results.append((
        dataset,
        table_name,
        expected_count,
        actual_count,
        status,
    ))

    print(f"{dataset}: {status} | {actual_count:,} registros")

display(
    spark.createDataFrame(
        bronze_load_results,
        schema="""
            dataset STRING,
            tabela STRING,
            registros_esperados LONG,
            registros_na_bronze_do_arquivo LONG,
            status STRING
        """,
    ).orderBy("dataset")
)

Execução Bronze: ff25f8e4-0d3b-49e8-ad79-80088e180fd1
producao_mar: LOADED | 16,690 registros
producao_terra: LOADED | 29,802 registros
intervencoes: LOADED | 8,065 registros
situacao_pocos: LOADED | 27,316 registros


dataset,tabela,registros_esperados,registros_na_bronze_do_arquivo,status
intervencoes,workspace.oil_gas_bronze.intervencoes,8065,8065,LOADED
producao_mar,workspace.oil_gas_bronze.producao_mar,16690,16690,LOADED
producao_terra,workspace.oil_gas_bronze.producao_terra,29802,29802,LOADED
situacao_pocos,workspace.oil_gas_bronze.situacao_pocos,27316,27316,LOADED


In [0]:
from pyspark.sql import functions as F

key_candidates = {
    "producao_mar": [
        ["poco", "mes_ano"],
        ["poco", "mes_ano", "instalacao"],
    ],
    "producao_terra": [
        ["poco", "mes_ano"],
        ["poco", "mes_ano", "instalacao"],
    ],
    "intervencoes": [
        [
            "nome_poco_anp",
            "data_inicio",
            "data_termino",
            "sonda",
            "objetivo",
        ],
    ],
    "situacao_pocos": [
        ["nome_poco_anp"],
    ],
}

def summarize_duplicates(grouped_counts):
    result = (
        grouped_counts
        .filter(F.col("count") > 1)
        .agg(
            F.count("*").alias("grupos"),
            F.sum("count").alias("registros"),
            F.sum(F.col("count") - 1).alias("excedentes"),
        )
        .first()
    )

    return (
        int(result["grupos"]),
        int(result["registros"] or 0),
        int(result["excedentes"] or 0),
    )

exact_duplicate_results = []
candidate_key_results = []
bronze_current_frames = {}

for dataset in required_datasets:
    source = validated_sources[dataset]
    table_name = (
        f"{PROJECT_CATALOG}."
        f"{PROJECT_SCHEMAS['bronze']}."
        f"{dataset}"
    )

    df = (
        spark.table(table_name)
        .filter(
            F.col("_source_file_sha256") == source["file_sha256"]
        )
    )
    bronze_current_frames[dataset] = df

    business_columns = [
        column for column in df.columns
        if not column.startswith("_")
    ]

    exact_stats = summarize_duplicates(
        df.groupBy(*business_columns).count()
    )

    exact_duplicate_results.append((
        dataset,
        source["expected_row_count"],
        *exact_stats,
    ))

    for columns in key_candidates[dataset]:
        missing_columns = set(columns) - set(df.columns)

        if missing_columns:
            raise RuntimeError(
                f"{dataset}: colunas ausentes: {sorted(missing_columns)}"
            )


        keys_df = df.select(*[
            F.when(
                F.length(F.trim(F.col(column))) > 0,
                F.upper(F.trim(F.col(column))),
            ).otherwise(F.lit(None)).alias(column)
            for column in columns
        ])

        complete_key = F.lit(True)

        for column in columns:
            complete_key = complete_key & F.col(column).isNotNull()

        incomplete_count = keys_df.filter(~complete_key).count()

        grouped_keys = (
            keys_df
            .filter(complete_key)
            .groupBy(*columns)
            .count()
        )

        key_stats = summarize_duplicates(grouped_keys)

        candidate_key_results.append((
            dataset,
            " + ".join(columns),
            incomplete_count,
            *key_stats,
        ))

        if key_stats[0] > 0:
            print(f"\nColisões de chave: {dataset} | {columns}")
            display(
                grouped_keys
                .filter(F.col("count") > 1)
                .orderBy(
                    F.desc("count"),
                    *[F.col(column) for column in columns],
                )
                .limit(5)
            )

print("\nDUPLICIDADES EXATAS — COLUNAS DE NEGÓCIO:")
display(
    spark.createDataFrame(
        exact_duplicate_results,
        schema="""
            dataset STRING,
            registros_do_arquivo LONG,
            grupos_duplicados LONG,
            registros_nos_grupos_duplicados LONG,
            registros_excedentes LONG
        """,
    ).orderBy("dataset")
)

print("\nAVALIAÇÃO DAS CHAVES CANDIDATAS:")
display(
    spark.createDataFrame(
        candidate_key_results,
        schema="""
            dataset STRING,
            chave_candidata STRING,
            registros_com_chave_incompleta LONG,
            grupos_com_chave_repetida LONG,
            registros_nos_grupos_repetidos LONG,
            registros_excedentes LONG
        """,
    ).orderBy("dataset", "chave_candidata")
)


Colisões de chave: producao_mar | ['poco', 'mes_ano']


poco,mes_ano,count
1-BRSA-1146-RJS,01/2025,4
1-BRSA-1146-RJS,02/2025,4
1-BRSA-1146-RJS,03/2025,4
1-BRSA-1146-RJS,04/2025,4
1-BRSA-1146-RJS,05/2025,4



Colisões de chave: producao_mar | ['poco', 'mes_ano', 'instalacao']


poco,mes_ano,instalacao,count
1-BRSA-1146-RJS,01/2025,PETROBRAS 70,4
1-BRSA-1146-RJS,02/2025,PETROBRAS 70,4
1-BRSA-1146-RJS,03/2025,PETROBRAS 70,4
1-BRSA-1146-RJS,04/2025,PETROBRAS 70,4
1-BRSA-1146-RJS,05/2025,PETROBRAS 70,4



Colisões de chave: producao_terra | ['poco', 'mes_ano']


...

poco,mes_ano,instalacao,count
1-STAR-11-RN,01/2025,ESTAÇÃO COLETORA DE PEREIRO (SDM),3
1-STAR-11-RN,02/2025,ESTAÇÃO COLETORA DE PEREIRO (SDM),3
1-STAR-11-RN,03/2025,ESTAÇÃO COLETORA DE PEREIRO (SDM),3
3-STAR-14-RN,01/2025,ESTAÇÃO COLETORA DE PEREIRO (SDM),3
3-STAR-14-RN,02/2025,ESTAÇÃO COLETORA DE PEREIRO (SDM),3



DUPLICIDADES EXATAS — COLUNAS DE NEGÓCIO:


dataset,registros_do_arquivo,grupos_duplicados,registros_nos_grupos_duplicados,registros_excedentes
intervencoes,8065,0,0,0
producao_mar,16690,0,0,0
producao_terra,29802,0,0,0
situacao_pocos,27316,0,0,0



AVALIAÇÃO DAS CHAVES CANDIDATAS:


dataset,chave_candidata,registros_com_chave_incompleta,grupos_com_chave_repetida,registros_nos_grupos_repetidos,registros_excedentes
intervencoes,nome_poco_anp + data_inicio + data_termino + sonda + objetivo,1,0,0,0
producao_mar,poco + mes_ano,0,3777,10318,6541
producao_mar,poco + mes_ano + instalacao,0,3777,10318,6541
producao_terra,poco + mes_ano,0,542,1187,645
producao_terra,poco + mes_ano + instalacao,0,537,1177,640
situacao_pocos,nome_poco_anp,112,0,0,0


In [0]:
from pyspark.sql import functions as F

inspection_cases = [
    ("producao_mar", "1-BRSA-1146-RJS", "01/2025"),
    ("producao_terra", "1-STAR-11-RN", "01/2025"),
]

for dataset, well_name, month in inspection_cases:
    df = bronze_current_frames[dataset]

    business_columns = [
        column for column in df.columns
        if not column.startswith("_")
    ]

    selected = (
        df.filter(
            (F.upper(F.trim(F.col("poco"))) == well_name)
            & (F.trim(F.col("mes_ano")) == month)
        )
        .select(*business_columns)
    )

    print(f"\nCONJUNTO: {dataset}")
    print(f"POÇO: {well_name} | COMPETÊNCIA: {month}")

    variations = selected.agg(*[
        F.countDistinct(
            F.struct(F.col(column))
        ).alias(column)
        for column in business_columns
    ]).first().asDict()

    varying_columns = [
        column for column, distinct_count in variations.items()
        if distinct_count > 1
    ]

    print("Colunas que diferem entre essas linhas:")
    print(varying_columns)

    display(
        selected.select(
            F.to_json(
                F.struct(*[F.col(c) for c in business_columns]),
                options={"ignoreNullFields": "false"},
            ).alias("registro_completo")
        )
    )

extended_key_results = []

extended_candidates = [
    ["poco", "mes_ano", "campo", "instalacao"],
    ["poco", "mes_ano", "campo", "instalacao", "estado", "bacia"],
]

for dataset in ["producao_mar", "producao_terra"]:
    df = bronze_current_frames[dataset]

    for columns in extended_candidates:
        keys_df = df.select(*[
            F.when(
                F.length(F.trim(F.col(column))) > 0,
                F.upper(F.trim(F.col(column))),
            ).otherwise(F.lit(None)).alias(column)
            for column in columns
        ])

        complete_key = F.lit(True)
        for column in columns:
            complete_key = complete_key & F.col(column).isNotNull()

        incomplete_count = keys_df.filter(~complete_key).count()

        duplicate_stats = summarize_duplicates(
            keys_df
            .filter(complete_key)
            .groupBy(*columns)
            .count()
        )

        extended_key_results.append((
            dataset,
            " + ".join(columns),
            incomplete_count,
            *duplicate_stats,
        ))

print("\nCHAVES CANDIDATAS COM CONTEXTO ADICIONAL:")
display(
    spark.createDataFrame(
        extended_key_results,
        schema="""
            dataset STRING,
            chave_candidata STRING,
            registros_com_chave_incompleta LONG,
            grupos_com_chave_repetida LONG,
            registros_nos_grupos_repetidos LONG,
            registros_excedentes LONG
        """,
    ).orderBy("dataset", "chave_candidata")
)


CONJUNTO: producao_mar
POÇO: 1-BRSA-1146-RJS | COMPETÊNCIA: 01/2025
Colunas que diferem entre essas linhas:
['campo', 'producao_de_oleo_m3', 'producao_de_gas_associado_mm3', 'producao_de_agua_m3']


registro_completo
"{""ano"":""2025"",""mes_ano"":""01/2025"",""estado"":""Rio de Janeiro"",""bacia"":""Santos"",""campo"":""AnC_NORTE_ATAPU"",""poco"":""1-BRSA-1146-RJS"",""ambiente"":""MAR"",""instalacao"":""PETROBRAS 70"",""producao_de_oleo_m3"":""47,985"",""producao_de_condensado_m3"":""0"",""producao_de_gas_associado_mm3"":""16,146"",""producao_de_gas_nao_associado_mm3"":""0"",""producao_de_agua_m3"":""0,012"",""injecao_de_gas_mm3"":null,""injecao_de_agua_para_recuperacao_secundaria_m3"":null,""injecao_de_agua_para_descarte_m3"":null,""injecao_de_gas_carbonico_mm3"":null,""injecao_de_nitrogenio_mm3"":null,""injecao_de_vapor_de_agua_t"":null,""injecao_de_polimeros_m3"":null,""injecao_de_outros_fluidos_m3"":null}"
"{""ano"":""2025"",""mes_ano"":""01/2025"",""estado"":""Rio de Janeiro"",""bacia"":""Santos"",""campo"":""ATAPU_ECO"",""poco"":""1-BRSA-1146-RJS"",""ambiente"":""MAR"",""instalacao"":""PETROBRAS 70"",""producao_de_oleo_m3"":""2506,429"",""producao_de_condensado_m3"":""0"",""producao_de_gas_associado_mm3"":""843,392"",""producao_de_gas_nao_associado_mm3"":""0"",""producao_de_agua_m3"":""0,646"",""injecao_de_gas_mm3"":null,""injecao_de_agua_para_recuperacao_secundaria_m3"":null,""injecao_de_agua_para_descarte_m3"":null,""injecao_de_gas_carbonico_mm3"":null,""injecao_de_nitrogenio_mm3"":null,""injecao_de_vapor_de_agua_t"":null,""injecao_de_polimeros_m3"":null,""injecao_de_outros_fluidos_m3"":null}"
"{""ano"":""2025"",""mes_ano"":""01/2025"",""estado"":""Rio de Janeiro"",""bacia"":""Santos"",""campo"":""OESTE DE ATAPU"",""poco"":""1-BRSA-1146-RJS"",""ambiente"":""MAR"",""instalacao"":""PETROBRAS 70"",""producao_de_oleo_m3"":""860,311"",""producao_de_condensado_m3"":""0"",""producao_de_gas_associado_mm3"":""289,488"",""producao_de_gas_nao_associado_mm3"":""0"",""producao_de_agua_m3"":""0,221"",""injecao_de_gas_mm3"":null,""injecao_de_agua_para_recuperacao_secundaria_m3"":null,""injecao_de_agua_para_descarte_m3"":null,""injecao_de_gas_carbonico_mm3"":null,""injecao_de_nitrogenio_mm3"":null,""injecao_de_vapor_de_agua_t"":null,""injecao_de_polimeros_m3"":null,""injecao_de_outros_fluidos_m3"":null}"
"{""ano"":""2025"",""mes_ano"":""01/2025"",""estado"":""Rio de Janeiro"",""bacia"":""Santos"",""campo"":""ATAPU"",""poco"":""1-BRSA-1146-RJS"",""ambiente"":""MAR"",""instalacao"":""PETROBRAS 70"",""producao_de_oleo_m3"":""1636,419"",""producao_de_condensado_m3"":""0"",""producao_de_gas_associado_mm3"":""550,641"",""producao_de_gas_nao_associado_mm3"":""0"",""producao_de_agua_m3"":""0,422"",""injecao_de_gas_mm3"":null,""injecao_de_agua_para_recuperacao_secundaria_m3"":null,""injecao_de_agua_para_descarte_m3"":null,""injecao_de_gas_carbonico_mm3"":null,""injecao_de_nitrogenio_mm3"":null,""injecao_de_vapor_de_agua_t"":null,""injecao_de_polimeros_m3"":null,""injecao_de_outros_fluidos_m3"":null}"



CONJUNTO: producao_terra
POÇO: 1-STAR-11-RN | COMPETÊNCIA: 01/2025
Colunas que diferem entre essas linhas:
['campo', 'producao_de_oleo_m3', 'producao_de_gas_associado_mm3', 'producao_de_agua_m3']


registro_completo
"{""ano"":""2025"",""mes_ano"":""01/2025"",""estado"":""Rio Grande do Norte"",""bacia"":""Potiguar"",""campo"":""SABIÁ DA MATA"",""poco"":""1-STAR-11-RN"",""ambiente"":""TERRA"",""instalacao"":""Estação Coletora de Pereiro (SDM)"",""producao_de_oleo_m3"":""100,584"",""producao_de_condensado_m3"":""0"",""producao_de_gas_associado_mm3"":""4,764"",""producao_de_gas_nao_associado_mm3"":""0"",""producao_de_agua_m3"":""0,108"",""injecao_de_gas_mm3"":null,""injecao_de_agua_para_recuperacao_secundaria_m3"":null,""injecao_de_agua_para_descarte_m3"":null,""injecao_de_gas_carbonico_mm3"":null,""injecao_de_nitrogenio_mm3"":null,""injecao_de_vapor_de_agua_t"":null,""injecao_de_polimeros_m3"":null,""injecao_de_outros_fluidos_m3"":null}"
"{""ano"":""2025"",""mes_ano"":""01/2025"",""estado"":""Rio Grande do Norte"",""bacia"":""Potiguar"",""campo"":""ANGICO"",""poco"":""1-STAR-11-RN"",""ambiente"":""TERRA"",""instalacao"":""Estação Coletora de Pereiro (SDM)"",""producao_de_oleo_m3"":""29,34"",""producao_de_condensado_m3"":""0"",""producao_de_gas_associado_mm3"":""1,389"",""producao_de_gas_nao_associado_mm3"":""0"",""producao_de_agua_m3"":""0,031"",""injecao_de_gas_mm3"":null,""injecao_de_agua_para_recuperacao_secundaria_m3"":null,""injecao_de_agua_para_descarte_m3"":null,""injecao_de_gas_carbonico_mm3"":null,""injecao_de_nitrogenio_mm3"":null,""injecao_de_vapor_de_agua_t"":null,""injecao_de_polimeros_m3"":null,""injecao_de_outros_fluidos_m3"":null}"
"{""ano"":""2025"",""mes_ano"":""01/2025"",""estado"":""Rio Grande do Norte"",""bacia"":""Potiguar"",""campo"":""JANDUÍ"",""poco"":""1-STAR-11-RN"",""ambiente"":""TERRA"",""instalacao"":""Estação Coletora de Pereiro (SDM)"",""producao_de_oleo_m3"":""32,56"",""producao_de_condensado_m3"":""0"",""producao_de_gas_associado_mm3"":""1,542"",""producao_de_gas_nao_associado_mm3"":""0"",""producao_de_agua_m3"":""0,035"",""injecao_de_gas_mm3"":null,""injecao_de_agua_para_recuperacao_secundaria_m3"":null,""injecao_de_agua_para_descarte_m3"":null,""injecao_de_gas_carbonico_mm3"":null,""injecao_de_nitrogenio_mm3"":null,""injecao_de_vapor_de_agua_t"":null,""injecao_de_polimeros_m3"":null,""injecao_de_outros_fluidos_m3"":null}"



CHAVES CANDIDATAS COM CONTEXTO ADICIONAL:


dataset,chave_candidata,registros_com_chave_incompleta,grupos_com_chave_repetida,registros_nos_grupos_repetidos,registros_excedentes
producao_mar,poco + mes_ano + campo + instalacao,0,0,0,0
producao_mar,poco + mes_ano + campo + instalacao + estado + bacia,0,0,0,0
producao_terra,poco + mes_ano + campo + instalacao,0,0,0,0
producao_terra,poco + mes_ano + campo + instalacao + estado + bacia,0,0,0,0


In [0]:
from pyspark.sql import functions as F

numeric_validation_results = []
period_validation_results = []

for dataset in ["producao_mar", "producao_terra"]:
    df = bronze_current_frames[dataset]

    numeric_columns = [
        column for column in df.columns
        if column.startswith(("producao_de_", "injecao_de_"))
    ]

    competence = F.to_date(
        F.try_to_timestamp(
            F.concat(F.lit("01/"), F.trim(F.col("mes_ano"))),
            F.lit("dd/MM/yyyy"),
        )
    )

    date_format_ok = F.coalesce(
        F.trim(F.col("mes_ano")).rlike(r"^\d{2}/\d{4}$"),
        F.lit(False),
    )

    valid_date = date_format_ok & competence.isNotNull()

    source_year = F.expr("try_cast(trim(ano) AS INT)")

    consistent_year = F.coalesce(
        source_year == F.year(competence),
        F.lit(False),
    )

    expected_period = F.year(competence) == 2025

    if dataset == "producao_terra":
        expected_period = (
            expected_period
            & F.month(competence).isin(1, 2, 3)
        )

    def count_when(condition):
        return F.sum(
            F.when(condition, 1).otherwise(0)
        ).cast("long")

    aggregations = [
        F.count("*").alias("total"),
        count_when(~valid_date).alias("invalid_dates"),
        count_when(
            valid_date & ~consistent_year
        ).alias("inconsistent_year"),
        count_when(
            valid_date & ~expected_period
        ).alias("outside_expected_period"),
        F.min(competence).alias("first_month"),
        F.max(competence).alias("last_month"),
    ]

    for index, column in enumerate(numeric_columns):
        raw_value = F.trim(F.col(column))

        missing = (
            raw_value.isNull()
            | (F.length(raw_value) == 0)
        )

        
        format_ok = F.coalesce(
            raw_value.rlike(r"^[+-]?\d+(,\d{1,9})?$"),
            F.lit(False),
        )

        decimal_value = F.expr(
            f"try_cast("
            f"replace(trim(`{column}`), ',', '.') "
            f"AS DECIMAL(28,9))"
        )

        conversion_ok = format_ok & decimal_value.isNotNull()
        invalid = ~missing & ~conversion_ok

        valid_value = F.when(
            ~missing & conversion_ok,
            decimal_value,
        )

        aggregations.extend([
            count_when(missing).alias(f"m_{index}"),
            count_when(invalid).alias(f"i_{index}"),
            count_when(valid_value < 0).alias(f"n_{index}"),
            F.min(valid_value).alias(f"min_{index}"),
            F.max(valid_value).alias(f"max_{index}"),
        ])

    result = df.agg(*aggregations).first().asDict()

    period_validation_results.append((
        dataset,
        result["total"],
        result["invalid_dates"],
        result["inconsistent_year"],
        result["outside_expected_period"],
        result["first_month"],
        result["last_month"],
    ))

    for index, column in enumerate(numeric_columns):
        numeric_validation_results.append((
            dataset,
            column,
            result[f"m_{index}"],
            result[f"i_{index}"],
            result[f"n_{index}"],
            result[f"min_{index}"],
            result[f"max_{index}"],
        ))

print("VALIDAÇÃO DAS COMPETÊNCIAS:")
display(
    spark.createDataFrame(
        period_validation_results,
        schema="""
            dataset STRING,
            registros LONG,
            competencias_invalidas LONG,
            ano_divergente_da_competencia LONG,
            competencias_fora_do_periodo_do_arquivo LONG,
            primeira_competencia DATE,
            ultima_competencia DATE
        """,
    ).orderBy("dataset")
)

print("VALIDAÇÃO DAS MEDIDAS:")
display(
    spark.createDataFrame(
        numeric_validation_results,
        schema="""
            dataset STRING,
            coluna STRING,
            valores_ausentes LONG,
            valores_invalidos LONG,
            valores_negativos LONG,
            menor_valor DECIMAL(28,9),
            maior_valor DECIMAL(28,9)
        """,
    ).orderBy("dataset", "coluna")
)

VALIDAÇÃO DAS COMPETÊNCIAS:


dataset,registros,competencias_invalidas,ano_divergente_da_competencia,competencias_fora_do_periodo_do_arquivo,primeira_competencia,ultima_competencia
producao_mar,16690,0,0,0,2025-01-01,2025-12-01
producao_terra,29802,0,0,0,2025-01-01,2025-03-01


VALIDAÇÃO DAS MEDIDAS:


dataset,coluna,valores_ausentes,valores_invalidos,valores_negativos,menor_valor,maior_valor
producao_mar,injecao_de_agua_para_descarte_m3,10892,0,0,0E-9,176374.070000000
producao_mar,injecao_de_agua_para_recuperacao_secundaria_m3,10892,0,0,0E-9,356721.115000000
producao_mar,injecao_de_gas_carbonico_mm3,10892,0,0,0E-9,60355.350841500
producao_mar,injecao_de_gas_mm3,10892,0,0,0E-9,132271.205000000
producao_mar,injecao_de_nitrogenio_mm3,10892,0,0,0E-9,5.210000000
producao_mar,injecao_de_outros_fluidos_m3,10892,0,0,0E-9,0E-9
producao_mar,injecao_de_polimeros_m3,10892,0,0,0E-9,101.344000000
producao_mar,injecao_de_vapor_de_agua_t,10892,0,0,0E-9,0E-9
producao_mar,producao_de_agua_m3,5790,0,0,0E-9,240420.063000000
producao_mar,producao_de_condensado_m3,5790,0,0,0E-9,3378.588000000


In [0]:
from functools import reduce
from pyspark.sql import functions as F

SILVER_TABLE = "workspace.oil_gas_silver.producao_injecao_mensal"
SILVER_TRANSFORM_VERSION = "v1"

if any(
    row[2] or row[3] or row[4]
    for row in period_validation_results
):
    raise RuntimeError("Há pendências na validação das competências.")

if any(
    row[3] or row[4]
    for row in numeric_validation_results
):
    raise RuntimeError("Há medidas inválidas ou negativas a investigar.")

silver_frames = []

for dataset in ["producao_mar", "producao_terra"]:
    df = bronze_current_frames[dataset]

    numeric_columns = [
        column for column in df.columns
        if column.startswith(("producao_de_", "injecao_de_"))
    ]

    dimension_columns = [
        "estado", "bacia", "campo",
        "poco", "ambiente", "instalacao",
    ]

    metadata_columns = [
        column for column in df.columns
        if column.startswith("_")
    ]

    silver_df = df.select(
        F.col("ano").cast("int").alias("ano"),
        F.to_date(
            F.try_to_timestamp(
                F.concat(F.lit("01/"), F.trim(F.col("mes_ano"))),
                F.lit("dd/MM/yyyy"),
            )
        ).alias("competencia"),

        *[
            F.when(
                F.length(F.trim(F.col(column))) > 0,
                F.upper(F.trim(F.col(column))),
            ).otherwise(F.lit(None)).alias(column)
            for column in dimension_columns
        ],

        *[
            F.expr(
                f"try_cast("
                f"replace(trim(`{column}`), ',', '.') "
                f"AS DECIMAL(28,9))"
            ).alias(column)
            for column in numeric_columns
        ],

        *[F.col(column) for column in metadata_columns],
        F.lit(dataset).alias("_source_dataset"),
        F.lit(SILVER_TRANSFORM_VERSION).alias("_transform_version"),
    )

    has_production = reduce(
        lambda left, right: left | right,
        [
            F.col(column).isNotNull()
            for column in numeric_columns
            if column.startswith("producao_de_")
        ],
    )

    has_injection = reduce(
        lambda left, right: left | right,
        [
            F.col(column).isNotNull()
            for column in numeric_columns
            if column.startswith("injecao_de_")
        ],
    )

    silver_df = silver_df.withColumn(
        "perfil_medidas",
        F.when(
            has_production & has_injection,
            "PRODUCAO_E_INJECAO",
        )
        .when(has_production, "SOMENTE_PRODUCAO")
        .when(has_injection, "SOMENTE_INJECAO")
        .otherwise("SEM_MEDIDAS"),
    )

    key_columns = [
        "ambiente", "poco", "competencia", "campo", "instalacao",
    ]

    silver_df = silver_df.withColumn(
        "registro_id",
        F.sha2(
            F.to_json(
                F.struct(*[F.col(c) for c in key_columns]),
                options={"ignoreNullFields": "false"},
            ),
            256,
        ),
    )

    silver_frames.append(silver_df)

silver_batch = silver_frames[0].unionByName(silver_frames[1])

collisions = (
    silver_batch
    .groupBy("_source_file_sha256", "registro_id")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if collisions:
    raise RuntimeError("Colisão na chave Silver. Nenhuma gravação iniciada.")

silver_batch.createOrReplaceTempView("oil_gas_silver_production_batch")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_TABLE}
    USING DELTA
    COMMENT 'Produção e injeção mensal por poço, campo e instalação.
    Preserva versões da fonte; medidas sem conversão de unidades.
    Unidade da injeção de gás possui divergência documental pendente.'
    AS
    SELECT *
    FROM oil_gas_silver_production_batch
    WHERE 1 = 0
""")

spark.sql(f"""
    MERGE INTO {SILVER_TABLE} AS target
    USING oil_gas_silver_production_batch AS source
    ON target._source_file_sha256 = source._source_file_sha256
       AND target.registro_id = source.registro_id
    WHEN NOT MATCHED THEN INSERT *
""")

batch_versions = silver_batch.select(
    "_source_dataset", "_source_file_sha256"
).distinct()

silver_current_production = (
    spark.table(SILVER_TABLE)
    .join(
        batch_versions,
        ["_source_dataset", "_source_file_sha256"],
        "inner",
    )
)

counts = {
    row["_source_dataset"]: row["count"]
    for row in (
        silver_current_production
        .groupBy("_source_dataset")
        .count()
        .collect()
    )
}

for dataset in ["producao_mar", "producao_terra"]:
    expected = validated_sources[dataset]["expected_row_count"]
    actual = counts.get(dataset, 0)

    if actual != expected:
        raise RuntimeError(
            f"{dataset}: esperado {expected}, encontrado {actual}."
        )

print(f"Tabela Silver: {SILVER_TABLE}")
print("Contagens conferidas com os arquivos de origem.")

display(
    silver_current_production
    .groupBy("_source_dataset", "perfil_medidas")
    .count()
    .orderBy("_source_dataset", "perfil_medidas")
)

display(
    silver_current_production
    .groupBy("_source_dataset")
    .agg(
        F.count("*").alias("registros"),
        F.min("competencia").alias("primeira_competencia"),
        F.max("competencia").alias("ultima_competencia"),
    )
    .orderBy("_source_dataset")
)

Tabela Silver: workspace.oil_gas_silver.producao_injecao_mensal
Contagens conferidas com os arquivos de origem.


_source_dataset,perfil_medidas,count
producao_mar,PRODUCAO_E_INJECAO,8
producao_mar,SOMENTE_INJECAO,5790
producao_mar,SOMENTE_PRODUCAO,10892
producao_terra,PRODUCAO_E_INJECAO,4709
producao_terra,SOMENTE_INJECAO,6589
producao_terra,SOMENTE_PRODUCAO,18504


_source_dataset,registros,primeira_competencia,ultima_competencia
producao_mar,16690,2025-01-01,2025-12-01
producao_terra,29802,2025-01-01,2025-03-01


In [0]:
import uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F

QUALITY_RUN_ID = str(uuid.uuid4())
QUALITY_TABLE = "workspace.oil_gas_ops.measure_reconciliation"
checked_at = datetime.now(timezone.utc)

def aggregate_measures(df, columns):
    expressions = []

    for index, column in enumerate(columns):
        expressions.extend([
            F.sum(F.col(column)).alias(f"sum_{index}"),
            F.count(F.col(column)).alias(f"filled_{index}"),
            F.sum(
                F.when(F.col(column) == 0, 1).otherwise(0)
            ).cast("long").alias(f"zeros_{index}"),
        ])

    return df.agg(*expressions).first().asDict()

reconciliation_records = []

for dataset in ["producao_mar", "producao_terra"]:
    bronze_df = bronze_current_frames[dataset]
    file_hash = validated_sources[dataset]["file_sha256"]

    measure_columns = [
        column for column in bronze_df.columns
        if column.startswith(("producao_de_", "injecao_de_"))
    ]

    # Reconstitui os valores numéricos diretamente da Bronze.
    expected_df = bronze_df.select(*[
        F.expr(
            f"try_cast("
            f"replace(trim(`{column}`), ',', '.') "
            f"AS DECIMAL(28,9))"
        ).alias(column)
        for column in measure_columns
    ])

    actual_df = (
        spark.table(SILVER_TABLE)
        .filter(
            (F.col("_source_dataset") == dataset)
            & (F.col("_source_file_sha256") == file_hash)
        )
        .select(*measure_columns)
    )

    expected = aggregate_measures(expected_df, measure_columns)
    actual = aggregate_measures(actual_df, measure_columns)

    for index, column in enumerate(measure_columns):
        expected_sum = expected[f"sum_{index}"]
        actual_sum = actual[f"sum_{index}"]

        expected_filled = expected[f"filled_{index}"]
        actual_filled = actual[f"filled_{index}"]

        expected_zeros = expected[f"zeros_{index}"]
        actual_zeros = actual[f"zeros_{index}"]

        passed = (
            expected_sum == actual_sum
            and expected_filled == actual_filled
            and expected_zeros == actual_zeros
        )

        # Se todos os valores forem ausentes, mantém total e
        # diferença como null, sem transformar ausência em zero.
        difference = (
            actual_sum - expected_sum
            if actual_sum is not None and expected_sum is not None
            else None
        )

        reconciliation_records.append((
            QUALITY_RUN_ID,
            checked_at,
            dataset,
            file_hash,
            column,
            expected_sum,
            actual_sum,
            difference,
            expected_filled,
            actual_filled,
            expected_zeros,
            actual_zeros,
            "PASS" if passed else "FAIL",
        ))

reconciliation_df = spark.createDataFrame(
    reconciliation_records,
    schema="""
        quality_run_id STRING,
        checked_at TIMESTAMP,
        dataset STRING,
        source_file_sha256 STRING,
        medida STRING,
        total_bronze DECIMAL(38,9),
        total_silver DECIMAL(38,9),
        diferenca DECIMAL(38,9),
        preenchidos_bronze LONG,
        preenchidos_silver LONG,
        zeros_bronze LONG,
        zeros_silver LONG,
        status STRING
    """,
)

(
    reconciliation_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(QUALITY_TABLE)
)

print(f"Execução de qualidade: {QUALITY_RUN_ID}")

display(
    reconciliation_df
    .select(
        "dataset",
        "medida",
        "total_bronze",
        "total_silver",
        "diferenca",
        "preenchidos_bronze",
        "preenchidos_silver",
        "zeros_bronze",
        "zeros_silver",
        "status",
    )
    .orderBy("dataset", "medida")
)

failed_checks = [
    (record[2], record[4])
    for record in reconciliation_records
    if record[-1] == "FAIL"
]

if failed_checks:
    raise RuntimeError(
        f"Reconciliação reprovada: {failed_checks}"
    )

print(
    f"\nReconciliação aprovada: "
    f"{len(reconciliation_records)} medidas verificadas."
)

Execução de qualidade: 6739054d-b924-47f9-b373-28cba5fa5a7d


dataset,medida,total_bronze,total_silver,diferenca,preenchidos_bronze,preenchidos_silver,zeros_bronze,zeros_silver,status
producao_mar,injecao_de_agua_para_descarte_m3,3073358.332000000,3073358.332000000,0E-9,5798,5798,5753,5753,PASS
producao_mar,injecao_de_agua_para_recuperacao_secundaria_m3,234740205.849000000,234740205.849000000,0E-9,5798,5798,1785,1785,PASS
producao_mar,injecao_de_gas_carbonico_mm3,10816714.130966000,10816714.130966000,0E-9,5798,5798,4297,4297,PASS
producao_mar,injecao_de_gas_mm3,32727996.134000000,32727996.134000000,0E-9,5798,5798,4318,4318,PASS
producao_mar,injecao_de_nitrogenio_mm3,919.560000000,919.560000000,0E-9,5798,5798,4304,4304,PASS
producao_mar,injecao_de_outros_fluidos_m3,0E-9,0E-9,0E-9,5798,5798,5798,5798,PASS
producao_mar,injecao_de_polimeros_m3,223.274000000,223.274000000,0E-9,5798,5798,5795,5795,PASS
producao_mar,injecao_de_vapor_de_agua_t,0E-9,0E-9,0E-9,5798,5798,5798,5798,PASS
producao_mar,producao_de_agua_m3,105134059.832000000,105134059.832000000,0E-9,10900,10900,1302,1302,PASS
producao_mar,producao_de_condensado_m3,133440.277000000,133440.277000000,0E-9,10900,10900,10680,10680,PASS



Reconciliação aprovada: 26 medidas verificadas.


In [0]:
from pyspark.sql import functions as F

RECOVERY_INGESTION_RUN_ID = (
    "1f0f7d61-7bae-4d59-af8a-316ab87ab334"
)

required_datasets = [
    "producao_mar",
    "producao_terra",
    "intervencoes",
    "situacao_pocos",
]

audit_rows = (
    spark.table("workspace.oil_gas_ops.ingestion_files")
    .filter(
        (F.col("run_id") == RECOVERY_INGESTION_RUN_ID)
        & F.col("status").isin("STORED", "REUSED")
    )
    .select("dataset", "file_sha256")
    .distinct()
    .collect()
)

versions = {}

for row in audit_rows:
    dataset = row["dataset"]

    if dataset in versions:
        raise RuntimeError(
            f"Mais de uma versão encontrada para {dataset}."
        )

    versions[dataset] = row["file_sha256"]

if set(versions) != set(required_datasets):
    raise RuntimeError(
        "A auditoria não contém os quatro conjuntos esperados."
    )

bronze_current_frames = {}
recovery_results = []

for dataset in required_datasets:
    table_name = f"workspace.oil_gas_bronze.{dataset}"

    df = (
        spark.table(table_name)
        .filter(
            F.col("_source_file_sha256") == versions[dataset]
        )
    )

    row_count = df.count()

    if row_count == 0:
        raise RuntimeError(
            f"Nenhum registro encontrado para {dataset} nessa versão."
        )

    bronze_current_frames[dataset] = df
    recovery_results.append((dataset, row_count))

display(
    spark.createDataFrame(
        recovery_results,
        schema="dataset STRING, registros LONG",
    ).orderBy("dataset")
)

dataset,registros
intervencoes,8065
producao_mar,16690
producao_terra,29802
situacao_pocos,27316


In [0]:
# Célula 16 — Validar datas e duração das intervenções
# Não modifica nem grava as bases.

from pyspark.sql import functions as F

def nonblank_text(column):
    value = F.trim(F.col(column))
    return F.when(
        F.length(value) > 0, value
    ).otherwise(F.lit(None))

def count_condition(condition):
    return F.sum(
        F.when(condition, 1).otherwise(0)
    ).cast("long")

date_validation_results = []

for dataset in ["intervencoes", "situacao_pocos"]:
    df = bronze_current_frames[dataset]

    date_columns = [
        column for column in df.columns
        if column.startswith("data_")
    ]

    expressions = []

    for index, column in enumerate(date_columns):
        value = nonblank_text(column)
        monthly = "mes_ano" in column

        pattern = "MM/yyyy" if monthly else "dd/MM/yyyy"
        expected_regex = (
            r"^\d{2}/\d{4}$"
            if monthly
            else r"^\d{2}/\d{2}/\d{4}$"
        )

        parse_input = (
            F.concat(F.lit("01/"), value)
            if monthly else value
        )

        parsed_date = F.to_date(
            F.try_to_timestamp(
                parse_input,
                F.lit("dd/MM/yyyy"),
            )
        )

        invalid = (
            value.isNotNull()
            & (
                ~value.rlike(expected_regex)
                | parsed_date.isNull()
            )
        )

        expressions.extend([
            count_condition(value.isNull()).alias(f"missing_{index}"),
            count_condition(invalid).alias(f"invalid_{index}"),
            F.min(
                F.when(~invalid, parsed_date)
            ).alias(f"min_{index}"),
            F.max(
                F.when(~invalid, parsed_date)
            ).alias(f"max_{index}"),
        ])

    result = df.agg(*expressions).first().asDict()

    for index, column in enumerate(date_columns):
        date_validation_results.append((
            dataset,
            column,
            "MM/yyyy" if "mes_ano" in column else "dd/MM/yyyy",
            result[f"missing_{index}"],
            result[f"invalid_{index}"],
            result[f"min_{index}"],
            result[f"max_{index}"],
        ))

print("VALIDAÇÃO DAS DATAS:")
display(
    spark.createDataFrame(
        date_validation_results,
        schema="""
            dataset STRING,
            coluna STRING,
            formato_esperado STRING,
            valores_ausentes LONG,
            valores_invalidos LONG,
            menor_data DATE,
            maior_data DATE
        """,
    ).orderBy("dataset", "coluna")
)

# Diagnóstico específico da duração das intervenções.
intervention_date_preview = (
    bronze_current_frames["intervencoes"]
    .withColumn(
        "_inicio_convertido",
        F.to_date(F.try_to_timestamp(
            nonblank_text("data_inicio"),
            F.lit("dd/MM/yyyy"),
        )),
    )
    .withColumn(
        "_termino_convertido",
        F.to_date(F.try_to_timestamp(
            nonblank_text("data_termino"),
            F.lit("dd/MM/yyyy"),
        )),
    )
    .withColumn(
        "_dias_informados",
        F.expr("try_cast(trim(dias_em_intervencao) AS INT)"),
    )
    .withColumn(
        "_dias_calendario_inclusivos",
        F.datediff(
            F.col("_termino_convertido"),
            F.col("_inicio_convertido"),
        ) + 1,
    )
)

days_text = nonblank_text("dias_em_intervencao")

invalid_days = (
    days_text.isNotNull()
    & (
        ~days_text.rlike(r"^[+-]?\d+$")
        | F.col("_dias_informados").isNull()
    )
)

reversed_dates = (
    F.col("_termino_convertido") < F.col("_inicio_convertido")
)

different_duration = (
    F.col("_dias_informados")
    != F.col("_dias_calendario_inclusivos")
)

print("DIAGNÓSTICO DA DURAÇÃO:")
display(
    intervention_date_preview.agg(
        F.count("*").alias("intervencoes"),
        count_condition(days_text.isNull()).alias("dias_ausentes"),
        count_condition(invalid_days).alias("dias_invalidos"),
        count_condition(
            F.col("_dias_informados") < 0
        ).alias("dias_negativos"),
        count_condition(reversed_dates).alias("termino_antes_do_inicio"),
        count_condition(
            F.col("_dias_informados").isNotNull()
            & F.col("_dias_calendario_inclusivos").isNotNull()
        ).alias("duracoes_comparaveis"),
        count_condition(different_duration).alias("duracoes_diferentes"),
    )
)

print("EXEMPLOS DE DIVERGÊNCIAS, SE EXISTIREM:")
display(
    intervention_date_preview
    .filter(invalid_days | reversed_dates | different_duration)
    .select(
        "nome_poco_anp",
        "nome_poco_operador",
        "objetivo",
        "data_inicio",
        "data_termino",
        "dias_em_intervencao",
        "_dias_calendario_inclusivos",
    )
    .orderBy("nome_poco_anp", "data_inicio")
    .limit(10)
)

VALIDAÇÃO DAS DATAS:


dataset,coluna,formato_esperado,valores_ausentes,valores_invalidos,menor_data,maior_data
intervencoes,data_inicio,dd/MM/yyyy,0,0,2025-01-01,2026-08-31
intervencoes,data_termino,dd/MM/yyyy,0,0,2025-01-01,2026-09-20
situacao_pocos,data_atualizacao_da_situacao,dd/MM/yyyy,135,0,2008-06-23,2026-09-18
situacao_pocos,data_conclusao_poco,dd/MM/yyyy,187,0,1939-12-23,2026-09-01
situacao_pocos,data_de_referencia_situacao,dd/MM/yyyy,135,0,1942-02-27,2026-09-17
situacao_pocos,data_inicio_injecao_mes_ano,MM/yyyy,22716,0,2003-05-01,2026-08-01
situacao_pocos,data_inicio_operacao_mes_ano,MM/yyyy,6249,0,1941-12-01,2026-08-01
situacao_pocos,data_inicio_perfuracao,dd/MM/yyyy,111,0,1939-09-30,2026-09-10
situacao_pocos,data_inicio_producao_mes_ano,MM/yyyy,7431,0,1941-12-01,2026-08-01
situacao_pocos,data_termino_perfuracao,dd/MM/yyyy,1824,0,1939-12-23,2026-09-05


DIAGNÓSTICO DA DURAÇÃO:


intervencoes,dias_ausentes,dias_invalidos,dias_negativos,termino_antes_do_inicio,duracoes_comparaveis,duracoes_diferentes
8065,0,0,0,0,8065,0


EXEMPLOS DE DIVERGÊNCIAS, SE EXISTIREM:


nome_poco_anp,nome_poco_operador,objetivo,data_inicio,data_termino,dias_em_intervencao,_dias_calendario_inclusivos


In [0]:
from pyspark.sql import functions as F

SOURCE_RUN_ID = "1f0f7d61-7bae-4d59-af8a-316ab87ab334"
INTERVENTION_TABLE = "workspace.oil_gas_silver.intervencoes"

versions = (
    spark.table("workspace.oil_gas_ops.ingestion_files")
    .filter(
        (F.col("run_id") == SOURCE_RUN_ID)
        & (F.col("dataset") == "intervencoes")
        & F.col("status").isin("STORED", "REUSED")
    )
    .select("file_sha256")
    .distinct()
    .collect()
)

if len(versions) != 1:
    raise RuntimeError("Esperada uma versão auditada de intervenções.")

intervention_source_hash = versions[0]["file_sha256"]

source_df = (
    spark.table("workspace.oil_gas_bronze.intervencoes")
    .filter(
        F.col("_source_file_sha256") == intervention_source_hash
    )
)

expected_count = source_df.count()

if expected_count == 0:
    raise RuntimeError("Nenhuma intervenção encontrada na Bronze.")

business_columns = [
    column for column in source_df.columns
    if not column.startswith("_")
]

def clean_text(column):
    value = F.trim(F.col(column))
    return F.when(
        F.length(value) > 0,
        F.upper(value),
    ).otherwise(F.lit(None))

source_df = source_df.withColumn(
    "_source_record_sha256",
    F.sha2(
        F.to_json(
            F.struct(*[F.col(c) for c in business_columns]),
            options={"ignoreNullFields": "false"},
        ),
        256,
    ),
)

expressions = []

for column in business_columns:
    if column in {"data_inicio", "data_termino"}:
        expression = F.to_date(
            F.try_to_timestamp(
                F.trim(F.col(column)),
                F.lit("dd/MM/yyyy"),
            )
        )
    elif column == "dias_em_intervencao":
        expression = F.expr(
            "try_cast(trim(dias_em_intervencao) AS INT)"
        )
    else:
        expression = clean_text(column)

    expressions.append(expression.alias(column))

metadata_columns = [
    column for column in source_df.columns
    if column.startswith("_")
]

intervention_batch = (
    source_df.select(
        *expressions,
        *[F.col(c) for c in metadata_columns],
    )
    .withColumn(
        "dias_calendario_inclusivos",
        F.datediff("data_termino", "data_inicio") + 1,
    )
    .withColumn(
        "identificador_anp_presente",
        F.col("nome_poco_anp").isNotNull(),
    )
    .withColumn(
        "status_identificacao",
        F.when(
            F.col("identificador_anp_presente"),
            "ID_ANP_INFORMADO",
        ).otherwise("SEM_ID_ANP"),
    )
    .withColumn("_transform_version", F.lit("v1"))
    .withColumn("_silver_loaded_at", F.current_timestamp())
)

invalid_rows = (
    intervention_batch
    .filter(
        F.col("data_inicio").isNull()
        | F.col("data_termino").isNull()
        | F.col("dias_em_intervencao").isNull()
        | (F.col("data_termino") < F.col("data_inicio"))
        | (F.col("dias_em_intervencao") < 0)
        | (
            F.col("dias_em_intervencao")
            != F.col("dias_calendario_inclusivos")
        )
    )
    .limit(1)
    .count()
)

if invalid_rows:
    raise RuntimeError("Validação da transformação reprovada.")

duplicate_records = (
    intervention_batch
    .groupBy("_source_file_sha256", "_source_record_sha256")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if duplicate_records:
    raise RuntimeError("Há registros repetidos na chave técnica.")

intervention_batch.createOrReplaceTempView(
    "oil_gas_intervention_batch"
)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {INTERVENTION_TABLE}
    USING DELTA
    COMMENT 'Intervenções com datas e duração tipadas.
    Preserva versões da fonte e eventos sem identificador ANP.'
    AS
    SELECT *
    FROM oil_gas_intervention_batch
    WHERE 1 = 0
""")

spark.sql(f"""
    MERGE INTO {INTERVENTION_TABLE} AS target
    USING oil_gas_intervention_batch AS source
    ON target._source_file_sha256 = source._source_file_sha256
       AND target._source_record_sha256 = source._source_record_sha256
    WHEN NOT MATCHED THEN INSERT *
""")

interventions_current = (
    spark.table(INTERVENTION_TABLE)
    .filter(
        F.col("_source_file_sha256") == intervention_source_hash
    )
)

actual_count = interventions_current.count()

if actual_count != expected_count:
    raise RuntimeError(
        f"Contagem divergente: Bronze={expected_count}, "
        f"Silver={actual_count}."
    )

print(f"Tabela: {INTERVENTION_TABLE}")
print(f"Bronze: {expected_count:,} | Silver: {actual_count:,}")

display(
    interventions_current
    .groupBy("status_identificacao")
    .count()
    .orderBy("status_identificacao")
)

display(
    interventions_current
    .groupBy("ambiente", "objetivo")
    .count()
    .orderBy("ambiente", F.desc("count"), "objetivo")
)

Tabela: workspace.oil_gas_silver.intervencoes
Bronze: 8,065 | Silver: 8,065


status_identificacao,count
ID_ANP_INFORMADO,8064
SEM_ID_ANP,1


ambiente,objetivo,count
MAR,ABANDONO,326
MAR,PERFURAÇÃO,209
MAR,RESTAURAÇÃO,157
MAR,COMPLETAÇÃO,136
MAR,AVALIAÇÃO,5
MAR,TLD,2
TERRA,RESTAURAÇÃO,4582
TERRA,COMPLETAÇÃO,1828
TERRA,ABANDONO,435
TERRA,PERFURAÇÃO,331


In [0]:
from pyspark.sql import functions as F

SOURCE_RUN_ID = "1f0f7d61-7bae-4d59-af8a-316ab87ab334"
WELL_TABLE = "workspace.oil_gas_silver.cadastro_pocos"

versions = (
    spark.table("workspace.oil_gas_ops.ingestion_files")
    .filter(
        (F.col("run_id") == SOURCE_RUN_ID)
        & (F.col("dataset") == "situacao_pocos")
        & F.col("status").isin("STORED", "REUSED")
    )
    .select("file_sha256")
    .distinct()
    .collect()
)

if len(versions) != 1:
    raise RuntimeError("Esperada uma versão auditada do cadastro.")

well_source_hash = versions[0]["file_sha256"]

source_df = (
    spark.table("workspace.oil_gas_bronze.situacao_pocos")
    .filter(F.col("_source_file_sha256") == well_source_hash)
)

expected_count = source_df.count()

if expected_count == 0:
    raise RuntimeError("Cadastro Bronze sem registros nessa versão.")

business_columns = [
    column for column in source_df.columns
    if not column.startswith("_")
]

def nullable_trimmed_text(column):
    value = F.trim(F.col(column))
    return F.when(
        F.length(value) > 0, value
    ).otherwise(F.lit(None))

expressions = []
invalid_date_condition = F.lit(False)

for column in business_columns:
    value = nullable_trimmed_text(column)

    if column.startswith("data_"):
        monthly = "mes_ano" in column

        date_input = (
            F.concat(F.lit("01/"), value)
            if monthly else value
        )

        parsed = F.to_date(
            F.try_to_timestamp(
                date_input,
                F.lit("dd/MM/yyyy"),
            )
        )

        invalid_date_condition = (
            invalid_date_condition
            | (value.isNotNull() & parsed.isNull())
        )

        expression = parsed

    elif column in {"latitude_da_base", "longitude_da_base"}:
        expression = value

    else:
        expression = F.upper(value)

    expressions.append(expression.alias(column))

if source_df.filter(invalid_date_condition).limit(1).count():
    raise RuntimeError("Há datas preenchidas que não foram convertidas.")

source_df = source_df.withColumn(
    "_source_record_sha256",
    F.sha2(
        F.to_json(
            F.struct(*[F.col(c) for c in business_columns]),
            options={"ignoreNullFields": "false"},
        ),
        256,
    ),
)

metadata_columns = [
    column for column in source_df.columns
    if column.startswith("_")
]

well_batch = (
    source_df.select(
        *expressions,
        *[F.col(c) for c in metadata_columns],
    )
    .withColumn(
        "status_identificacao",
        F.when(
            F.col("nome_poco_anp").isNotNull(),
            "ID_ANP_INFORMADO",
        ).otherwise("SEM_ID_ANP"),
    )
    .withColumn("_transform_version", F.lit("v1"))
    .withColumn("_silver_loaded_at", F.current_timestamp())
)

duplicate_well_ids = (
    well_batch
    .filter(F.col("nome_poco_anp").isNotNull())
    .groupBy("nome_poco_anp")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if duplicate_well_ids:
    raise RuntimeError(
        "Identificador ANP repetido após a padronização."
    )

duplicate_records = (
    well_batch
    .groupBy("_source_file_sha256", "_source_record_sha256")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if duplicate_records:
    raise RuntimeError("Registros repetidos na chave técnica.")

well_batch.createOrReplaceTempView("oil_gas_well_batch")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {WELL_TABLE}
    USING DELTA
    COMMENT 'Cadastro de poços por versão da fonte.
    Datas mes_ano representam competências mensais.
    Situação e operador não constituem histórico completo.
    Coordenadas preservadas no formato original.'
    AS
    SELECT *
    FROM oil_gas_well_batch
    WHERE 1 = 0
""")

spark.sql(f"""
    MERGE INTO {WELL_TABLE} AS target
    USING oil_gas_well_batch AS source
    ON target._source_file_sha256 = source._source_file_sha256
       AND target._source_record_sha256 = source._source_record_sha256
    WHEN NOT MATCHED THEN INSERT *
""")

wells_current = (
    spark.table(WELL_TABLE)
    .filter(F.col("_source_file_sha256") == well_source_hash)
)

actual_count = wells_current.count()

if actual_count != expected_count:
    raise RuntimeError(
        f"Contagem divergente: Bronze={expected_count}, "
        f"Silver={actual_count}."
    )

print(f"Tabela: {WELL_TABLE}")
print(f"Bronze: {expected_count:,} | Silver: {actual_count:,}")

display(
    wells_current
    .groupBy("status_identificacao")
    .count()
    .orderBy("status_identificacao")
)

display(
    wells_current
    .groupBy("ambiente", "situacao_do_poco")
    .count()
    .orderBy("ambiente", F.desc("count"), "situacao_do_poco")
)

Tabela: workspace.oil_gas_silver.cadastro_pocos
Bronze: 27,316 | Silver: 27,316


status_identificacao,count
ID_ANP_INFORMADO,27204
SEM_ID_ANP,112


ambiente,situacao_do_poco,count
MAR,ABANDONADO PERMANENTEMENTE,3045
MAR,ABANDONADO TEMPORARIAMENTE COM MONITORAMENTO,711
MAR,PRODUZINDO,531
MAR,FECHADO,379
MAR,ABANDONADO TEMPORARIAMENTE SEM MONITORAMENTO,370
MAR,ARRASADO,349
MAR,INJETANDO,242
MAR,null,48
MAR,EQUIPADO AGUARDANDO INÍCIO DE OPERAÇÃO,31
MAR,EM INTERVENÇÃO,21


In [0]:
from pyspark.sql import functions as F

SOURCE_RUN_ID = "1f0f7d61-7bae-4d59-af8a-316ab87ab334"

audit_versions = (
    spark.table("workspace.oil_gas_ops.ingestion_files")
    .filter(
        (F.col("run_id") == SOURCE_RUN_ID)
        & F.col("status").isin("STORED", "REUSED")
    )
    .select("dataset", "file_sha256")
    .distinct()
    .collect()
)

source_hashes = {}

for row in audit_versions:
    if row["dataset"] in source_hashes:
        raise RuntimeError(
            f"Mais de uma versão para {row['dataset']}."
        )
    source_hashes[row["dataset"]] = row["file_sha256"]

required = {
    "producao_mar", "producao_terra",
    "intervencoes", "situacao_pocos",
}

if not required.issubset(source_hashes):
    raise RuntimeError("Faltam versões auditadas do piloto.")

cadastro = (
    spark.table("workspace.oil_gas_silver.cadastro_pocos")
    .filter(
        F.col("_source_file_sha256")
        == source_hashes["situacao_pocos"]
    )
    .filter(F.col("nome_poco_anp").isNotNull())
    .select(
        F.col("nome_poco_anp").alias("_cad_poco_anp"),
        F.col("ambiente").alias("_cad_ambiente"),
    )
)

if (
    cadastro.groupBy("_cad_poco_anp")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Cadastro não é único por identificador ANP.")

coverage_results = []
environment_results = []
linkage_previews = {}

for dataset in ["producao_mar", "producao_terra", "intervencoes"]:
    if dataset == "intervencoes":
        source = (
            spark.table("workspace.oil_gas_silver.intervencoes")
            .filter(
                F.col("_source_file_sha256") == source_hashes[dataset]
            )
        )
        well_column = "nome_poco_anp"
        detail_columns = [
            "nome_poco_anp", "nome_poco_operador",
            "campo", "ambiente", "objetivo",
            "data_inicio", "data_termino",
        ]

    else:
        source = (
            spark.table(
                "workspace.oil_gas_silver.producao_injecao_mensal"
            )
            .filter(
                (F.col("_source_dataset") == dataset)
                & (
                    F.col("_source_file_sha256")
                    == source_hashes[dataset]
                )
            )
        )
        well_column = "poco"
        detail_columns = [
            "poco", "competencia", "campo",
            "ambiente", "instalacao",
        ]

    source_count = source.count()

    if source_count == 0:
        raise RuntimeError(f"{dataset}: fonte Silver vazia.")

    linked = (
        source.withColumn("_join_poco_anp", F.col(well_column))
        .join(
            cadastro,
            F.col("_join_poco_anp") == F.col("_cad_poco_anp"),
            "left",
        )
        .withColumn(
            "status_vinculo",
            F.when(
                F.col("_join_poco_anp").isNull(),
                "SEM_ID_ANP",
            )
            .when(
                F.col("_cad_poco_anp").isNull(),
                "NAO_LOCALIZADO_NO_CADASTRO",
            )
            .otherwise("LOCALIZADO"),
        )
    )

    summary = (
        linked.groupBy("status_vinculo")
        .agg(
            F.count("*").alias("registros"),
            F.countDistinct("_join_poco_anp").alias(
                "identificadores_distintos"
            ),
        )
        .collect()
    )

    linked_count = sum(row["registros"] for row in summary)

    if linked_count != source_count:
        raise RuntimeError(
            f"{dataset}: o cruzamento alterou a quantidade de linhas."
        )

    for row in summary:
        coverage_results.append((
            dataset,
            row["status_vinculo"],
            row["registros"],
            row["identificadores_distintos"],
            source_count,
        ))

    environment_mismatch = linked.filter(
        (F.col("status_vinculo") == "LOCALIZADO")
        & ~F.col("ambiente").eqNullSafe(F.col("_cad_ambiente"))
    )

    environment_results.append((
        dataset,
        source_count,
        linked_count,
        environment_mismatch.count(),
    ))

    linkage_previews[dataset] = linked

    print(f"\nEXEMPLOS SEM VÍNCULO: {dataset}")
    display(
        linked
        .filter(F.col("status_vinculo") != "LOCALIZADO")
        .select(*detail_columns, "status_vinculo")
        .orderBy(
            *[
                F.col(c).asc_nulls_first()
                for c in detail_columns
            ]
        )
        .limit(10)
    )

print("\nCOBERTURA DO VÍNCULO:")
display(
    spark.createDataFrame(
        coverage_results,
        schema="""
            dataset STRING,
            status_vinculo STRING,
            registros LONG,
            identificadores_distintos LONG,
            registros_da_fonte LONG
        """,
    )
    .withColumn(
        "percentual_registros",
        F.round(
            100.0 * F.col("registros") / F.col("registros_da_fonte"),
            2,
        ),
    )
    .orderBy("dataset", "status_vinculo")
)

print("\nINTEGRIDADE DO CRUZAMENTO:")
display(
    spark.createDataFrame(
        environment_results,
        schema="""
            dataset STRING,
            linhas_antes LONG,
            linhas_depois LONG,
            registros_com_ambiente_divergente LONG
        """,
    ).orderBy("dataset")
)


EXEMPLOS SEM VÍNCULO: producao_mar


poco,competencia,campo,ambiente,instalacao,status_vinculo



EXEMPLOS SEM VÍNCULO: producao_terra


poco,competencia,campo,ambiente,instalacao,status_vinculo
4-GO-13-BA,2025-03-01,GOMO,TERRA,ESTAÇÃO GOMO,NAO_LOCALIZADO_NO_CADASTRO



EXEMPLOS SEM VÍNCULO: intervencoes


nome_poco_anp,nome_poco_operador,campo,ambiente,objetivo,data_inicio,data_termino,status_vinculo
null,3-ORGM-13DP-AL,PILAR,TERRA,PERFURAÇÃO,2025-11-16,2025-11-16,SEM_ID_ANP



COBERTURA DO VÍNCULO:


dataset,status_vinculo,registros,identificadores_distintos,registros_da_fonte,percentual_registros
intervencoes,LOCALIZADO,8064,4882,8065,99.99
intervencoes,SEM_ID_ANP,1,0,8065,0.01
producao_mar,LOCALIZADO,16690,958,16690,100.0
producao_terra,LOCALIZADO,29801,10027,29802,100.0
producao_terra,NAO_LOCALIZADO_NO_CADASTRO,1,1,29802,0.0



INTEGRIDADE DO CRUZAMENTO:


dataset,linhas_antes,linhas_depois,registros_com_ambiente_divergente
intervencoes,8065,8065,0
producao_mar,16690,16690,0
producao_terra,29802,29802,124


In [0]:
from pyspark.sql import functions as F

SOURCE_RUN_ID = "1f0f7d61-7bae-4d59-af8a-316ab87ab334"

version_rows = (
    spark.table("workspace.oil_gas_ops.ingestion_files")
    .filter(
        (F.col("run_id") == SOURCE_RUN_ID)
        & F.col("dataset").isin("producao_terra", "situacao_pocos")
        & F.col("status").isin("STORED", "REUSED")
    )
    .select("dataset", "file_sha256")
    .distinct()
    .collect()
)

versions = {}

for row in version_rows:
    if row["dataset"] in versions:
        raise RuntimeError("Mais de uma versão para o mesmo conjunto.")
    versions[row["dataset"]] = row["file_sha256"]

if set(versions) != {"producao_terra", "situacao_pocos"}:
    raise RuntimeError("Versões necessárias não encontradas.")

production = (
    spark.table("workspace.oil_gas_silver.producao_injecao_mensal")
    .filter(
        (F.col("_source_dataset") == "producao_terra")
        & (
            F.col("_source_file_sha256")
            == versions["producao_terra"]
        )
    )
)

cadastro = (
    spark.table("workspace.oil_gas_silver.cadastro_pocos")
    .filter(
        F.col("_source_file_sha256") == versions["situacao_pocos"]
    )
    .filter(F.col("nome_poco_anp").isNotNull())
    .select(
        F.col("nome_poco_anp").alias("cad_poco"),
        F.col("ambiente").alias("cad_ambiente"),
        F.col("campo").alias("cad_campo"),
        F.col("bacia").alias("cad_bacia"),
        F.col("estado").alias("cad_estado"),
        F.col("situacao_do_poco").alias("cad_situacao"),
        F.col("data_de_referencia_situacao").alias(
            "cad_data_referencia_situacao"
        ),
        F.col("data_atualizacao_da_situacao").alias(
            "cad_data_atualizacao_situacao"
        ),
    )
)

if (
    cadastro.groupBy("cad_poco")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Cadastro não é único por identificador.")

comparison = production.join(
    cadastro,
    production["poco"] == cadastro["cad_poco"],
    "left",
)

environment_exceptions = comparison.filter(
    F.col("cad_poco").isNotNull()
    & ~F.col("ambiente").eqNullSafe(F.col("cad_ambiente"))
)

print("RESUMO DAS DIVERGÊNCIAS:")
display(
    environment_exceptions
    .groupBy("ambiente", "cad_ambiente")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("poco").alias("pocos_distintos"),
        F.min("competencia").alias("primeira_competencia"),
        F.max("competencia").alias("ultima_competencia"),
    )
)

print("POÇOS ENVOLVIDOS — ATÉ 30:")
display(
    environment_exceptions
    .groupBy(
        "poco",
        "ambiente",
        "cad_ambiente",
        "estado",
        "cad_estado",
        "bacia",
        "cad_bacia",
        "cad_situacao",
    )
    .agg(
        F.count("*").alias("registros"),
        F.sort_array(
            F.collect_set("campo")
        ).alias("campos_na_producao"),
        F.sort_array(
            F.collect_set("cad_campo")
        ).alias("campos_no_cadastro"),
        F.min("competencia").alias("primeira_competencia"),
        F.max("competencia").alias("ultima_competencia"),
        F.max("cad_data_referencia_situacao").alias(
            "data_referencia_situacao"
        ),
    )
    .orderBy(F.desc("registros"), "poco")
    .limit(30)
)

print("DETALHE DOS REGISTROS — ATÉ 20:")
display(
    environment_exceptions
    .select(
        "poco",
        "competencia",
        "campo",
        "cad_campo",
        "ambiente",
        "cad_ambiente",
        "instalacao",
        "producao_de_oleo_m3",
        "producao_de_gas_associado_mm3",
        "perfil_medidas",
    )
    .orderBy("poco", "competencia", "campo", "instalacao")
    .limit(20)
)

print("PRODUÇÃO SEM CORRESPONDÊNCIA NO CADASTRO:")
display(
    comparison
    .filter(F.col("cad_poco").isNull())
    .select(
        "poco",
        "competencia",
        "campo",
        "bacia",
        "ambiente",
        "instalacao",
        "perfil_medidas",
        "producao_de_oleo_m3",
        "producao_de_gas_associado_mm3",
    )
)

RESUMO DAS DIVERGÊNCIAS:


ambiente,cad_ambiente,registros,pocos_distintos,primeira_competencia,ultima_competencia
TERRA,MAR,124,42,2025-01-01,2025-03-01


POÇOS ENVOLVIDOS — ATÉ 30:


poco,ambiente,cad_ambiente,estado,cad_estado,bacia,cad_bacia,cad_situacao,registros,campos_na_producao,campos_no_cadastro,primeira_competencia,ultima_competencia,data_referencia_situacao
3-RNS-141D-RN,TERRA,MAR,RIO GRANDE DO NORTE,RIO GRANDE DO NORTE,POTIGUAR,POTIGUAR,FECHADO,3,List(MACAU),List(MACAU),2025-01-01,2025-03-01,2025-11-05
3-RNS-142D-RN,TERRA,MAR,RIO GRANDE DO NORTE,RIO GRANDE DO NORTE,POTIGUAR,POTIGUAR,INJETANDO,3,List(MACAU),List(MACAU),2025-01-01,2025-03-01,2025-11-09
7-C-173D-BA,TERRA,MAR,BAHIA,BAHIA,RECÔNCAVO,RECÔNCAVO,PRODUZINDO,3,List(CANDEIAS),List(CANDEIAS),2025-01-01,2025-03-01,2026-06-19
7-DJM-854H-BAS,TERRA,MAR,BAHIA,BAHIA,RECÔNCAVO,RECÔNCAVO,PRODUZINDO,3,List(DOM JOÃO),List(DOM JOÃO),2025-01-01,2025-03-01,2024-12-19
7-MA-31D-RNS,TERRA,MAR,RIO GRANDE DO NORTE,RIO GRANDE DO NORTE,POTIGUAR,POTIGUAR,PRODUZINDO,3,List(MACAU),List(MACAU),2025-01-01,2025-03-01,2026-08-15
7-SER-10D-RNS,TERRA,MAR,RIO GRANDE DO NORTE,RIO GRANDE DO NORTE,POTIGUAR,POTIGUAR,INJETANDO,3,List(MACAU),List(MACAU),2025-01-01,2025-03-01,2025-11-09
7-SER-11D-RNS,TERRA,MAR,RIO GRANDE DO NORTE,RIO GRANDE DO NORTE,POTIGUAR,POTIGUAR,INJETANDO,3,List(MACAU),List(MACAU),2025-01-01,2025-03-01,2025-11-09
7-SER-12D-RNS,TERRA,MAR,RIO GRANDE DO NORTE,RIO GRANDE DO NORTE,POTIGUAR,POTIGUAR,PRODUZINDO,3,List(MACAU),List(MACAU),2025-01-01,2025-03-01,2026-08-25
7-SER-14D-RNS,TERRA,MAR,RIO GRANDE DO NORTE,RIO GRANDE DO NORTE,POTIGUAR,POTIGUAR,PRODUZINDO,3,List(MACAU),List(MACAU),2025-01-01,2025-03-01,2026-06-03
7-SER-15D-RNS,TERRA,MAR,RIO GRANDE DO NORTE,RIO GRANDE DO NORTE,POTIGUAR,POTIGUAR,INJETANDO,3,List(MACAU),List(MACAU),2025-01-01,2025-03-01,2025-11-09


DETALHE DOS REGISTROS — ATÉ 20:


poco,competencia,campo,cad_campo,ambiente,cad_ambiente,instalacao,producao_de_oleo_m3,producao_de_gas_associado_mm3,perfil_medidas
3-RNS-141D-RN,2025-01-01,MACAU,MACAU,TERRA,MAR,ESTAÇÃO COLETORA DE MACAU,null,null,SOMENTE_INJECAO
3-RNS-141D-RN,2025-02-01,MACAU,MACAU,TERRA,MAR,ESTAÇÃO COLETORA DE MACAU,null,null,SOMENTE_INJECAO
3-RNS-141D-RN,2025-03-01,MACAU,MACAU,TERRA,MAR,ESTAÇÃO COLETORA DE MACAU,null,null,SOMENTE_INJECAO
3-RNS-142D-RN,2025-01-01,MACAU,MACAU,TERRA,MAR,ESTAÇÃO COLETORA DE MACAU,null,null,SOMENTE_INJECAO
3-RNS-142D-RN,2025-02-01,MACAU,MACAU,TERRA,MAR,ESTAÇÃO COLETORA DE MACAU,null,null,SOMENTE_INJECAO
3-RNS-142D-RN,2025-03-01,MACAU,MACAU,TERRA,MAR,ESTAÇÃO COLETORA DE MACAU,null,null,SOMENTE_INJECAO
7-C-173D-BA,2025-01-01,CANDEIAS,CANDEIAS,TERRA,MAR,ESTAÇÃO PEDRA BRANCA,114.043000000,92.013000000,SOMENTE_PRODUCAO
7-C-173D-BA,2025-02-01,CANDEIAS,CANDEIAS,TERRA,MAR,ESTAÇÃO PEDRA BRANCA,116.830000000,84.400000000,SOMENTE_PRODUCAO
7-C-173D-BA,2025-03-01,CANDEIAS,CANDEIAS,TERRA,MAR,ESTAÇÃO PEDRA BRANCA,110.397000000,87.358000000,SOMENTE_PRODUCAO
7-DJM-854H-BAS,2025-01-01,DOM JOÃO,DOM JOÃO,TERRA,MAR,ESTAÇÃO MARAPÉ,184.324000000,6.322000000,SOMENTE_PRODUCAO


PRODUÇÃO SEM CORRESPONDÊNCIA NO CADASTRO:


poco,competencia,campo,bacia,ambiente,instalacao,perfil_medidas,producao_de_oleo_m3,producao_de_gas_associado_mm3
4-GO-13-BA,2025-03-01,GOMO,RECÔNCAVO,TERRA,ESTAÇÃO GOMO,SOMENTE_INJECAO,null,null


In [0]:
from pyspark.sql import functions as F

SOURCE_RUN_ID = "1f0f7d61-7bae-4d59-af8a-316ab87ab334"

CONTEXT_VIEW = (
    "workspace.oil_gas_silver.producao_contexto_cadastral_piloto"
)
ISSUES_TABLE = (
    "workspace.oil_gas_ops.production_catalog_issues"
)

required = {"producao_mar", "producao_terra", "situacao_pocos"}

audit_rows = (
    spark.table("workspace.oil_gas_ops.ingestion_files")
    .filter(
        (F.col("run_id") == SOURCE_RUN_ID)
        & F.col("dataset").isin(*sorted(required))
        & F.col("status").isin("STORED", "REUSED")
    )
    .select("dataset", "file_sha256")
    .distinct()
    .collect()
)

hashes = {}

for row in audit_rows:
    dataset = row["dataset"]

    if dataset in hashes:
        raise RuntimeError(f"Mais de uma versão para {dataset}.")

    hashes[dataset] = row["file_sha256"]

if set(hashes) != required:
    raise RuntimeError("Faltam versões auditadas.")

import re

if not all(
    re.fullmatch(r"[0-9a-f]{64}", value)
    for value in hashes.values()
):
    raise RuntimeError("Hash de arquivo fora do formato esperado.")

cadastro_keys = (
    spark.table("workspace.oil_gas_silver.cadastro_pocos")
    .filter(
        F.col("_source_file_sha256") == hashes["situacao_pocos"]
    )
    .filter(F.col("nome_poco_anp").isNotNull())
)

if (
    cadastro_keys.groupBy("nome_poco_anp")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Cadastro não é único pelo identificador ANP.")

production_filter = f"""
    (
        p._source_dataset = 'producao_mar'
        AND p._source_file_sha256 = '{hashes["producao_mar"]}'
    )
    OR
    (
        p._source_dataset = 'producao_terra'
        AND p._source_file_sha256 = '{hashes["producao_terra"]}'
    )
"""

spark.sql(f"""
    CREATE OR REPLACE VIEW {CONTEXT_VIEW} AS
    SELECT
        p.*,
        c.ambiente AS ambiente_cadastro,
        '{hashes["situacao_pocos"]}' AS _cadastro_file_sha256,
        CASE
            WHEN p.poco IS NULL THEN 'SEM_ID_ANP'
            WHEN c.nome_poco_anp IS NULL
                THEN 'NAO_LOCALIZADO_NO_CADASTRO'
            ELSE 'LOCALIZADO'
        END AS status_vinculo_cadastro,
        CASE
            WHEN c.nome_poco_anp IS NULL THEN 'NAO_COMPARAVEL'
            WHEN p.ambiente <=> c.ambiente THEN 'COINCIDENTE'
            ELSE 'DIVERGENTE'
        END AS status_comparacao_ambiente
    FROM workspace.oil_gas_silver.producao_injecao_mensal p
    LEFT JOIN (
        SELECT nome_poco_anp, ambiente
        FROM workspace.oil_gas_silver.cadastro_pocos
        WHERE _source_file_sha256 = '{hashes["situacao_pocos"]}'
          AND nome_poco_anp IS NOT NULL
    ) c
        ON p.poco = c.nome_poco_anp
    WHERE {production_filter}
""")

context = spark.table(CONTEXT_VIEW)

source_count = (
    spark.table("workspace.oil_gas_silver.producao_injecao_mensal")
    .alias("p")
    .filter(F.expr(production_filter))
    .count()
)

if context.count() != source_count:
    raise RuntimeError("A view alterou a quantidade de linhas.")

issues = (
    context
    .withColumn(
        "regra",
        F.when(
            F.col("status_vinculo_cadastro") != "LOCALIZADO",
            "POCO_SEM_VINCULO_CADASTRAL",
        )
        .when(
            F.col("status_comparacao_ambiente") == "DIVERGENTE",
            "AMBIENTE_DIVERGENTE_ENTRE_FONTES",
        ),
    )
    .filter(F.col("regra").isNotNull())
    .select(
        "_source_dataset",
        "_source_file_sha256",
        "_cadastro_file_sha256",
        "registro_id",
        "poco",
        "competencia",
        "campo",
        "instalacao",
        F.col("ambiente").alias("ambiente_producao"),
        "ambiente_cadastro",
        "regra",
    )
    .withColumn(
        "ocorrencia_id",
        F.sha2(
            F.to_json(F.struct(
                "_source_file_sha256",
                "_cadastro_file_sha256",
                "registro_id",
                "regra",
            )),
            256,
        ),
    )
    .withColumn("severidade", F.lit("AVISO"))
    .withColumn("registrado_em", F.current_timestamp())
)

issues.createOrReplaceTempView("oil_gas_catalog_issues_batch")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {ISSUES_TABLE}
    USING DELTA
    COMMENT 'Ocorrências de vínculo e ambiente por versões das fontes.
    Avisos não excluem registros nem determinam qual fonte está correta.'
    AS
    SELECT *
    FROM oil_gas_catalog_issues_batch
    WHERE 1 = 0
""")

spark.sql(f"""
    MERGE INTO {ISSUES_TABLE} AS target
    USING oil_gas_catalog_issues_batch AS source
    ON target.ocorrencia_id = source.ocorrencia_id
    WHEN NOT MATCHED THEN INSERT *
""")

print(f"View persistente: {CONTEXT_VIEW}")
print(f"Tabela de ocorrências: {ISSUES_TABLE}")
print(f"Registros preservados na view: {source_count:,}")

print("\nOCORRÊNCIAS DESTE PILOTO:")
display(
    issues
    .groupBy("_source_dataset", "regra", "severidade")
    .agg(
        F.count("*").alias("ocorrencias"),
        F.countDistinct("poco").alias("pocos_distintos"),
    )
    .orderBy("_source_dataset", "regra")
)

View persistente: workspace.oil_gas_silver.producao_contexto_cadastral_piloto
Tabela de ocorrências: workspace.oil_gas_ops.production_catalog_issues
Registros preservados na view: 46,492

OCORRÊNCIAS DESTE PILOTO:


_source_dataset,regra,severidade,ocorrencias,pocos_distintos
producao_terra,AMBIENTE_DIVERGENTE_ENTRE_FONTES,AVISO,124,42
producao_terra,POCO_SEM_VINCULO_CADASTRAL,AVISO,1,1


In [0]:
from pyspark.sql import functions as F

SOURCE_RUN_ID = "1f0f7d61-7bae-4d59-af8a-316ab87ab334"

versions = (
    spark.table("workspace.oil_gas_ops.ingestion_files")
    .filter(
        (F.col("run_id") == SOURCE_RUN_ID)
        & (F.col("dataset") == "situacao_pocos")
        & F.col("status").isin("STORED", "REUSED")
    )
    .select("file_sha256")
    .distinct()
    .collect()
)

if len(versions) != 1:
    raise RuntimeError("Esperada uma versão auditada do cadastro.")

cadastro_geo_source_hash = versions[0]["file_sha256"]

cadastro_source = (
    spark.table("workspace.oil_gas_silver.cadastro_pocos")
    .filter(
        F.col("_source_file_sha256") == cadastro_geo_source_hash
    )
)

def parse_dms(column, degree_limit):
    raw = F.trim(F.col(column))
    value = F.when(F.length(raw) > 0, raw)

    pattern = r"^([+-]?)(\d{1,3}):(\d{1,2}):(\d{1,2}(?:[,.]\d+)?)$"

    format_ok = F.coalesce(
        value.rlike(pattern), F.lit(False)
    )

    sign = F.when(
        F.regexp_extract(value, pattern, 1) == "-",
        F.lit(-1.0),
    ).otherwise(F.lit(1.0))

    degrees = F.when(
        format_ok, F.regexp_extract(value, pattern, 2)
    ).cast("double")

    minutes = F.when(
        format_ok, F.regexp_extract(value, pattern, 3)
    ).cast("double")

    seconds = F.when(
        format_ok,
        F.regexp_replace(
            F.regexp_extract(value, pattern, 4), ",", "."
        ),
    ).cast("double")

    absolute_decimal = (
        degrees + minutes / 60.0 + seconds / 3600.0
    )

    bounds_ok = F.coalesce(
        (degrees <= degree_limit)
        & (minutes < 60)
        & (seconds < 60)
        & (absolute_decimal <= degree_limit),
        F.lit(False),
    )

    valid = format_ok & bounds_ok

    decimal_value = F.when(
        valid,
        F.round(sign * absolute_decimal, 8),
    )

    status = (
        F.when(value.isNull(), "AUSENTE")
        .when(~format_ok, "FORMATO_INVALIDO")
        .when(~bounds_ok, "FORA_DO_INTERVALO")
        .otherwise("VALIDA")
    )

    return decimal_value, status

latitude_decimal, latitude_status = parse_dms(
    "latitude_da_base", 90
)
longitude_decimal, longitude_status = parse_dms(
    "longitude_da_base", 180
)

cadastro_geo_preview = (
    cadastro_source
    .withColumn("latitude_decimal", latitude_decimal)
    .withColumn("longitude_decimal", longitude_decimal)
    .withColumn("status_latitude", latitude_status)
    .withColumn("status_longitude", longitude_status)
    .withColumn(
        "par_coordenadas_valido",
        (F.col("status_latitude") == "VALIDA")
        & (F.col("status_longitude") == "VALIDA"),
    )
)

print("QUALIDADE DAS COORDENADAS:")
display(
    cadastro_geo_preview
    .groupBy("status_latitude", "status_longitude")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("nome_poco_anp").alias(
            "pocos_com_id_distintos"
        ),
    )
    .orderBy("status_latitude", "status_longitude")
)

print("LIMITES OBSERVADOS — SOMENTE PARES VÁLIDOS:")
display(
    cadastro_geo_preview
    .filter(F.col("par_coordenadas_valido"))
    .agg(
        F.count("*").alias("registros_com_par_valido"),
        F.min("latitude_decimal").alias("menor_latitude"),
        F.max("latitude_decimal").alias("maior_latitude"),
        F.min("longitude_decimal").alias("menor_longitude"),
        F.max("longitude_decimal").alias("maior_longitude"),
    )
)

print("EXEMPLOS DE CONVERSÃO:")
display(
    cadastro_geo_preview
    .filter(F.col("par_coordenadas_valido"))
    .select(
        "nome_poco_anp",
        "latitude_da_base",
        "longitude_da_base",
        "latitude_decimal",
        "longitude_decimal",
    )
    .orderBy(F.col("nome_poco_anp").asc_nulls_last())
    .limit(5)
)

print("EXEMPLOS DE COORDENADAS AUSENTES OU INVÁLIDAS:")
display(
    cadastro_geo_preview
    .filter(~F.col("par_coordenadas_valido"))
    .select(
        "nome_poco_anp",
        "latitude_da_base",
        "longitude_da_base",
        "status_latitude",
        "status_longitude",
    )
    .orderBy(F.col("nome_poco_anp").asc_nulls_first())
    .limit(10)
)

QUALIDADE DAS COORDENADAS:


status_latitude,status_longitude,registros,pocos_com_id_distintos
AUSENTE,AUSENTE,143,32
VALIDA,VALIDA,27173,27172


LIMITES OBSERVADOS — SOMENTE PARES VÁLIDOS:


registros_com_par_valido,menor_latitude,maior_latitude,menor_longitude,maior_longitude
27173,-26.96998167,0.99536111,-66.51883472,-35.57581833


EXEMPLOS DE CONVERSÃO:


nome_poco_anp,latitude_da_base,longitude_da_base,latitude_decimal,longitude_decimal
1-A-1-BA,"-12:47:39,460","-38:26:10,974",-12.79429444,-38.43638167
1-ABV-1D-RN,"-05:25:05,784","-37:38:39,104",-5.41827333,-37.64419556
1-AG-1-BA,"-12:24:20,640","-38:22:00,094",-12.40573333,-38.36669278
1-AG-1-SE,"-10:41:12,530","-36:53:17,299",-10.68681389,-36.88813861
1-AG-5-SE,"-10:42:04,772","-36:52:32,982",-10.70132556,-36.87582833


EXEMPLOS DE COORDENADAS AUSENTES OU INVÁLIDAS:


nome_poco_anp,latitude_da_base,longitude_da_base,status_latitude,status_longitude
null,null,null,AUSENTE,AUSENTE
null,null,null,AUSENTE,AUSENTE
null,null,null,AUSENTE,AUSENTE
null,null,null,AUSENTE,AUSENTE
null,null,null,AUSENTE,AUSENTE
null,null,null,AUSENTE,AUSENTE
null,null,null,AUSENTE,AUSENTE
null,null,null,AUSENTE,AUSENTE
null,null,null,AUSENTE,AUSENTE
null,null,null,AUSENTE,AUSENTE


In [0]:
import uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F

if "cadastro_geo_preview" not in globals():
    raise RuntimeError("Execute novamente a célula 22 antes desta.")

GEO_TABLE = "workspace.oil_gas_silver.cadastro_pocos_georreferenciado"
GEO_AUDIT_TABLE = "workspace.oil_gas_ops.geo_quality_runs"
GEO_TRANSFORM_VERSION = "v1"

batch = (
    cadastro_geo_preview
    .withColumn(
        "_geo_transform_version",
        F.lit(GEO_TRANSFORM_VERSION),
    )
    .withColumn("_geo_loaded_at", F.current_timestamp())
)

hash_rows = (
    batch.select("_source_file_sha256")
    .distinct()
    .collect()
)

if len(hash_rows) != 1:
    raise RuntimeError("Esperada uma única versão do cadastro.")

source_hash = hash_rows[0]["_source_file_sha256"]

def geo_summary(df):
    return df.agg(
        F.count("*").alias("registros"),
        F.sum(
            F.when(F.col("par_coordenadas_valido"), 1).otherwise(0)
        ).cast("long").alias("pares_validos"),
        F.sum(
            F.when(
                (F.col("status_latitude") == "AUSENTE")
                & (F.col("status_longitude") == "AUSENTE"),
                1,
            ).otherwise(0)
        ).cast("long").alias("pares_ausentes"),
        F.sum(
            F.when(
                F.col("status_latitude").isin(
                    "FORMATO_INVALIDO", "FORA_DO_INTERVALO"
                )
                | F.col("status_longitude").isin(
                    "FORMATO_INVALIDO", "FORA_DO_INTERVALO"
                ),
                1,
            ).otherwise(0)
        ).cast("long").alias("registros_com_coordenada_invalida"),
    ).first().asDict()

expected = geo_summary(batch)

if expected["registros"] == 0:
    raise RuntimeError("Cadastro georreferenciado vazio.")

if (
    batch.groupBy("_source_file_sha256", "_source_record_sha256")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Registros repetidos na chave técnica.")

batch.createOrReplaceTempView("oil_gas_geo_batch")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {GEO_TABLE}
    USING DELTA
    COMMENT 'Coordenadas da base do poço convertidas de DMS.
    Preserva coordenadas originais e registros sem localização.
    Validade de formato e limites não comprova localização real.'
    AS
    SELECT *
    FROM oil_gas_geo_batch
    WHERE 1 = 0
""")

spark.sql(f"""
    MERGE INTO {GEO_TABLE} AS target
    USING oil_gas_geo_batch AS source
    ON target._source_file_sha256 = source._source_file_sha256
       AND target._source_record_sha256 = source._source_record_sha256
       AND target._geo_transform_version = source._geo_transform_version
    WHEN NOT MATCHED THEN INSERT *
""")

stored = (
    spark.table(GEO_TABLE)
    .filter(
        (F.col("_source_file_sha256") == source_hash)
        & (
            F.col("_geo_transform_version")
            == GEO_TRANSFORM_VERSION
        )
    )
)

actual = geo_summary(stored)

if actual != expected:
    raise RuntimeError(
        f"Reconciliação geográfica divergente. "
        f"Esperado: {expected}. Gravado: {actual}."
    )

audit_df = spark.createDataFrame(
    [(
        str(uuid.uuid4()),
        datetime.now(timezone.utc),
        source_hash,
        GEO_TRANSFORM_VERSION,
        actual["registros"],
        actual["pares_validos"],
        actual["pares_ausentes"],
        actual["registros_com_coordenada_invalida"],
        "PASS",
    )],
    schema="""
        quality_run_id STRING,
        checked_at TIMESTAMP,
        source_file_sha256 STRING,
        geo_transform_version STRING,
        registros LONG,
        pares_validos LONG,
        pares_ausentes LONG,
        registros_com_coordenada_invalida LONG,
        status STRING
    """,
)

(
    audit_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(GEO_AUDIT_TABLE)
)

print(f"Cadastro georreferenciado: {GEO_TABLE}")
print(f"Auditoria: {GEO_AUDIT_TABLE}")

display(
    audit_df.select(
        "registros",
        "pares_validos",
        "pares_ausentes",
        "registros_com_coordenada_invalida",
        "status",
    )
)

Cadastro georreferenciado: workspace.oil_gas_silver.cadastro_pocos_georreferenciado
Auditoria: workspace.oil_gas_ops.geo_quality_runs


registros,pares_validos,pares_ausentes,registros_com_coordenada_invalida,status
27316,27173,143,0,PASS


In [0]:
from datetime import date
from pyspark.sql import functions as F

SOURCE_RUN_ID = "1f0f7d61-7bae-4d59-af8a-316ab87ab334"
DATE_TABLE = "workspace.oil_gas_gold.dim_data"

versions = (
    spark.table("workspace.oil_gas_ops.ingestion_files")
    .filter(
        (F.col("run_id") == SOURCE_RUN_ID)
        & (F.col("dataset") == "intervencoes")
        & F.col("status").isin("STORED", "REUSED")
    )
    .select("file_sha256")
    .distinct()
    .collect()
)

if len(versions) != 1:
    raise RuntimeError("Esperada uma versão auditada de intervenções.")

interventions = (
    spark.table("workspace.oil_gas_silver.intervencoes")
    .filter(
        F.col("_source_file_sha256") == versions[0]["file_sha256"]
    )
)

production = spark.table(
    "workspace.oil_gas_silver.producao_contexto_cadastral_piloto"
)

reference_dates = (
    production.select(F.col("competencia").alias("data"))
    .unionByName(
        interventions.select(F.col("data_inicio").alias("data"))
    )
    .unionByName(
        interventions.select(F.col("data_termino").alias("data"))
    )
    .filter(F.col("data").isNotNull())
    .distinct()
)

bounds = reference_dates.agg(
    F.min("data").alias("inicio"),
    F.max("data").alias("fim"),
).first()

if bounds["inicio"] is None or bounds["fim"] is None:
    raise RuntimeError("Nenhuma data analítica disponível.")

calendar_start = date(bounds["inicio"].year, 1, 1)
calendar_end = date(bounds["fim"].year, 12, 31)
expected_days = (calendar_end - calendar_start).days + 1

calendar = spark.sql(f"""
    SELECT explode(
        sequence(
            DATE '{calendar_start.isoformat()}',
            DATE '{calendar_end.isoformat()}',
            INTERVAL 1 DAY
        )
    ) AS data
""")

month_names = F.array(*[
    F.lit(name) for name in [
        "JANEIRO", "FEVEREIRO", "MARÇO", "ABRIL",
        "MAIO", "JUNHO", "JULHO", "AGOSTO",
        "SETEMBRO", "OUTUBRO", "NOVEMBRO", "DEZEMBRO",
    ]
])

calendar = calendar.select(
    F.date_format("data", "yyyyMMdd").cast("int").alias("data_id"),
    "data",
    F.year("data").alias("ano"),
    F.quarter("data").alias("trimestre"),
    F.month("data").alias("mes"),
    F.element_at(
        month_names, F.month("data")
    ).alias("nome_mes"),
    F.date_format("data", "yyyyMM").cast("int").alias("mes_id"),
    F.dayofmonth("data").alias("dia"),
    F.trunc("data", "month").alias("primeiro_dia_mes"),
    F.last_day("data").alias("ultimo_dia_mes"),
    (F.dayofmonth("data") == 1).alias("eh_primeiro_dia_mes"),
    F.dayofweek("data").isin(1, 7).alias("eh_fim_de_semana"),
)

calendar.createOrReplaceTempView("oil_gas_calendar_batch")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DATE_TABLE}
    USING DELTA
    COMMENT 'Calendário diário. Competências mensais usam o primeiro
    dia do mês como referência, sem precisão diária do evento.'
    AS
    SELECT *
    FROM oil_gas_calendar_batch
    WHERE 1 = 0
""")

spark.sql(f"""
    MERGE INTO {DATE_TABLE} AS target
    USING oil_gas_calendar_batch AS source
    ON target.data_id = source.data_id
    WHEN NOT MATCHED THEN INSERT *
""")

stored_calendar = (
    spark.table(DATE_TABLE)
    .filter(F.col("data").between(calendar_start, calendar_end))
)

summary = stored_calendar.agg(
    F.count("*").alias("linhas"),
    F.countDistinct("data_id").alias("chaves_distintas"),
    F.countDistinct("data").alias("datas_distintas"),
).first()

if any(summary[c] != expected_days for c in summary.asDict()):
    raise RuntimeError("Calendário com lacunas ou duplicidades.")

missing_dates = (
    reference_dates
    .join(
        stored_calendar.select("data"),
        "data",
        "left_anti",
    )
    .count()
)

if missing_dates:
    raise RuntimeError("Existem datas analíticas fora do calendário.")

print(f"Tabela: {DATE_TABLE}")
print(f"Período: {calendar_start} até {calendar_end}")
print(f"Dias: {expected_days}")
print("Unicidade, continuidade e cobertura das datas: PASS")

display(
    stored_calendar.groupBy("ano")
    .agg(
        F.count("*").alias("dias"),
        F.min("data").alias("inicio"),
        F.max("data").alias("fim"),
    )
    .orderBy("ano")
)

display(stored_calendar.orderBy("data").limit(5))

Tabela: workspace.oil_gas_gold.dim_data
Período: 2025-01-01 até 2026-12-31
Dias: 730
Unicidade, continuidade e cobertura das datas: PASS


ano,dias,inicio,fim
2025,365,2025-01-01,2025-12-31
2026,365,2026-01-01,2026-12-31


data_id,data,ano,trimestre,mes,nome_mes,mes_id,dia,primeiro_dia_mes,ultimo_dia_mes,eh_primeiro_dia_mes,eh_fim_de_semana
20250101,2025-01-01,2025,1,1,JANEIRO,202501,1,2025-01-01,2025-01-31,true,false
20250102,2025-01-02,2025,1,1,JANEIRO,202501,2,2025-01-01,2025-01-31,false,false
20250103,2025-01-03,2025,1,1,JANEIRO,202501,3,2025-01-01,2025-01-31,false,false
20250104,2025-01-04,2025,1,1,JANEIRO,202501,4,2025-01-01,2025-01-31,false,true
20250105,2025-01-05,2025,1,1,JANEIRO,202501,5,2025-01-01,2025-01-31,false,true


In [0]:
from pyspark.sql import functions as F

SOURCE_RUN_ID = "1f0f7d61-7bae-4d59-af8a-316ab87ab334"
WELL_DIM_TABLE = "workspace.oil_gas_gold.dim_poco"

required = {
    "producao_mar", "producao_terra",
    "intervencoes", "situacao_pocos",
}

audit_rows = (
    spark.table("workspace.oil_gas_ops.ingestion_files")
    .filter(
        (F.col("run_id") == SOURCE_RUN_ID)
        & F.col("dataset").isin(*sorted(required))
        & F.col("status").isin("STORED", "REUSED")
    )
    .select("dataset", "file_sha256")
    .distinct()
    .collect()
)

hashes = {}

for row in audit_rows:
    if row["dataset"] in hashes:
        raise RuntimeError("Mais de uma versão para o mesmo conjunto.")
    hashes[row["dataset"]] = row["file_sha256"]

if set(hashes) != required:
    raise RuntimeError("Faltam versões auditadas.")

cadastro_ids = (
    spark.table("workspace.oil_gas_silver.cadastro_pocos")
    .filter(
        F.col("_source_file_sha256") == hashes["situacao_pocos"]
    )
    .select(
        F.lit("CADASTRO").alias("fonte"),
        "nome_poco_anp",
    )
)

intervention_ids = (
    spark.table("workspace.oil_gas_silver.intervencoes")
    .filter(
        F.col("_source_file_sha256") == hashes["intervencoes"]
    )
    .select(
        F.lit("INTERVENCOES").alias("fonte"),
        "nome_poco_anp",
    )
)

production_ids = (
    spark.table("workspace.oil_gas_silver.producao_injecao_mensal")
    .filter(
        (
            (F.col("_source_dataset") == "producao_mar")
            & (
                F.col("_source_file_sha256")
                == hashes["producao_mar"]
            )
        )
        | (
            (F.col("_source_dataset") == "producao_terra")
            & (
                F.col("_source_file_sha256")
                == hashes["producao_terra"]
            )
        )
    )
    .select(
        F.upper(F.col("_source_dataset")).alias("fonte"),
        F.col("poco").alias("nome_poco_anp"),
    )
)

identity_observations = (
    cadastro_ids
    .unionByName(intervention_ids)
    .unionByName(production_ids)
)

known_wells = (
    identity_observations
    .filter(F.col("nome_poco_anp").isNotNull())
    .select("nome_poco_anp")
    .distinct()
    .select(
        F.sha2("nome_poco_anp", 256).alias("poco_id"),
        "nome_poco_anp",
        F.lit("ID_ANP_INFORMADO").alias("tipo_membro"),
    )
)

unknown_member = spark.createDataFrame(
    [("SEM_ID_ANP", None, "SEM_ID_ANP")],
    schema="""
        poco_id STRING,
        nome_poco_anp STRING,
        tipo_membro STRING
    """,
)

well_dimension_batch = known_wells.unionByName(unknown_member)

if (
    well_dimension_batch.groupBy("poco_id")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Colisão ou repetição de chave na dimensão.")

if spark.catalog.tableExists(WELL_DIM_TABLE):
    incompatible = (
        spark.table(WELL_DIM_TABLE).alias("existing")
        .join(
            well_dimension_batch.alias("incoming"),
            F.col("existing.poco_id") == F.col("incoming.poco_id"),
            "inner",
        )
        .filter(
            ~F.col("existing.nome_poco_anp").eqNullSafe(
                F.col("incoming.nome_poco_anp")
            )
            | ~F.col("existing.tipo_membro").eqNullSafe(
                F.col("incoming.tipo_membro")
            )
        )
        .limit(1)
        .count()
    )

    if incompatible:
        raise RuntimeError("Chave existente com identidade incompatível.")

well_dimension_batch.createOrReplaceTempView("oil_gas_well_dim_batch")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {WELL_DIM_TABLE}
    USING DELTA
    COMMENT 'Identidade por nome ANP observado nas fontes.
    SEM_ID_ANP representa identificação ausente, não um poço físico.
    Atributos cadastrais permanecem separados.'
    AS
    SELECT *
    FROM oil_gas_well_dim_batch
    WHERE 1 = 0
""")

spark.sql(f"""
    MERGE INTO {WELL_DIM_TABLE} AS target
    USING oil_gas_well_dim_batch AS source
    ON target.poco_id = source.poco_id
    WHEN NOT MATCHED THEN INSERT *
""")

stored_dimension = spark.table(WELL_DIM_TABLE)

if (
    stored_dimension.groupBy("poco_id")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Dimensão persistida com chaves repetidas.")

missing_members = (
    well_dimension_batch.select("poco_id")
    .join(
        stored_dimension.select("poco_id"),
        "poco_id",
        "left_anti",
    )
    .count()
)

if missing_members:
    raise RuntimeError("Existem membros do piloto ausentes na dimensão.")

print(f"Dimensão: {WELL_DIM_TABLE}")
print("Unicidade e cobertura dos membros: PASS")

print("\nMEMBROS DESTE PILOTO:")
display(
    well_dimension_batch.groupBy("tipo_membro")
    .count()
    .orderBy("tipo_membro")
)

print("\nCOBERTURA POR FONTE — CONTAGENS NÃO ADITIVAS:")
display(
    identity_observations.groupBy("fonte")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("nome_poco_anp").alias(
            "identificadores_distintos"
        ),
        F.sum(
            F.when(F.col("nome_poco_anp").isNull(), 1).otherwise(0)
        ).cast("long").alias("registros_sem_identificador"),
    )
    .orderBy("fonte")
)

print("\nIDENTIFICADORES OBSERVADOS FORA DO CADASTRO:")
display(
    identity_observations
    .filter(F.col("nome_poco_anp").isNotNull())
    .groupBy("nome_poco_anp")
    .agg(
        F.sort_array(F.collect_set("fonte")).alias("fontes_no_piloto")
    )
    .filter(
        ~F.array_contains(F.col("fontes_no_piloto"), "CADASTRO")
    )
    .orderBy("nome_poco_anp")
)

Dimensão: workspace.oil_gas_gold.dim_poco
Unicidade e cobertura dos membros: PASS

MEMBROS DESTE PILOTO:


tipo_membro,count
ID_ANP_INFORMADO,27205
SEM_ID_ANP,1



COBERTURA POR FONTE — CONTAGENS NÃO ADITIVAS:


fonte,registros,identificadores_distintos,registros_sem_identificador
CADASTRO,27316,27204,112
INTERVENCOES,8065,4882,1
PRODUCAO_MAR,16690,958,0
PRODUCAO_TERRA,29802,10028,0



IDENTIFICADORES OBSERVADOS FORA DO CADASTRO:


nome_poco_anp,fontes_no_piloto
4-GO-13-BA,List(PRODUCAO_TERRA)


In [0]:
from pyspark.sql import functions as F

SOURCE_RUN_ID = "1f0f7d61-7bae-4d59-af8a-316ab87ab334"
FIELD_DIM_TABLE = "workspace.oil_gas_gold.dim_campo"

required = {
    "producao_mar", "producao_terra",
    "intervencoes", "situacao_pocos",
}

audit_rows = (
    spark.table("workspace.oil_gas_ops.ingestion_files")
    .filter(
        (F.col("run_id") == SOURCE_RUN_ID)
        & F.col("dataset").isin(*sorted(required))
        & F.col("status").isin("STORED", "REUSED")
    )
    .select("dataset", "file_sha256")
    .distinct()
    .collect()
)

pilot_source_hashes = {}

for row in audit_rows:
    if row["dataset"] in pilot_source_hashes:
        raise RuntimeError("Mais de uma versão para o mesmo conjunto.")
    pilot_source_hashes[row["dataset"]] = row["file_sha256"]

if set(pilot_source_hashes) != required:
    raise RuntimeError("Faltam versões auditadas.")

cadastro = (
    spark.table("workspace.oil_gas_silver.cadastro_pocos")
    .filter(
        F.col("_source_file_sha256")
        == pilot_source_hashes["situacao_pocos"]
    )
    .select(
        F.lit("CADASTRO").alias("fonte"),
        "campo", "bacia",
    )
)

interventions = (
    spark.table("workspace.oil_gas_silver.intervencoes")
    .filter(
        F.col("_source_file_sha256")
        == pilot_source_hashes["intervencoes"]
    )
    .select(
        F.lit("INTERVENCOES").alias("fonte"),
        "campo", "bacia",
    )
)

production = (
    spark.table("workspace.oil_gas_silver.producao_injecao_mensal")
    .filter(
        (
            (F.col("_source_dataset") == "producao_mar")
            & (
                F.col("_source_file_sha256")
                == pilot_source_hashes["producao_mar"]
            )
        )
        | (
            (F.col("_source_dataset") == "producao_terra")
            & (
                F.col("_source_file_sha256")
                == pilot_source_hashes["producao_terra"]
            )
        )
    )
    .select(
        F.upper("_source_dataset").alias("fonte"),
        "campo", "bacia",
    )
)

field_observations = (
    cadastro
    .unionByName(interventions)
    .unionByName(production)
    .select(
        "fonte",
        F.when(
            F.length(F.trim("campo")) > 0,
            F.upper(F.trim("campo")),
        ).alias("nome_campo_reportado"),
        F.when(
            F.length(F.trim("bacia")) > 0,
            F.upper(F.trim("bacia")),
        ).alias("bacia_reportada"),
    )
)

complete_context = (
    F.col("nome_campo_reportado").isNotNull()
    & F.col("bacia_reportada").isNotNull()
)

known_fields = (
    field_observations
    .filter(complete_context)
    .select("nome_campo_reportado", "bacia_reportada")
    .distinct()
    .select(
        F.sha2(
            F.to_json(F.struct(
                F.lit("CAMPO_REPORTADO_V1").alias("dominio"),
                "nome_campo_reportado",
                "bacia_reportada",
            )),
            256,
        ).alias("campo_id"),
        "nome_campo_reportado",
        "bacia_reportada",
        F.lit("CONTEXTO_INFORMADO").alias("tipo_membro"),
    )
)

unknown_field = spark.createDataFrame(
    [("SEM_CONTEXTO_CAMPO", None, None, "CONTEXTO_INCOMPLETO")],
    schema="""
        campo_id STRING,
        nome_campo_reportado STRING,
        bacia_reportada STRING,
        tipo_membro STRING
    """,
)

field_dimension_batch = known_fields.unionByName(unknown_field)

if (
    field_dimension_batch.groupBy("campo_id")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Colisão ou repetição de chave de campo.")

field_dimension_batch.createOrReplaceTempView(
    "oil_gas_field_dim_batch"
)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {FIELD_DIM_TABLE}
    USING DELTA
    COMMENT 'Contexto de campo reportado identificado por nome e bacia.
    Não representa um cadastro oficial por código ANP.
    Nomes distintos são preservados sem associação aproximada.'
    AS
    SELECT *
    FROM oil_gas_field_dim_batch
    WHERE 1 = 0
""")

spark.sql(f"""
    MERGE INTO {FIELD_DIM_TABLE} AS target
    USING oil_gas_field_dim_batch AS source
    ON target.campo_id = source.campo_id
    WHEN NOT MATCHED THEN INSERT *
""")

stored = spark.table(FIELD_DIM_TABLE)

if (
    stored.groupBy("campo_id")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Dimensão persistida com chaves repetidas.")

if (
    field_dimension_batch.select("campo_id")
    .join(stored.select("campo_id"), "campo_id", "left_anti")
    .limit(1)
    .count()
):
    raise RuntimeError("Contextos do piloto ausentes na dimensão.")

print(f"Dimensão: {FIELD_DIM_TABLE}")
print("Unicidade e cobertura dos membros: PASS")

print("\nMEMBROS DESTE PILOTO:")
display(
    field_dimension_batch.groupBy("tipo_membro")
    .count()
    .orderBy("tipo_membro")
)

print("\nCONTEXTOS INFORMADOS POR BACIA:")
display(
    known_fields.groupBy("bacia_reportada")
    .count()
    .orderBy(F.desc("count"), "bacia_reportada")
)

print("\nPENDÊNCIAS DE CONTEXTO POR FONTE:")
display(
    field_observations.groupBy("fonte")
    .agg(
        F.count("*").alias("registros"),
        F.sum(
            F.when(~complete_context, 1).otherwise(0)
        ).cast("long").alias("registros_com_contexto_incompleto"),
    )
    .orderBy("fonte")
)

Dimensão: workspace.oil_gas_gold.dim_campo
Unicidade e cobertura dos membros: PASS

MEMBROS DESTE PILOTO:


tipo_membro,count
CONTEXTO_INCOMPLETO,1
CONTEXTO_INFORMADO,539



CONTEXTOS INFORMADOS POR BACIA:


bacia_reportada,count
POTIGUAR,114
RECÔNCAVO,112
CAMPOS,69
ESPÍRITO SANTO,66
SERGIPE,50
SANTOS,49
ALAGOAS,21
PARNAÍBA,13
TUCANO SUL,10
SOLIMÕES,9



PENDÊNCIAS DE CONTEXTO POR FONTE:


fonte,registros,registros_com_contexto_incompleto
CADASTRO,27316,0
INTERVENCOES,8065,0
PRODUCAO_MAR,16690,0
PRODUCAO_TERRA,29802,0


In [0]:
from pyspark.sql import functions as F

SOURCE_RUN_ID = "1f0f7d61-7bae-4d59-af8a-316ab87ab334"
INSTALLATION_DIM_TABLE = "workspace.oil_gas_gold.dim_instalacao"

required = {"producao_mar", "producao_terra"}

audit_rows = (
    spark.table("workspace.oil_gas_ops.ingestion_files")
    .filter(
        (F.col("run_id") == SOURCE_RUN_ID)
        & F.col("dataset").isin(*sorted(required))
        & F.col("status").isin("STORED", "REUSED")
    )
    .select("dataset", "file_sha256")
    .distinct()
    .collect()
)

hashes = {}

for row in audit_rows:
    if row["dataset"] in hashes:
        raise RuntimeError("Mais de uma versão para o mesmo conjunto.")
    hashes[row["dataset"]] = row["file_sha256"]

if set(hashes) != required:
    raise RuntimeError("Faltam versões auditadas da produção.")

production = (
    spark.table("workspace.oil_gas_silver.producao_injecao_mensal")
    .filter(
        (
            (F.col("_source_dataset") == "producao_mar")
            & (F.col("_source_file_sha256") == hashes["producao_mar"])
        )
        | (
            (F.col("_source_dataset") == "producao_terra")
            & (F.col("_source_file_sha256") == hashes["producao_terra"])
        )
    )
)

def context_text(column):
    value = F.trim(F.col(column))
    return F.when(
        F.length(value) > 0,
        F.upper(value),
    ).otherwise(F.lit(None))

installation_observations = production.select(
    "_source_dataset",
    context_text("instalacao").alias("nome_instalacao_reportado"),
    context_text("ambiente").alias("ambiente_reportado"),
    context_text("estado").alias("estado_reportado"),
    context_text("bacia").alias("bacia_reportada"),
)

context_columns = [
    "nome_instalacao_reportado",
    "ambiente_reportado",
    "estado_reportado",
    "bacia_reportada",
]

complete_context = F.lit(True)
for column in context_columns:
    complete_context = complete_context & F.col(column).isNotNull()

known_installations = (
    installation_observations
    .filter(complete_context)
    .select(*context_columns)
    .distinct()
    .select(
        F.sha2(
            F.to_json(F.struct(
                F.lit("INSTALACAO_REPORTADA_V1").alias("dominio"),
                *[F.col(c) for c in context_columns],
            )),
            256,
        ).alias("instalacao_id"),
        *context_columns,
        F.lit("CONTEXTO_INFORMADO").alias("tipo_membro"),
    )
)

unknown_installation = spark.createDataFrame(
    [(
        "SEM_CONTEXTO_INSTALACAO",
        None, None, None, None,
        "CONTEXTO_INCOMPLETO",
    )],
    schema="""
        instalacao_id STRING,
        nome_instalacao_reportado STRING,
        ambiente_reportado STRING,
        estado_reportado STRING,
        bacia_reportada STRING,
        tipo_membro STRING
    """,
)

installation_batch = known_installations.unionByName(
    unknown_installation
)

if (
    installation_batch.groupBy("instalacao_id")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Colisão ou repetição de chave de instalação.")

installation_batch.createOrReplaceTempView(
    "oil_gas_installation_dim_batch"
)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {INSTALLATION_DIM_TABLE}
    USING DELTA
    COMMENT 'Instalações reportadas por nome, ambiente, estado e bacia.
    Chave contextual; não equivale a identificador físico oficial.'
    AS
    SELECT *
    FROM oil_gas_installation_dim_batch
    WHERE 1 = 0
""")

spark.sql(f"""
    MERGE INTO {INSTALLATION_DIM_TABLE} AS target
    USING oil_gas_installation_dim_batch AS source
    ON target.instalacao_id = source.instalacao_id
    WHEN NOT MATCHED THEN INSERT *
""")

stored = spark.table(INSTALLATION_DIM_TABLE)

if (
    stored.groupBy("instalacao_id")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Dimensão persistida com chaves repetidas.")

if (
    installation_batch.select("instalacao_id")
    .join(
        stored.select("instalacao_id"),
        "instalacao_id",
        "left_anti",
    )
    .limit(1)
    .count()
):
    raise RuntimeError("Contextos do piloto ausentes na dimensão.")

print(f"Dimensão: {INSTALLATION_DIM_TABLE}")
print("Unicidade e cobertura dos membros: PASS")

print("\nMEMBROS DESTE PILOTO:")
display(
    installation_batch.groupBy("tipo_membro")
    .count()
    .orderBy("tipo_membro")
)

print("\nCONTEXTOS POR AMBIENTE:")
display(
    known_installations.groupBy("ambiente_reportado")
    .count()
    .orderBy("ambiente_reportado")
)

print("\nPENDÊNCIAS POR FONTE:")
display(
    installation_observations.groupBy("_source_dataset")
    .agg(
        F.count("*").alias("registros"),
        F.sum(
            F.when(~complete_context, 1).otherwise(0)
        ).cast("long").alias("registros_com_contexto_incompleto"),
    )
    .orderBy("_source_dataset")
)

print("\nNOMES PRESENTES EM MAIS DE UM CONTEXTO:")
display(
    known_installations
    .groupBy("nome_instalacao_reportado")
    .agg(F.count("*").alias("contextos_distintos"))
    .filter(F.col("contextos_distintos") > 1)
    .orderBy(F.desc("contextos_distintos"), "nome_instalacao_reportado")
    .limit(20)
)

Dimensão: workspace.oil_gas_gold.dim_instalacao
Unicidade e cobertura dos membros: PASS

MEMBROS DESTE PILOTO:


tipo_membro,count
CONTEXTO_INCOMPLETO,1
CONTEXTO_INFORMADO,413



CONTEXTOS POR AMBIENTE:


ambiente_reportado,count
MAR,79
TERRA,334



PENDÊNCIAS POR FONTE:


_source_dataset,registros,registros_com_contexto_incompleto
producao_mar,16690,0
producao_terra,29802,0



NOMES PRESENTES EM MAIS DE UM CONTEXTO:


nome_instalacao_reportado,contextos_distintos


In [0]:
from functools import reduce
from pyspark.sql import functions as F

FACT_TABLE = "workspace.oil_gas_gold.fato_producao_injecao_mensal"
GOLD_VERSION = "v1"

source = spark.table(
    "workspace.oil_gas_silver.producao_contexto_cadastral_piloto"
)

expected_count = source.count()

if expected_count == 0:
    raise RuntimeError("Fonte do piloto vazia.")

measure_columns = [
    column for column in source.columns
    if column.startswith(("producao_de_", "injecao_de_"))
]

date_dimension = (
    spark.table("workspace.oil_gas_gold.dim_data")
    .select(F.col("data").alias("competencia"), "data_id")
)

well_dimension = (
    spark.table("workspace.oil_gas_gold.dim_poco")
    .filter(F.col("tipo_membro") == "ID_ANP_INFORMADO")
    .select(F.col("nome_poco_anp").alias("poco"), "poco_id")
)

field_dimension = (
    spark.table("workspace.oil_gas_gold.dim_campo")
    .filter(F.col("tipo_membro") == "CONTEXTO_INFORMADO")
    .select(
        F.col("nome_campo_reportado").alias("campo"),
        F.col("bacia_reportada").alias("bacia"),
        "campo_id",
    )
)

installation_dimension = (
    spark.table("workspace.oil_gas_gold.dim_instalacao")
    .filter(F.col("tipo_membro") == "CONTEXTO_INFORMADO")
    .select(
        F.col("nome_instalacao_reportado").alias("instalacao"),
        F.col("ambiente_reportado").alias("ambiente"),
        F.col("estado_reportado").alias("estado"),
        F.col("bacia_reportada").alias("bacia"),
        "instalacao_id",
    )
)

dimension_specs = [
    (date_dimension, ["competencia"], "data_id", None),
    (well_dimension, ["poco"], "poco_id", "SEM_ID_ANP"),
    (
        field_dimension,
        ["campo", "bacia"],
        "campo_id",
        "SEM_CONTEXTO_CAMPO",
    ),
    (
        installation_dimension,
        ["instalacao", "ambiente", "estado", "bacia"],
        "instalacao_id",
        "SEM_CONTEXTO_INSTALACAO",
    ),
]

linked = source

for dimension, natural_keys, foreign_key, unknown_key in dimension_specs:
    if (
        dimension.groupBy(*natural_keys)
        .count()
        .filter(F.col("count") > 1)
        .limit(1)
        .count()
    ):
        raise RuntimeError(
            f"Dimensão não é única pelas chaves de {foreign_key}."
        )

    linked = linked.join(dimension, natural_keys, "left")

    if unknown_key is not None:
        incomplete = reduce(
            lambda left, right: left | right,
            [F.col(column).isNull() for column in natural_keys],
        )

        
        linked = linked.withColumn(
            foreign_key,
            F.when(incomplete, F.lit(unknown_key))
            .otherwise(F.col(foreign_key)),
        )

foreign_keys = ["data_id", "poco_id", "campo_id", "instalacao_id"]

missing_foreign_key = reduce(
    lambda left, right: left | right,
    [F.col(column).isNull() for column in foreign_keys],
)

if linked.filter(missing_foreign_key).limit(1).count():
    raise RuntimeError("Existem chaves dimensionais sem correspondência.")

if linked.count() != expected_count:
    raise RuntimeError("Os cruzamentos alteraram a quantidade de linhas.")

fact_batch = (
    linked.select(
        "registro_id",
        *foreign_keys,
        *measure_columns,
        "perfil_medidas",
        "_source_dataset",
        "_source_file_sha256",
        "_source_url",
        "_source_file_path",
        "_source_collected_at",
        "_ingestion_run_id",
        F.col("_transform_version").alias("_silver_transform_version"),
    )
    .withColumn("_gold_transform_version", F.lit(GOLD_VERSION))
    .withColumn(
        "fato_id",
        F.sha2(
            F.to_json(F.struct(
                F.lit("FATO_PRODUCAO_INJECAO").alias("dominio"),
                "_source_file_sha256",
                "registro_id",
                "_gold_transform_version",
            )),
            256,
        ),
    )
    .withColumn("_gold_loaded_at", F.current_timestamp())
)

if (
    fact_batch.groupBy("fato_id")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Chave do fato repetida no lote.")

fact_batch.createOrReplaceTempView("oil_gas_production_fact_batch")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {FACT_TABLE}
    USING DELTA
    COMMENT 'Produção e injeção por poço, mês, campo e instalação.
    Preserva versões da fonte e unidades recebidas.
    Revisões distintas não devem ser somadas entre si.'
    AS
    SELECT *
    FROM oil_gas_production_fact_batch
    WHERE 1 = 0
""")

spark.sql(f"""
    MERGE INTO {FACT_TABLE} AS target
    USING oil_gas_production_fact_batch AS source
    ON target.fato_id = source.fato_id
    WHEN NOT MATCHED THEN INSERT *
""")

batch_versions = fact_batch.select(
    "_source_dataset",
    "_source_file_sha256",
    "_gold_transform_version",
).distinct()

stored = spark.table(FACT_TABLE).join(
    batch_versions,
    [
        "_source_dataset",
        "_source_file_sha256",
        "_gold_transform_version",
    ],
    "inner",
)

if stored.count() != expected_count:
    raise RuntimeError("Contagem do fato persistido divergente.")

if (
    stored.groupBy("fato_id")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Fato persistido com chaves repetidas.")

if (
    fact_batch.select("fato_id")
    .join(stored.select("fato_id"), "fato_id", "left_anti")
    .limit(1)
    .count()
):
    raise RuntimeError("Existem registros do lote ausentes no fato.")

comparison = fact_batch.alias("expected").join(
    stored.alias("actual"), "fato_id", "inner"
)

checked_columns = foreign_keys + measure_columns + ["perfil_medidas"]

different_values = reduce(
    lambda left, right: left | right,
    [
        ~F.col(f"expected.{column}").eqNullSafe(
            F.col(f"actual.{column}")
        )
        for column in checked_columns
    ],
)

if comparison.filter(different_values).limit(1).count():
    raise RuntimeError("Valores ou chaves divergentes após a gravação.")

print(f"Fato Gold: {FACT_TABLE}")
print("Contagens, chaves e valores conferidos: PASS")

display(
    stored.groupBy("_source_dataset", "perfil_medidas")
    .count()
    .orderBy("_source_dataset", "perfil_medidas")
)

display(
    stored.groupBy("_source_dataset")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("fato_id").alias("fatos_distintos"),
        F.countDistinct("poco_id").alias("pocos_distintos"),
        F.countDistinct("campo_id").alias("campos_distintos"),
        F.countDistinct("instalacao_id").alias("instalacoes_distintas"),
    )
    .orderBy("_source_dataset")
)

Fato Gold: workspace.oil_gas_gold.fato_producao_injecao_mensal
Contagens, chaves e valores conferidos: PASS


_source_dataset,perfil_medidas,count
producao_mar,PRODUCAO_E_INJECAO,8
producao_mar,SOMENTE_INJECAO,5790
producao_mar,SOMENTE_PRODUCAO,10892
producao_terra,PRODUCAO_E_INJECAO,4709
producao_terra,SOMENTE_INJECAO,6589
producao_terra,SOMENTE_PRODUCAO,18504


_source_dataset,registros,fatos_distintos,pocos_distintos,campos_distintos,instalacoes_distintas
producao_mar,16690,16690,958,72,79
producao_terra,29802,29802,10028,226,334


In [0]:
from functools import reduce
from pyspark.sql import functions as F

SOURCE_RUN_ID = "1f0f7d61-7bae-4d59-af8a-316ab87ab334"
FACT_TABLE = "workspace.oil_gas_gold.fato_intervencoes"
GOLD_VERSION = "v1"

versions = (
    spark.table("workspace.oil_gas_ops.ingestion_files")
    .filter(
        (F.col("run_id") == SOURCE_RUN_ID)
        & (F.col("dataset") == "intervencoes")
        & F.col("status").isin("STORED", "REUSED")
    )
    .select("file_sha256")
    .distinct()
    .collect()
)

if len(versions) != 1:
    raise RuntimeError("Esperada uma versão auditada de intervenções.")

source_hash = versions[0]["file_sha256"]

source = (
    spark.table("workspace.oil_gas_silver.intervencoes")
    .filter(F.col("_source_file_sha256") == source_hash)
)

expected_count = source.count()

if expected_count == 0:
    raise RuntimeError("Fonte de intervenções vazia.")

dates = spark.table("workspace.oil_gas_gold.dim_data")
wells = spark.table("workspace.oil_gas_gold.dim_poco")
fields = spark.table("workspace.oil_gas_gold.dim_campo")

# Confere os membros especiais antes de utilizá-los.
for dimension, key_column, special_key in [
    (wells, "poco_id", "SEM_ID_ANP"),
    (fields, "campo_id", "SEM_CONTEXTO_CAMPO"),
]:
    if dimension.filter(F.col(key_column) == special_key).count() != 1:
        raise RuntimeError(f"Membro especial ausente ou repetido: {special_key}")

dimension_specs = [
    (
        dates.select(
            F.col("data").alias("data_inicio"),
            F.col("data_id").alias("data_inicio_id"),
        ),
        ["data_inicio"],
        "data_inicio_id",
        None,
    ),
    (
        dates.select(
            F.col("data").alias("data_termino"),
            F.col("data_id").alias("data_termino_id"),
        ),
        ["data_termino"],
        "data_termino_id",
        None,
    ),
    (
        wells.filter(F.col("tipo_membro") == "ID_ANP_INFORMADO")
        .select("nome_poco_anp", "poco_id"),
        ["nome_poco_anp"],
        "poco_id",
        "SEM_ID_ANP",
    ),
    (
        fields.filter(F.col("tipo_membro") == "CONTEXTO_INFORMADO")
        .select(
            F.col("nome_campo_reportado").alias("campo"),
            F.col("bacia_reportada").alias("bacia"),
            "campo_id",
        ),
        ["campo", "bacia"],
        "campo_id",
        "SEM_CONTEXTO_CAMPO",
    ),
]

linked = source

for dimension, natural_keys, foreign_key, unknown_key in dimension_specs:
    if (
        dimension.groupBy(*natural_keys)
        .count()
        .filter(F.col("count") > 1)
        .limit(1)
        .count()
    ):
        raise RuntimeError(f"Dimensão não é única para {foreign_key}.")

    linked = linked.join(dimension, natural_keys, "left")

    if unknown_key is not None:
        incomplete = reduce(
            lambda left, right: left | right,
            [F.col(column).isNull() for column in natural_keys],
        )

        linked = linked.withColumn(
            foreign_key,
            F.when(incomplete, F.lit(unknown_key))
            .otherwise(F.col(foreign_key)),
        )

foreign_keys = [
    "data_inicio_id", "data_termino_id", "poco_id", "campo_id",
]

if linked.filter(
    reduce(
        lambda left, right: left | right,
        [F.col(column).isNull() for column in foreign_keys],
    )
).limit(1).count():
    raise RuntimeError("Existem chaves dimensionais sem correspondência.")

if linked.count() != expected_count:
    raise RuntimeError("O cruzamento alterou a quantidade de intervenções.")

business_attributes = [
    "nome_poco_operador",
    "ambiente",
    "sonda",
    "objetivo",
    "operador_epoca",
    "status_identificacao",
    "dias_em_intervencao",
    "dias_calendario_inclusivos",
]

fact_batch = (
    linked.select(
        *foreign_keys,
        *business_attributes,
        "_source_record_sha256",
        "_source_file_sha256",
        "_source_url",
        "_source_file_path",
        "_source_collected_at",
        "_ingestion_run_id",
        F.col("_transform_version").alias("_silver_transform_version"),
    )
    .withColumn("quantidade_intervencoes", F.lit(1))
    .withColumn("_gold_transform_version", F.lit(GOLD_VERSION))
    .withColumn(
        "fato_id",
        F.sha2(
            F.to_json(F.struct(
                F.lit("FATO_INTERVENCAO").alias("dominio"),
                "_source_file_sha256",
                "_source_record_sha256",
                "_gold_transform_version",
            )),
            256,
        ),
    )
    .withColumn("_gold_loaded_at", F.current_timestamp())
)

if (
    fact_batch.groupBy("fato_id")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Chave do fato repetida no lote.")

fact_batch.createOrReplaceTempView("oil_gas_intervention_fact_batch")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {FACT_TABLE}
    USING DELTA
    COMMENT 'Registro de intervenção por versão da fonte.
    Dias informados não representam perda comprovada de produção.
    Eventos podem se sobrepor; durações não equivalem a dias únicos.'
    AS
    SELECT *
    FROM oil_gas_intervention_fact_batch
    WHERE 1 = 0
""")

spark.sql(f"""
    MERGE INTO {FACT_TABLE} AS target
    USING oil_gas_intervention_fact_batch AS source
    ON target.fato_id = source.fato_id
    WHEN NOT MATCHED THEN INSERT *
""")

stored = (
    spark.table(FACT_TABLE)
    .filter(
        (F.col("_source_file_sha256") == source_hash)
        & (F.col("_gold_transform_version") == GOLD_VERSION)
    )
)

summary = stored.agg(
    F.count("*").alias("registros"),
    F.countDistinct("fato_id").alias("fatos_distintos"),
    F.sum("quantidade_intervencoes").alias("intervencoes"),
).first()

if any(summary[c] != expected_count for c in summary.asDict()):
    raise RuntimeError("Contagem ou unicidade do fato divergente.")

if (
    fact_batch.select("fato_id")
    .join(stored.select("fato_id"), "fato_id", "left_anti")
    .limit(1)
    .count()
):
    raise RuntimeError("Registros do lote ausentes no fato.")

comparison = fact_batch.alias("expected").join(
    stored.alias("actual"), "fato_id", "inner"
)

checked_columns = (
    foreign_keys + business_attributes + ["quantidade_intervencoes"]
)

different_values = reduce(
    lambda left, right: left | right,
    [
        ~F.col(f"expected.{column}").eqNullSafe(
            F.col(f"actual.{column}")
        )
        for column in checked_columns
    ],
)

if comparison.filter(different_values).limit(1).count():
    raise RuntimeError("Valores divergentes após a gravação.")

print(f"Fato Gold: {FACT_TABLE}")
print("Contagens, chaves e valores conferidos: PASS")

display(
    stored.groupBy("status_identificacao")
    .agg(
        F.count("*").alias("registros"),
        F.sum("quantidade_intervencoes").alias("intervencoes"),
        F.countDistinct("poco_id").alias("membros_dim_poco"),
    )
    .orderBy("status_identificacao")
)

display(
    stored.groupBy("ambiente", "objetivo")
    .agg(F.sum("quantidade_intervencoes").alias("intervencoes"))
    .orderBy("ambiente", F.desc("intervencoes"), "objetivo")
)

Fato Gold: workspace.oil_gas_gold.fato_intervencoes
Contagens, chaves e valores conferidos: PASS


status_identificacao,registros,intervencoes,membros_dim_poco
ID_ANP_INFORMADO,8064,8064,4882
SEM_ID_ANP,1,1,1


ambiente,objetivo,intervencoes
MAR,ABANDONO,326
MAR,PERFURAÇÃO,209
MAR,RESTAURAÇÃO,157
MAR,COMPLETAÇÃO,136
MAR,AVALIAÇÃO,5
MAR,TLD,2
TERRA,RESTAURAÇÃO,4582
TERRA,COMPLETAÇÃO,1828
TERRA,ABANDONO,435
TERRA,PERFURAÇÃO,331


In [0]:
from functools import reduce
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

SOURCE_RUN_ID = "1f0f7d61-7bae-4d59-af8a-316ab87ab334"
FACT_TABLE = "workspace.oil_gas_gold.fato_snapshot_cadastro"
GOLD_VERSION = "v1"
GEO_VERSION = "v1"

versions = (
    spark.table("workspace.oil_gas_ops.ingestion_files")
    .filter(
        (F.col("run_id") == SOURCE_RUN_ID)
        & (F.col("dataset") == "situacao_pocos")
        & F.col("status").isin("STORED", "REUSED")
    )
    .select("file_sha256")
    .distinct()
    .collect()
)

if len(versions) != 1:
    raise RuntimeError("Esperada uma versão auditada do cadastro.")

source_hash = versions[0]["file_sha256"]

source = (
    spark.table(
        "workspace.oil_gas_silver.cadastro_pocos_georreferenciado"
    )
    .filter(
        (F.col("_source_file_sha256") == source_hash)
        & (F.col("_geo_transform_version") == GEO_VERSION)
    )
    .withColumn(
        "data_coleta_versao_utc",
        F.to_date("_source_collected_at"),
    )
)

expected_count = source.count()

if expected_count == 0:
    raise RuntimeError("Versão georreferenciada não encontrada.")

dates = spark.table("workspace.oil_gas_gold.dim_data")
wells = spark.table("workspace.oil_gas_gold.dim_poco")
fields = spark.table("workspace.oil_gas_gold.dim_campo")

for dimension, key, member in [
    (wells, "poco_id", "SEM_ID_ANP"),
    (fields, "campo_id", "SEM_CONTEXTO_CAMPO"),
]:
    if dimension.filter(F.col(key) == member).count() != 1:
        raise RuntimeError(f"Membro especial ausente ou repetido: {member}")

dimension_specs = [
    (
        dates.select(
            F.col("data").alias("data_coleta_versao_utc"),
            F.col("data_id").alias("data_coleta_versao_id"),
        ),
        ["data_coleta_versao_utc"],
        "data_coleta_versao_id",
        None,
    ),
    (
        wells.filter(F.col("tipo_membro") == "ID_ANP_INFORMADO")
        .select("nome_poco_anp", "poco_id"),
        ["nome_poco_anp"],
        "poco_id",
        "SEM_ID_ANP",
    ),
    (
        fields.filter(F.col("tipo_membro") == "CONTEXTO_INFORMADO")
        .select(
            F.col("nome_campo_reportado").alias("campo"),
            F.col("bacia_reportada").alias("bacia"),
            "campo_id",
        ),
        ["campo", "bacia"],
        "campo_id",
        "SEM_CONTEXTO_CAMPO",
    ),
]

linked = source

for dimension, natural_keys, foreign_key, unknown_key in dimension_specs:
    if (
        dimension.groupBy(*natural_keys)
        .count()
        .filter(F.col("count") > 1)
        .limit(1)
        .count()
    ):
        raise RuntimeError(f"Dimensão não é única para {foreign_key}.")

    linked = linked.join(dimension, natural_keys, "left")

    if unknown_key is not None:
        incomplete = reduce(
            lambda left, right: left | right,
            [F.col(column).isNull() for column in natural_keys],
        )
        linked = linked.withColumn(
            foreign_key,
            F.when(incomplete, F.lit(unknown_key))
            .otherwise(F.col(foreign_key)),
        )

foreign_keys = ["data_coleta_versao_id", "poco_id", "campo_id"]

if linked.filter(
    reduce(
        lambda left, right: left | right,
        [F.col(column).isNull() for column in foreign_keys],
    )
).limit(1).count():
    raise RuntimeError("Existem chaves dimensionais sem correspondência.")

if linked.count() != expected_count:
    raise RuntimeError("Os cruzamentos alteraram a quantidade de registros.")

attribute_columns = [
    column for column in source.columns
    if not column.startswith("_")
    and column not in {
        "nome_poco_anp", "campo", "bacia", "data_coleta_versao_utc",
    }
]

attribute_names = {
    "ambiente": "ambiente_cadastro",
    "operador_atual": "operador_atual_observado",
    "situacao_do_poco": "situacao_observada",
}

fact_batch = (
    linked.select(
        *foreign_keys,
        *[
            F.col(column).alias(attribute_names.get(column, column))
            for column in attribute_columns
        ],
        "_source_record_sha256",
        "_source_file_sha256",
        "_source_url",
        "_source_file_path",
        "_source_collected_at",
        "_ingestion_run_id",
        "_geo_transform_version",
    )
    .withColumn("_gold_transform_version", F.lit(GOLD_VERSION))
    .withColumn(
        "snapshot_id",
        F.sha2(
            F.to_json(F.struct(
                "_source_file_sha256",
                "_geo_transform_version",
                "_gold_transform_version",
            )),
            256,
        ),
    )
    .withColumn(
        "fato_id",
        F.sha2(
            F.to_json(F.struct(
                F.lit("SNAPSHOT_CADASTRO").alias("dominio"),
                "snapshot_id",
                "_source_record_sha256",
            )),
            256,
        ),
    )
    .withColumn("quantidade_registros_cadastrais", F.lit(1))
    .withColumn("_gold_loaded_at", F.current_timestamp())
)

if (
    fact_batch.groupBy("fato_id")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Chave repetida no lote cadastral.")

fact_batch.createOrReplaceTempView("oil_gas_snapshot_fact_batch")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {FACT_TABLE}
    USING DELTA
    COMMENT 'Registro cadastral por versão de conteúdo recebida.
    Data de coleta não determina início de validade dos atributos.
    Registros sem ID ANP são preservados individualmente.
    Quantidades de snapshots distintos não devem ser somadas
    como quantidade de poços físicos.'
    AS
    SELECT *
    FROM oil_gas_snapshot_fact_batch
    WHERE 1 = 0
""")

spark.sql(f"""
    MERGE INTO {FACT_TABLE} AS target
    USING oil_gas_snapshot_fact_batch AS source
    ON target.fato_id = source.fato_id
    WHEN NOT MATCHED THEN INSERT *
""")

stored = (
    spark.table(FACT_TABLE)
    .filter(
        (F.col("_source_file_sha256") == source_hash)
        & (F.col("_geo_transform_version") == GEO_VERSION)
        & (F.col("_gold_transform_version") == GOLD_VERSION)
    )
)

summary = stored.agg(
    F.count("*").alias("registros"),
    F.countDistinct("fato_id").alias("fatos_distintos"),
    F.sum("quantidade_registros_cadastrais").alias("quantidade"),
).first()

if any(summary[c] != expected_count for c in summary.asDict()):
    raise RuntimeError("Contagem ou unicidade do fato divergente.")

if (
    fact_batch.select("fato_id")
    .join(stored.select("fato_id"), "fato_id", "left_anti")
    .limit(1)
    .count()
):
    raise RuntimeError("Registros ausentes no fato cadastral.")

comparison = fact_batch.alias("expected").join(
    stored.alias("actual"), "fato_id", "inner"
)

checked_columns = (
    foreign_keys
    + [attribute_names.get(c, c) for c in attribute_columns]
    + ["snapshot_id", "quantidade_registros_cadastrais"]
)

if comparison.filter(
    reduce(
        lambda left, right: left | right,
        [
            ~F.col(f"expected.{column}").eqNullSafe(
                F.col(f"actual.{column}")
            )
            for column in checked_columns
        ],
    )
).limit(1).count():
    raise RuntimeError("Valores divergentes após a gravação.")

print(f"Fato cadastral: {FACT_TABLE}")
print("Contagens, chaves e atributos conferidos: PASS")

display(
    stored.groupBy("status_identificacao")
    .count()
    .orderBy("status_identificacao")
)

display(
    stored.groupBy("par_coordenadas_valido")
    .count()
    .orderBy("par_coordenadas_valido")
)

display(
    stored.groupBy("data_coleta_versao_id")
    .agg(
        F.countDistinct("snapshot_id").alias("versoes_cadastrais"),
        F.count("*").alias("registros"),
    )
)

Fato cadastral: workspace.oil_gas_gold.fato_snapshot_cadastro
Contagens, chaves e atributos conferidos: PASS


status_identificacao,count
ID_ANP_INFORMADO,27204
SEM_ID_ANP,112


par_coordenadas_valido,count
false,143
true,27173


data_coleta_versao_id,versoes_cadastrais,registros
20261015,1,27316


In [0]:
from pyspark.sql import functions as F

SOURCE_RUN_ID = "1f0f7d61-7bae-4d59-af8a-316ab87ab334"
SCOPE_ID = "piloto_anp_v1"
SCOPE_TABLE = "workspace.oil_gas_ops.analytical_scope_sources"

required = {
    "producao_mar", "producao_terra",
    "intervencoes", "situacao_pocos",
}

audit_rows = (
    spark.table("workspace.oil_gas_ops.ingestion_files")
    .filter(
        (F.col("run_id") == SOURCE_RUN_ID)
        & F.col("dataset").isin(*sorted(required))
        & F.col("status").isin("STORED", "REUSED")
    )
    .select("dataset", "file_sha256")
    .distinct()
    .collect()
)

hashes = {}

for row in audit_rows:
    if row["dataset"] in hashes:
        raise RuntimeError("Mais de uma versão para o mesmo conjunto.")
    hashes[row["dataset"]] = row["file_sha256"]

if set(hashes) != required:
    raise RuntimeError("Faltam versões auditadas.")

scope_columns = [
    "escopo_id",
    "dataset",
    "source_file_sha256",
    "gold_transform_version",
    "geo_transform_version",
    "ingestion_run_id",
]

scope_batch = spark.createDataFrame(
    [
        (
            SCOPE_ID,
            dataset,
            hashes[dataset],
            "v1",
            "v1" if dataset == "situacao_pocos" else None,
            SOURCE_RUN_ID,
        )
        for dataset in sorted(required)
    ],
    schema="""
        escopo_id STRING,
        dataset STRING,
        source_file_sha256 STRING,
        gold_transform_version STRING,
        geo_transform_version STRING,
        ingestion_run_id STRING
    """,
)

scope_batch.createOrReplaceTempView("oil_gas_scope_batch")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SCOPE_TABLE}
    USING DELTA
    COMMENT 'Seleção explícita de arquivos e transformações por escopo.
    Escopos publicados não devem ser alterados silenciosamente.'
    AS
    SELECT *, current_timestamp() AS registrado_em
    FROM oil_gas_scope_batch
    WHERE 1 = 0
""")

existing = (
    spark.table(SCOPE_TABLE)
    .filter(F.col("escopo_id") == SCOPE_ID)
    .select(*scope_columns)
)

if existing.limit(1).count():
    changed = (
        existing.exceptAll(scope_batch).limit(1).count()
        or scope_batch.exceptAll(existing).limit(1).count()
    )

    if changed:
        raise RuntimeError(
            "O escopo já existe com outra seleção. "
            "Use um novo escopo para publicar essa alteração."
        )

spark.sql(f"""
    MERGE INTO {SCOPE_TABLE} AS target
    USING oil_gas_scope_batch AS source
    ON target.escopo_id = source.escopo_id
       AND target.dataset = source.dataset
       AND target.source_file_sha256 = source.source_file_sha256
    WHEN NOT MATCHED THEN INSERT (
        escopo_id,
        dataset,
        source_file_sha256,
        gold_transform_version,
        geo_transform_version,
        ingestion_run_id,
        registrado_em
    ) VALUES (
        source.escopo_id,
        source.dataset,
        source.source_file_sha256,
        source.gold_transform_version,
        source.geo_transform_version,
        source.ingestion_run_id,
        current_timestamp()
    )
""")

spark.sql(f"""
    CREATE OR REPLACE VIEW
        workspace.oil_gas_gold.vw_producao_piloto AS
    SELECT s.escopo_id, f.*
    FROM workspace.oil_gas_gold.fato_producao_injecao_mensal f
    JOIN {SCOPE_TABLE} s
      ON f._source_dataset = s.dataset
     AND f._source_file_sha256 = s.source_file_sha256
     AND f._gold_transform_version = s.gold_transform_version
    WHERE s.escopo_id = '{SCOPE_ID}'
      AND s.dataset IN ('producao_mar', 'producao_terra')
""")

spark.sql(f"""
    CREATE OR REPLACE VIEW
        workspace.oil_gas_gold.vw_intervencoes_piloto AS
    SELECT s.escopo_id, f.*
    FROM workspace.oil_gas_gold.fato_intervencoes f
    JOIN {SCOPE_TABLE} s
      ON f._source_file_sha256 = s.source_file_sha256
     AND f._gold_transform_version = s.gold_transform_version
    WHERE s.escopo_id = '{SCOPE_ID}'
      AND s.dataset = 'intervencoes'
""")

spark.sql(f"""
    CREATE OR REPLACE VIEW
        workspace.oil_gas_gold.vw_cadastro_piloto AS
    SELECT s.escopo_id, f.*
    FROM workspace.oil_gas_gold.fato_snapshot_cadastro f
    JOIN {SCOPE_TABLE} s
      ON f._source_file_sha256 = s.source_file_sha256
     AND f._gold_transform_version = s.gold_transform_version
     AND f._geo_transform_version = s.geo_transform_version
    WHERE s.escopo_id = '{SCOPE_ID}'
      AND s.dataset = 'situacao_pocos'
""")

expected_counts = {}

for dataset in sorted(required):
    expected_counts[dataset] = (
        spark.table(f"workspace.oil_gas_bronze.{dataset}")
        .filter(F.col("_source_file_sha256") == hashes[dataset])
        .count()
    )

view_specs = [
    (
        "workspace.oil_gas_gold.vw_producao_piloto",
        expected_counts["producao_mar"] + expected_counts["producao_terra"],
    ),
    (
        "workspace.oil_gas_gold.vw_intervencoes_piloto",
        expected_counts["intervencoes"],
    ),
    (
        "workspace.oil_gas_gold.vw_cadastro_piloto",
        expected_counts["situacao_pocos"],
    ),
]

validation_results = []

for view_name, expected in view_specs:
    result = spark.table(view_name).agg(
        F.count("*").alias("registros"),
        F.countDistinct("fato_id").alias("fatos_distintos"),
    ).first()

    if expected == 0 or any(
        result[column] != expected for column in result.asDict()
    ):
        raise RuntimeError(f"Contagem ou unicidade divergente: {view_name}")

    validation_results.append((
        view_name,
        expected,
        result["registros"],
        result["fatos_distintos"],
        "PASS",
    ))

if (
    spark.table("workspace.oil_gas_gold.vw_producao_piloto")
    .groupBy("poco_id", "data_id", "campo_id", "instalacao_id")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Há sobreposição do grão mensal no escopo.")

print(f"Escopo publicado: {SCOPE_ID}")

display(
    spark.table(SCOPE_TABLE)
    .filter(F.col("escopo_id") == SCOPE_ID)
    .select(*scope_columns)
    .orderBy("dataset")
)

display(
    spark.createDataFrame(
        validation_results,
        schema="""
            view STRING,
            registros_esperados LONG,
            registros_publicados LONG,
            fatos_distintos LONG,
            status STRING
        """,
    )
)

Escopo publicado: piloto_anp_v1


escopo_id,dataset,source_file_sha256,gold_transform_version,geo_transform_version,ingestion_run_id
piloto_anp_v1,intervencoes,0994f2ec9a1dea96b7f861bbb0c8a93cb25dfe6b431164d73d51f26ce9133361,v1,null,1f0f7d61-7bae-4d59-af8a-316ab87ab334
piloto_anp_v1,producao_mar,c78d1f56ade7ead05a19afe23f980d5c7ef2097616334a5802d149eddd858677,v1,null,1f0f7d61-7bae-4d59-af8a-316ab87ab334
piloto_anp_v1,producao_terra,363786979ea678c9be58638335255aafa05b0807fac63ec9286309846b83e2a0,v1,null,1f0f7d61-7bae-4d59-af8a-316ab87ab334
piloto_anp_v1,situacao_pocos,babba11ca5128d0e657efd01888af37ea6d57e4ed0aea387a71919eb70c3c197,v1,v1,1f0f7d61-7bae-4d59-af8a-316ab87ab334


view,registros_esperados,registros_publicados,fatos_distintos,status
workspace.oil_gas_gold.vw_producao_piloto,46492,46492,46492,PASS
workspace.oil_gas_gold.vw_intervencoes_piloto,8065,8065,8065,PASS
workspace.oil_gas_gold.vw_cadastro_piloto,27316,27316,27316,PASS


In [0]:
from pyspark.sql import functions as F

SOURCE_VIEW = "workspace.oil_gas_gold.vw_producao_piloto"
MONTHLY_VIEW = (
    "workspace.oil_gas_gold.vw_indicadores_producao_mensal_piloto"
)

source = spark.table(SOURCE_VIEW)

measure_columns = [
    column for column in source.columns
    if column.startswith(("producao_de_", "injecao_de_"))
]

if not measure_columns:
    raise RuntimeError("Nenhuma medida de produção ou injeção encontrada.")

production_columns = [
    column for column in measure_columns
    if column.startswith("producao_de_")
    and column != "producao_de_agua_m3"
]

positive_production_sql = " OR ".join(
    f"f.`{column}` > 0"
    for column in production_columns
)

measure_aggregations = ",\n".join(
    expression
    for column in measure_columns
    for expression in [
        f"SUM(f.`{column}`) AS `{column}`",
        f"COUNT(f.`{column}`) AS `preenchidos_{column}`",
    ]
)

monthly_sql = f"""
    SELECT
        f.escopo_id,
        f._source_dataset,
        d.primeiro_dia_mes AS competencia,
        d.mes_id,
        i.ambiente_reportado AS ambiente,

        COUNT(*) AS registros_fonte,

        COUNT(DISTINCT CASE
            WHEN f.poco_id <> 'SEM_ID_ANP'
            THEN f.poco_id
        END) AS pocos_com_id_distintos,

        COUNT(DISTINCT CASE
            WHEN f.poco_id <> 'SEM_ID_ANP'
             AND ({positive_production_sql})
            THEN f.poco_id
        END) AS pocos_com_producao_hidrocarbonetos_positiva,

        SUM(CASE
            WHEN f.perfil_medidas IN (
                'SOMENTE_PRODUCAO', 'PRODUCAO_E_INJECAO'
            ) THEN 1 ELSE 0
        END) AS registros_com_medidas_producao,

        SUM(CASE
            WHEN f.perfil_medidas IN (
                'SOMENTE_INJECAO', 'PRODUCAO_E_INJECAO'
            ) THEN 1 ELSE 0
        END) AS registros_com_medidas_injecao,

        {measure_aggregations}

    FROM {SOURCE_VIEW} f
    JOIN workspace.oil_gas_gold.dim_data d
      ON f.data_id = d.data_id
    JOIN workspace.oil_gas_gold.dim_instalacao i
      ON f.instalacao_id = i.instalacao_id

    GROUP BY
        f.escopo_id,
        f._source_dataset,
        d.primeiro_dia_mes,
        d.mes_id,
        i.ambiente_reportado
"""

monthly_candidate = spark.sql(monthly_sql)

source_count = source.count()

monthly_row_total = (
    monthly_candidate
    .agg(F.sum("registros_fonte").alias("registros"))
    .first()["registros"]
)

if monthly_row_total != source_count:
    raise RuntimeError(
        "A agregação mensal não preservou a contagem de origem."
    )

expected_expressions = []
actual_expressions = []

for index, column in enumerate(measure_columns):
    expected_expressions.extend([
        F.sum(column).alias(f"total_{index}"),
        F.count(column).alias(f"filled_{index}"),
    ])
    actual_expressions.extend([
        F.sum(column).alias(f"total_{index}"),
        F.sum(f"preenchidos_{column}").alias(f"filled_{index}"),
    ])

expected = {
    row["_source_dataset"]: row.asDict()
    for row in (
        source.groupBy("_source_dataset")
        .agg(*expected_expressions)
        .collect()
    )
}

actual = {
    row["_source_dataset"]: row.asDict()
    for row in (
        monthly_candidate.groupBy("_source_dataset")
        .agg(*actual_expressions)
        .collect()
    )
}

if set(expected) != set(actual):
    raise RuntimeError("Fontes divergentes na agregação mensal.")

validation_results = []

for dataset in sorted(expected):
    for index, column in enumerate(measure_columns):
        expected_total = expected[dataset][f"total_{index}"]
        actual_total = actual[dataset][f"total_{index}"]
        expected_filled = expected[dataset][f"filled_{index}"]
        actual_filled = actual[dataset][f"filled_{index}"]

        passed = (
            expected_total == actual_total
            and expected_filled == actual_filled
        )

        validation_results.append((
            dataset,
            column,
            expected_total,
            actual_total,
            expected_filled,
            actual_filled,
            "PASS" if passed else "FAIL",
        ))

if any(row[-1] != "PASS" for row in validation_results):
    raise RuntimeError("Reconciliação mensal reprovada. View não publicada.")

spark.sql(f"""
    CREATE OR REPLACE VIEW {MONTHLY_VIEW}
    COMMENT 'Indicadores mensais do escopo piloto.
    Somas dos valores reportados, nas unidades da fonte.
    Contagens distintas de poços não são aditivas entre meses.
    Ausência de mês na fonte não equivale a produção zero.'
    AS
    {monthly_sql}
""")

print(f"View publicada: {MONTHLY_VIEW}")
print(
    f"Linhas de origem preservadas: {source_count:,}. "
    f"Medidas reconciliadas: {len(validation_results)}."
)

print("\\nCOBERTURA PUBLICADA:")
display(
    monthly_candidate.groupBy("_source_dataset", "ambiente")
    .agg(
        F.count("*").alias("meses"),
        F.min("competencia").alias("primeira_competencia"),
        F.max("competencia").alias("ultima_competencia"),
        F.sum("registros_fonte").alias("registros_fonte"),
    )
    .orderBy("_source_dataset")
)

print("\\nINDICADORES MENSAIS:")
display(
    monthly_candidate.select(
        "_source_dataset",
        "competencia",
        "ambiente",
        "registros_fonte",
        "pocos_com_id_distintos",
        "pocos_com_producao_hidrocarbonetos_positiva",
        "producao_de_oleo_m3",
        "producao_de_condensado_m3",
        "producao_de_agua_m3",
    )
    .orderBy("_source_dataset", "competencia")
)

print("\\nRECONCILIAÇÃO:")
display(
    spark.createDataFrame(
        validation_results,
        schema="""
            dataset STRING,
            medida STRING,
            total_fato DECIMAL(38,9),
            total_mensal DECIMAL(38,9),
            preenchidos_fato LONG,
            preenchidos_mensal LONG,
            status STRING
        """,
    ).groupBy("dataset", "status").count()
)

View publicada: workspace.oil_gas_gold.vw_indicadores_producao_mensal_piloto
Linhas de origem preservadas: 46,492. Medidas reconciliadas: 26.
\nCOBERTURA PUBLICADA:


_source_dataset,ambiente,meses,primeira_competencia,ultima_competencia,registros_fonte
producao_mar,MAR,12,2025-01-01,2025-12-01,16690
producao_terra,TERRA,3,2025-01-01,2025-03-01,29802


\nINDICADORES MENSAIS:


_source_dataset,competencia,ambiente,registros_fonte,pocos_com_id_distintos,pocos_com_producao_hidrocarbonetos_positiva,producao_de_oleo_m3,producao_de_condensado_m3,producao_de_agua_m3
producao_mar,2025-01-01,MAR,1314,820,502,16546072.731000000,11656.604000000,10184885.750000000
producao_mar,2025-02-01,MAR,1322,824,496,15113233.310000000,10525.960000000,8233725.532000000
producao_mar,2025-03-01,MAR,1327,828,497,17408324.146000000,11078.164000000,9315788.468000000
producao_mar,2025-04-01,MAR,1353,843,507,16890502.111000000,11133.425000000,8716976.160000000
producao_mar,2025-05-01,MAR,1330,827,501,17678440.839000000,11547.254000000,8289394.056000000
producao_mar,2025-06-01,MAR,1334,830,507,17474632.083000000,10072.802000000,7598302.458000000
producao_mar,2025-07-01,MAR,1365,851,539,19047576.457000000,10974.255000000,9626630.113000000
producao_mar,2025-08-01,MAR,1464,865,537,18735832.632000000,12223.574000000,9159894.429000000
producao_mar,2025-09-01,MAR,1440,850,494,18218455.383000000,11615.755000000,8416078.868000000
producao_mar,2025-10-01,MAR,1491,881,522,19416091.790000000,11339.054000000,8415862.412000000


\nRECONCILIAÇÃO:


dataset,status,count
producao_mar,PASS,13
producao_terra,PASS,13


In [0]:
from pyspark.sql import functions as F

COVERAGE_VIEW = (
    "workspace.oil_gas_gold.vw_cobertura_intervencoes_piloto"
)

coverage_sql = """
WITH poco_mes AS (
    SELECT
        f.poco_id,
        d.primeiro_dia_mes AS competencia,
        COUNT(*) AS registros_fonte,
        MAX(CASE
            WHEN f.producao_de_oleo_m3 IS NOT NULL THEN 1 ELSE 0
        END) AS oleo_informado,
        MAX(CASE
            WHEN f.producao_de_oleo_m3 > 0 THEN 1 ELSE 0
        END) AS oleo_positivo
    FROM workspace.oil_gas_gold.vw_producao_piloto f
    JOIN workspace.oil_gas_gold.dim_data d
      ON f.data_id = d.data_id
    WHERE f.poco_id <> 'SEM_ID_ANP'
    GROUP BY f.poco_id, d.primeiro_dia_mes
),

eventos AS (
    SELECT
        f.escopo_id,
        f.fato_id,
        f.poco_id,
        f.objetivo,
        f.ambiente,
        inicio.data AS data_inicio,
        termino.data AS data_termino
    FROM workspace.oil_gas_gold.vw_intervencoes_piloto f
    JOIN workspace.oil_gas_gold.dim_data inicio
      ON f.data_inicio_id = inicio.data_id
    JOIN workspace.oil_gas_gold.dim_data termino
      ON f.data_termino_id = termino.data_id
),

janelas AS (
    SELECT
        fato_id,
        poco_id,
        'ANTES' AS fase,
        explode(sequence(
            add_months(trunc(data_inicio, 'MONTH'), -3),
            add_months(trunc(data_inicio, 'MONTH'), -1),
            INTERVAL 1 MONTH
        )) AS competencia
    FROM eventos

    UNION ALL

    SELECT
        fato_id,
        poco_id,
        'DEPOIS' AS fase,
        explode(sequence(
            add_months(trunc(data_termino, 'MONTH'), 1),
            add_months(trunc(data_termino, 'MONTH'), 3),
            INTERVAL 1 MONTH
        )) AS competencia
    FROM eventos
),

cobertura AS (
    SELECT
        j.fato_id,

        SUM(CASE
            WHEN j.fase = 'ANTES' AND p.registros_fonte IS NOT NULL
            THEN 1 ELSE 0
        END) AS meses_com_registro_antes,

        SUM(CASE
            WHEN j.fase = 'DEPOIS' AND p.registros_fonte IS NOT NULL
            THEN 1 ELSE 0
        END) AS meses_com_registro_depois,

        SUM(CASE
            WHEN j.fase = 'ANTES' AND p.oleo_informado = 1
            THEN 1 ELSE 0
        END) AS meses_com_oleo_informado_antes,

        SUM(CASE
            WHEN j.fase = 'DEPOIS' AND p.oleo_informado = 1
            THEN 1 ELSE 0
        END) AS meses_com_oleo_informado_depois,

        SUM(CASE
            WHEN j.fase = 'ANTES'
             AND p.oleo_informado = 1 AND p.oleo_positivo = 0
            THEN 1 ELSE 0
        END) AS meses_com_oleo_zero_reportado_antes,

        SUM(CASE
            WHEN j.fase = 'DEPOIS'
             AND p.oleo_informado = 1 AND p.oleo_positivo = 0
            THEN 1 ELSE 0
        END) AS meses_com_oleo_zero_reportado_depois

    FROM janelas j
    LEFT JOIN poco_mes p
      ON j.poco_id = p.poco_id
     AND j.competencia = p.competencia
    GROUP BY j.fato_id
)

SELECT
    e.*,
    3 AS meses_esperados_por_janela,
    c.meses_com_registro_antes,
    c.meses_com_registro_depois,
    c.meses_com_oleo_informado_antes,
    c.meses_com_oleo_informado_depois,
    c.meses_com_oleo_zero_reportado_antes,
    c.meses_com_oleo_zero_reportado_depois,
    CASE
        WHEN e.poco_id = 'SEM_ID_ANP'
            THEN 'SEM_IDENTIFICADOR'
        WHEN c.meses_com_oleo_informado_antes = 3
         AND c.meses_com_oleo_informado_depois = 3
            THEN 'OLEO_INFORMADO_NAS_DUAS_JANELAS'
        WHEN c.meses_com_oleo_informado_antes > 0
         AND c.meses_com_oleo_informado_depois > 0
            THEN 'DUAS_JANELAS_PARCIAIS'
        WHEN c.meses_com_oleo_informado_antes > 0
            THEN 'OLEO_INFORMADO_SOMENTE_ANTES'
        WHEN c.meses_com_oleo_informado_depois > 0
            THEN 'OLEO_INFORMADO_SOMENTE_DEPOIS'
        ELSE 'SEM_OLEO_INFORMADO_NAS_JANELAS'
    END AS status_cobertura
FROM eventos e
JOIN cobertura c ON e.fato_id = c.fato_id
"""

candidate = spark.sql(coverage_sql)

expected_count = spark.table(
    "workspace.oil_gas_gold.vw_intervencoes_piloto"
).count()

summary = candidate.agg(
    F.count("*").alias("registros"),
    F.countDistinct("fato_id").alias("eventos_distintos"),
).first()

if any(summary[c] != expected_count for c in summary.asDict()):
    raise RuntimeError("A análise perdeu ou multiplicou intervenções.")

count_columns = [
    column for column in candidate.columns
    if column.startswith("meses_com_")
]

invalid_count = F.lit(False)
for column in count_columns:
    invalid_count = (
        invalid_count
        | F.col(column).isNull()
        | ~F.col(column).between(0, 3)
    )

for phase in ["antes", "depois"]:
    invalid_count = (
        invalid_count
        | (
            F.col(f"meses_com_oleo_informado_{phase}")
            > F.col(f"meses_com_registro_{phase}")
        )
        | (
            F.col(f"meses_com_oleo_zero_reportado_{phase}")
            > F.col(f"meses_com_oleo_informado_{phase}")
        )
    )

if candidate.filter(invalid_count).limit(1).count():
    raise RuntimeError("Contagens de cobertura inconsistentes.")

spark.sql(f"""
    CREATE OR REPLACE VIEW {COVERAGE_VIEW}
    COMMENT 'Cobertura de observações de óleo em três meses completos
    antes do início e três após o término da intervenção.
    Ausência não equivale a zero. Cobertura não comprova completude
    física do poço nem efeito causal da intervenção.'
    AS
    {coverage_sql}
""")

print(f"View publicada: {COVERAGE_VIEW}")
print(f"Intervenções preservadas: {expected_count:,} | PASS")

print("\\nCOBERTURA POR AMBIENTE:")
display(
    candidate.groupBy("ambiente", "status_cobertura")
    .count()
    .orderBy("ambiente", F.desc("count"), "status_cobertura")
)

print("\\nCOBERTURA POR OBJETIVO:")
display(
    candidate.groupBy("objetivo", "status_cobertura")
    .count()
    .orderBy("objetivo", "status_cobertura")
)

print("\\nEXEMPLOS COM AS DUAS JANELAS INFORMADAS:")
display(
    candidate
    .filter(
        F.col("status_cobertura")
        == "OLEO_INFORMADO_NAS_DUAS_JANELAS"
    )
    .select(
        "fato_id",
        "poco_id",
        "objetivo",
        "ambiente",
        "data_inicio",
        "data_termino",
        "meses_com_oleo_informado_antes",
        "meses_com_oleo_informado_depois",
        "meses_com_oleo_zero_reportado_antes",
        "meses_com_oleo_zero_reportado_depois",
    )
    .orderBy("data_inicio", "fato_id")
    .limit(5)
)

View publicada: workspace.oil_gas_gold.vw_cobertura_intervencoes_piloto
Intervenções preservadas: 8,065 | PASS
\nCOBERTURA POR AMBIENTE:


ambiente,status_cobertura,count
MAR,SEM_OLEO_INFORMADO_NAS_JANELAS,728
MAR,DUAS_JANELAS_PARCIAIS,29
MAR,OLEO_INFORMADO_SOMENTE_DEPOIS,29
MAR,OLEO_INFORMADO_NAS_DUAS_JANELAS,26
MAR,OLEO_INFORMADO_SOMENTE_ANTES,23
TERRA,SEM_OLEO_INFORMADO_NAS_JANELAS,5900
TERRA,OLEO_INFORMADO_SOMENTE_ANTES,876
TERRA,OLEO_INFORMADO_SOMENTE_DEPOIS,321
TERRA,DUAS_JANELAS_PARCIAIS,132
TERRA,SEM_IDENTIFICADOR,1


\nCOBERTURA POR OBJETIVO:


objetivo,status_cobertura,count
ABANDONO,DUAS_JANELAS_PARCIAIS,3
ABANDONO,OLEO_INFORMADO_NAS_DUAS_JANELAS,1
ABANDONO,OLEO_INFORMADO_SOMENTE_ANTES,6
ABANDONO,OLEO_INFORMADO_SOMENTE_DEPOIS,3
ABANDONO,SEM_OLEO_INFORMADO_NAS_JANELAS,748
AVALIAÇÃO,DUAS_JANELAS_PARCIAIS,6
AVALIAÇÃO,OLEO_INFORMADO_SOMENTE_ANTES,7
AVALIAÇÃO,OLEO_INFORMADO_SOMENTE_DEPOIS,3
AVALIAÇÃO,SEM_OLEO_INFORMADO_NAS_JANELAS,41
COMPLETAÇÃO,DUAS_JANELAS_PARCIAIS,16


\nEXEMPLOS COM AS DUAS JANELAS INFORMADAS:


fato_id,poco_id,objetivo,ambiente,data_inicio,data_termino,meses_com_oleo_informado_antes,meses_com_oleo_informado_depois,meses_com_oleo_zero_reportado_antes,meses_com_oleo_zero_reportado_depois
23aec1f788e5c4b337fed4ef094ab5fb2b9b2dc9bc206608599edc8ef64c4b2f,fe1c0f5da62eb098ce352a8283669d8e63859ceea30d897492ed676e7ad5eafb,RESTAURAÇÃO,MAR,2025-04-05,2025-04-24,3,3,3,0
abf243ee7408dffdb7a2b631ef03dfc03b658ff19a314127dc7997a622f1832e,1920e286cd188752dc1525a5371532866695eb6b0aebf04a90be3da51627b8b6,RESTAURAÇÃO,MAR,2025-04-09,2025-05-03,3,3,3,0
f31b3955f906ca3a04c4bf4d33635afc50150218bd968e52bc1a306e8495d11e,8be611eb10f1ae8b1f2c8c0601597e6430064a584ec40e3d4a058d103165c547,RESTAURAÇÃO,MAR,2025-04-11,2025-04-17,3,3,3,3
93628f7ecfc84fd7be3d4d703d75026dfc32c2994e5feecebd873f9178009bdf,02170aff2ca012591bce5d886ceaa9de2287137dccedcbe9ecfee6c5fc5cb69d,RESTAURAÇÃO,MAR,2025-04-24,2025-05-12,3,3,3,1
c3c660d983fff70942fda288cf8866d754738ed038a9b907ba4d8c095e3b681a,b3bbaeeff215bef05e391c1b022778d9db4029296ef8dcccd2ccce5aba248ec1,RESTAURAÇÃO,MAR,2025-04-27,2025-05-16,3,3,3,0


In [0]:
from pyspark.sql import functions as F

COVERAGE_VIEW = (
    "workspace.oil_gas_gold.vw_cobertura_intervencoes_piloto"
)
EVENT_CONTEXT_VIEW = (
    "workspace.oil_gas_gold.vw_contexto_intervencoes_cobertas_piloto"
)

context_sql = f"""
WITH alvos AS (
    SELECT
        *,
        add_months(
            trunc(data_inicio, 'MONTH'), -3
        ) AS janela_antes_inicio,
        date_sub(
            trunc(data_inicio, 'MONTH'), 1
        ) AS janela_antes_fim,
        add_months(
            trunc(data_termino, 'MONTH'), 1
        ) AS janela_depois_inicio,
        last_day(add_months(
            trunc(data_termino, 'MONTH'), 3
        )) AS janela_depois_fim
    FROM {COVERAGE_VIEW}
    WHERE status_cobertura = 'OLEO_INFORMADO_NAS_DUAS_JANELAS'
),

outros_eventos AS (
    SELECT
        f.fato_id,
        f.poco_id,
        inicio.data AS data_inicio,
        termino.data AS data_termino
    FROM workspace.oil_gas_gold.vw_intervencoes_piloto f
    JOIN workspace.oil_gas_gold.dim_data inicio
      ON f.data_inicio_id = inicio.data_id
    JOIN workspace.oil_gas_gold.dim_data termino
      ON f.data_termino_id = termino.data_id
    WHERE f.poco_id <> 'SEM_ID_ANP'
),

sobreposicoes AS (
    SELECT
        a.fato_id,

        COUNT(DISTINCT b.fato_id) AS outros_eventos_observados,

        COUNT(DISTINCT CASE
            WHEN b.data_inicio <= a.janela_antes_fim
             AND b.data_termino >= a.janela_antes_inicio
            THEN b.fato_id
        END) AS outros_eventos_antes,

        COUNT(DISTINCT CASE
            WHEN b.data_inicio <= a.data_termino
             AND b.data_termino >= a.data_inicio
            THEN b.fato_id
        END) AS outros_eventos_durante,

        COUNT(DISTINCT CASE
            WHEN b.data_inicio <= a.janela_depois_fim
             AND b.data_termino >= a.janela_depois_inicio
            THEN b.fato_id
        END) AS outros_eventos_depois,

        sort_array(
            collect_set(b.fato_id)
        ) AS outros_eventos_ids

    FROM alvos a
    LEFT JOIN outros_eventos b
      ON a.poco_id = b.poco_id
     AND a.fato_id <> b.fato_id
     AND (
        (
            b.data_inicio <= a.janela_antes_fim
            AND b.data_termino >= a.janela_antes_inicio
        )
        OR (
            b.data_inicio <= a.data_termino
            AND b.data_termino >= a.data_inicio
        )
        OR (
            b.data_inicio <= a.janela_depois_fim
            AND b.data_termino >= a.janela_depois_inicio
        )
     )
    GROUP BY a.fato_id
)

SELECT
    a.*,
    s.outros_eventos_observados,
    s.outros_eventos_antes,
    s.outros_eventos_durante,
    s.outros_eventos_depois,
    s.outros_eventos_ids,
    CASE
        WHEN s.outros_eventos_observados = 0
            THEN 'SEM_OUTRA_INTERVENCAO_OBSERVADA'
        ELSE 'COM_OUTRA_INTERVENCAO_OBSERVADA'
    END AS status_contexto,
    CASE
        WHEN a.objetivo <> 'RESTAURAÇÃO'
            THEN 'OUTRO_OBJETIVO'
        WHEN s.outros_eventos_observados > 0
            THEN 'RESTAURACAO_COM_OUTROS_EVENTOS'
        ELSE 'RESTAURACAO_SEM_OUTRO_EVENTO_OBSERVADO'
    END AS grupo_analise
FROM alvos a
JOIN sobreposicoes s ON a.fato_id = s.fato_id
"""

candidate = spark.sql(context_sql)

expected_count = (
    spark.table(COVERAGE_VIEW)
    .filter(
        F.col("status_cobertura")
        == "OLEO_INFORMADO_NAS_DUAS_JANELAS"
    )
    .count()
)

summary = candidate.agg(
    F.count("*").alias("registros"),
    F.countDistinct("fato_id").alias("eventos_distintos"),
).first()

if any(summary[c] != expected_count for c in summary.asDict()):
    raise RuntimeError("A análise perdeu ou multiplicou eventos-alvo.")

phase_columns = [
    "outros_eventos_antes",
    "outros_eventos_durante",
    "outros_eventos_depois",
]

invalid_counts = (
    F.col("outros_eventos_observados").isNull()
    | (F.col("outros_eventos_observados") < 0)
    | (
        F.size("outros_eventos_ids")
        != F.col("outros_eventos_observados")
    )
)

for column in phase_columns:
    invalid_counts = (
        invalid_counts
        | F.col(column).isNull()
        | (F.col(column) < 0)
        | (
            F.col(column)
            > F.col("outros_eventos_observados")
        )
    )

if candidate.filter(invalid_counts).limit(1).count():
    raise RuntimeError("Contagens de outras intervenções inconsistentes.")

spark.sql(f"""
    CREATE OR REPLACE VIEW {EVENT_CONTEXT_VIEW}
    COMMENT 'Outras intervenções observadas no mesmo poço nas janelas
    de comparação e durante o evento. Ausência de outro evento no
    escopo não comprova isolamento nem efeito causal.'
    AS
    {context_sql}
""")

print(f"View publicada: {EVENT_CONTEXT_VIEW}")
print(f"Eventos-alvo preservados: {expected_count} | PASS")

print("\\nGRUPOS PARA ANÁLISE:")
display(
    candidate.groupBy("objetivo", "status_contexto", "grupo_analise")
    .count()
    .orderBy("objetivo", "grupo_analise")
)

print("\\nEVENTOS E SUAS JANELAS:")
display(
    candidate.alias("e")
    .join(
        spark.table("workspace.oil_gas_gold.dim_poco").alias("p"),
        F.col("e.poco_id") == F.col("p.poco_id"),
        "left",
    )
    .select(
        F.col("p.nome_poco_anp"),
        F.col("e.fato_id"),
        F.col("e.objetivo"),
        F.col("e.data_inicio"),
        F.col("e.data_termino"),
        F.col("e.janela_antes_inicio"),
        F.col("e.janela_antes_fim"),
        F.col("e.janela_depois_inicio"),
        F.col("e.janela_depois_fim"),
        F.col("e.outros_eventos_observados"),
        F.col("e.outros_eventos_antes"),
        F.col("e.outros_eventos_durante"),
        F.col("e.outros_eventos_depois"),
        F.col("e.grupo_analise"),
    )
    .orderBy("data_inicio", "fato_id")
)

View publicada: workspace.oil_gas_gold.vw_contexto_intervencoes_cobertas_piloto
Eventos-alvo preservados: 26 | PASS
\nGRUPOS PARA ANÁLISE:


objetivo,status_contexto,grupo_analise,count
ABANDONO,SEM_OUTRA_INTERVENCAO_OBSERVADA,OUTRO_OBJETIVO,1
RESTAURAÇÃO,COM_OUTRA_INTERVENCAO_OBSERVADA,RESTAURACAO_COM_OUTROS_EVENTOS,5
RESTAURAÇÃO,SEM_OUTRA_INTERVENCAO_OBSERVADA,RESTAURACAO_SEM_OUTRO_EVENTO_OBSERVADO,20


\nEVENTOS E SUAS JANELAS:


nome_poco_anp,fato_id,objetivo,data_inicio,data_termino,janela_antes_inicio,janela_antes_fim,janela_depois_inicio,janela_depois_fim,outros_eventos_observados,outros_eventos_antes,outros_eventos_durante,outros_eventos_depois,grupo_analise
7-CRP-59D-RJS,23aec1f788e5c4b337fed4ef094ab5fb2b9b2dc9bc206608599edc8ef64c4b2f,RESTAURAÇÃO,2025-04-05,2025-04-24,2025-01-01,2025-03-31,2025-05-01,2025-07-31,0,0,0,0,RESTAURACAO_SEM_OUTRO_EVENTO_OBSERVADO
7-PRG-60H-RJS,abf243ee7408dffdb7a2b631ef03dfc03b658ff19a314127dc7997a622f1832e,RESTAURAÇÃO,2025-04-09,2025-05-03,2025-01-01,2025-03-31,2025-06-01,2025-08-31,0,0,0,0,RESTAURACAO_SEM_OUTRO_EVENTO_OBSERVADO
7-EN-37D-RJS,f31b3955f906ca3a04c4bf4d33635afc50150218bd968e52bc1a306e8495d11e,RESTAURAÇÃO,2025-04-11,2025-04-17,2025-01-01,2025-03-31,2025-05-01,2025-07-31,1,1,0,0,RESTAURACAO_COM_OUTROS_EVENTOS
7-PM-35DP-RJS,93628f7ecfc84fd7be3d4d703d75026dfc32c2994e5feecebd873f9178009bdf,RESTAURAÇÃO,2025-04-24,2025-05-12,2025-01-01,2025-03-31,2025-06-01,2025-08-31,0,0,0,0,RESTAURACAO_SEM_OUTRO_EVENTO_OBSERVADO
7-CRP-57D-RJS,c3c660d983fff70942fda288cf8866d754738ed038a9b907ba4d8c095e3b681a,RESTAURAÇÃO,2025-04-27,2025-05-16,2025-01-01,2025-03-31,2025-06-01,2025-08-31,0,0,0,0,RESTAURACAO_SEM_OUTRO_EVENTO_OBSERVADO
7-VM-49D-RJS,3ed64df466eec7f7c3e7257c7a2808e4cf175fdb8737a6a426a4a2ed71e20a81,RESTAURAÇÃO,2025-05-01,2025-05-10,2025-02-01,2025-04-30,2025-06-01,2025-08-31,1,0,0,1,RESTAURACAO_COM_OUTROS_EVENTOS
7-TBMT-10H-RJS,4e4aedc4e37e3d972f2fc8e8cc63077606967569b67869bc958c33639b5886db,RESTAURAÇÃO,2025-05-02,2025-05-04,2025-02-01,2025-04-30,2025-06-01,2025-08-31,1,0,0,1,RESTAURACAO_COM_OUTROS_EVENTOS
7-PRG-73HP-RJS,c82c8ddbc3dd7402d1edc67b1dc48a646c2e95e3413ae40b865f8f4af394596e,RESTAURAÇÃO,2025-05-04,2025-05-10,2025-02-01,2025-04-30,2025-06-01,2025-08-31,0,0,0,0,RESTAURACAO_SEM_OUTRO_EVENTO_OBSERVADO
7-PRG-81H-RJS,c5b9e87ec0d99ba09aa11545988142c8b7d3b49da1234fc745238b8acd65503d,RESTAURAÇÃO,2025-05-11,2025-05-20,2025-02-01,2025-04-30,2025-06-01,2025-08-31,0,0,0,0,RESTAURACAO_SEM_OUTRO_EVENTO_OBSERVADO
7-VM-46D-RJS,548bfff2ee82ec2ad6356d7a86e1b6a92c4cfe8929cbce8bf0f5acb2f1c3beb2,RESTAURAÇÃO,2025-05-15,2025-06-24,2025-02-01,2025-04-30,2025-07-01,2025-09-30,0,0,0,0,RESTAURACAO_SEM_OUTRO_EVENTO_OBSERVADO


In [0]:
from pyspark.sql import functions as F

PREVIOUS_VIEW = (
    "workspace.oil_gas_gold.vw_contexto_intervencoes_cobertas_piloto"
)
REVISED_VIEW = (
    "workspace.oil_gas_gold.vw_contexto_intervencoes_revisado_piloto"
)

replaced_columns = {
    "outros_eventos_observados",
    "outros_eventos_ids",
    "status_contexto",
    "grupo_analise",
}

preserved_columns_sql = ",\n".join(
    f"a.`{column}`"
    for column in spark.table(PREVIOUS_VIEW).columns
    if column not in replaced_columns
)

revised_sql = f"""
WITH outros_eventos AS (
    SELECT
        f.fato_id,
        f.poco_id,
        inicio.data AS data_inicio,
        termino.data AS data_termino
    FROM workspace.oil_gas_gold.vw_intervencoes_piloto f
    JOIN workspace.oil_gas_gold.dim_data inicio
      ON f.data_inicio_id = inicio.data_id
    JOIN workspace.oil_gas_gold.dim_data termino
      ON f.data_termino_id = termino.data_id
    WHERE f.poco_id <> 'SEM_ID_ANP'
),

revisao AS (
    SELECT
        a.fato_id,
        COUNT(DISTINCT b.fato_id) AS outros_eventos_observados,
        sort_array(
            collect_set(b.fato_id)
        ) AS outros_eventos_ids,

        COUNT(DISTINCT CASE
            WHEN b.fato_id IS NOT NULL
             AND NOT (
                (
                    b.data_inicio <= a.janela_antes_fim
                    AND b.data_termino >= a.janela_antes_inicio
                )
                OR (
                    b.data_inicio <= a.data_termino
                    AND b.data_termino >= a.data_inicio
                )
                OR (
                    b.data_inicio <= a.janela_depois_fim
                    AND b.data_termino >= a.janela_depois_inicio
                )
             )
            THEN b.fato_id
        END) AS outros_eventos_adicionais_nos_trechos_excluidos

    FROM {PREVIOUS_VIEW} a
    LEFT JOIN outros_eventos b
      ON a.poco_id = b.poco_id
     AND a.fato_id <> b.fato_id
     AND b.data_inicio <= a.janela_depois_fim
     AND b.data_termino >= a.janela_antes_inicio
    GROUP BY a.fato_id
)

SELECT
    {preserved_columns_sql},
    a.grupo_analise AS grupo_analise_anterior,
    a.outros_eventos_observados AS outros_eventos_criterio_anterior,
    r.outros_eventos_observados,
    r.outros_eventos_ids,
    r.outros_eventos_adicionais_nos_trechos_excluidos,
    CASE
        WHEN r.outros_eventos_observados = 0
            THEN 'SEM_OUTRA_INTERVENCAO_OBSERVADA'
        ELSE 'COM_OUTRA_INTERVENCAO_OBSERVADA'
    END AS status_contexto,
    CASE
        WHEN a.objetivo <> 'RESTAURAÇÃO'
            THEN 'OUTRO_OBJETIVO'
        WHEN r.outros_eventos_observados > 0
            THEN 'RESTAURACAO_COM_OUTROS_EVENTOS'
        ELSE 'RESTAURACAO_SEM_OUTRO_EVENTO_OBSERVADO'
    END AS grupo_analise
FROM {PREVIOUS_VIEW} a
JOIN revisao r ON a.fato_id = r.fato_id
"""

candidate = spark.sql(revised_sql)
expected_count = spark.table(PREVIOUS_VIEW).count()

summary = candidate.agg(
    F.count("*").alias("registros"),
    F.countDistinct("fato_id").alias("eventos_distintos"),
).first()

if any(summary[c] != expected_count for c in summary.asDict()):
    raise RuntimeError("A revisão perdeu ou multiplicou eventos.")

invalid = (
    (
        F.col("outros_eventos_observados")
        < F.col("outros_eventos_criterio_anterior")
    )
    | (
        F.col("outros_eventos_observados")
        != (
            F.col("outros_eventos_criterio_anterior")
            + F.col("outros_eventos_adicionais_nos_trechos_excluidos")
        )
    )
    | (
        F.size("outros_eventos_ids")
        != F.col("outros_eventos_observados")
    )
)

if candidate.filter(invalid).limit(1).count():
    raise RuntimeError("Contagens inconsistentes na revisão.")

spark.sql(f"""
    CREATE OR REPLACE VIEW {REVISED_VIEW}
    COMMENT 'Contexto revisado: busca outros eventos em todo o intervalo
    entre o início da janela anterior e o fim da posterior.
    Inclui os meses excluídos da comparação de produção.
    Não comprova isolamento nem efeito causal.'
    AS
    {revised_sql}
""")

print(f"View revisada: {REVISED_VIEW}")
print(f"Eventos preservados: {expected_count} | PASS")

print("\\nGRUPOS REVISADOS:")
display(
    candidate.groupBy("objetivo", "grupo_analise")
    .count()
    .orderBy("objetivo", "grupo_analise")
)

print("\\nEVENTOS COM CONTEXTO ALTERADO:")
display(
    candidate.alias("e")
    .filter(
        F.col("outros_eventos_observados")
        > F.col("outros_eventos_criterio_anterior")
    )
    .join(
        spark.table("workspace.oil_gas_gold.dim_poco").alias("p"),
        F.col("e.poco_id") == F.col("p.poco_id"),
        "left",
    )
    .select(
        F.col("p.nome_poco_anp"),
        F.col("e.fato_id"),
        F.col("e.data_inicio"),
        F.col("e.data_termino"),
        F.col("e.outros_eventos_criterio_anterior"),
        F.col("e.outros_eventos_adicionais_nos_trechos_excluidos"),
        F.col("e.outros_eventos_observados"),
        F.col("e.grupo_analise_anterior"),
        F.col("e.grupo_analise"),
    )
    .orderBy("data_inicio", "fato_id")
)

View revisada: workspace.oil_gas_gold.vw_contexto_intervencoes_revisado_piloto
Eventos preservados: 26 | PASS
\nGRUPOS REVISADOS:


objetivo,grupo_analise,count
ABANDONO,OUTRO_OBJETIVO,1
RESTAURAÇÃO,RESTAURACAO_COM_OUTROS_EVENTOS,7
RESTAURAÇÃO,RESTAURACAO_SEM_OUTRO_EVENTO_OBSERVADO,18


\nEVENTOS COM CONTEXTO ALTERADO:


nome_poco_anp,fato_id,data_inicio,data_termino,outros_eventos_criterio_anterior,outros_eventos_adicionais_nos_trechos_excluidos,outros_eventos_observados,grupo_analise_anterior,grupo_analise
7-VM-39D-RJS,f0547a3bbd9d723046e4fadbb0cad9bde28d4646bf13f84e5d8e2a74719d93e5,2025-05-28,2025-06-02,0,1,1,RESTAURACAO_SEM_OUTRO_EVENTO_OBSERVADO,RESTAURACAO_COM_OUTROS_EVENTOS
7-TBMT-10H-RJS,a2d56346225334cab27323f7c5e5fed02be8d2293c97b4b53308a10bbda39723,2025-05-29,2025-06-09,0,1,1,RESTAURACAO_SEM_OUTRO_EVENTO_OBSERVADO,RESTAURACAO_COM_OUTROS_EVENTOS


In [0]:
from pyspark.sql import functions as F

CONTEXT_VIEW = (
    "workspace.oil_gas_gold.vw_contexto_intervencoes_revisado_piloto"
)
DETAIL_VIEW = (
    "workspace.oil_gas_gold.vw_oleo_janelas_intervencoes_piloto"
)
COMPARISON_VIEW = (
    "workspace.oil_gas_gold.vw_comparacao_oleo_intervencoes_piloto"
)

detail_sql = f"""
WITH poco_mes AS (
    SELECT
        f.escopo_id,
        f.poco_id,
        d.primeiro_dia_mes AS competencia,
        SUM(f.producao_de_oleo_m3) AS oleo_reportado_m3,
        COUNT(f.producao_de_oleo_m3) AS registros_com_oleo,
        SUM(CASE
            WHEN f.perfil_medidas IN (
                'SOMENTE_PRODUCAO', 'PRODUCAO_E_INJECAO'
            ) THEN 1 ELSE 0
        END) AS registros_com_medidas_producao,
        COUNT(DISTINCT f._source_dataset) AS fontes_distintas
    FROM workspace.oil_gas_gold.vw_producao_piloto f
    JOIN workspace.oil_gas_gold.dim_data d
      ON f.data_id = d.data_id
    WHERE f.poco_id <> 'SEM_ID_ANP'
    GROUP BY f.escopo_id, f.poco_id, d.primeiro_dia_mes
),

janelas AS (
    SELECT
        escopo_id,
        fato_id,
        poco_id,
        'ANTES' AS fase,
        explode(sequence(
            janela_antes_inicio,
            trunc(janela_antes_fim, 'MONTH'),
            INTERVAL 1 MONTH
        )) AS competencia
    FROM {CONTEXT_VIEW}

    UNION ALL

    SELECT
        escopo_id,
        fato_id,
        poco_id,
        'DEPOIS' AS fase,
        explode(sequence(
            janela_depois_inicio,
            trunc(janela_depois_fim, 'MONTH'),
            INTERVAL 1 MONTH
        )) AS competencia
    FROM {CONTEXT_VIEW}
)

SELECT
    j.*,
    day(last_day(j.competencia)) AS dias_calendario_mes,
    p.oleo_reportado_m3,
    p.registros_com_oleo,
    p.registros_com_medidas_producao,
    p.fontes_distintas
FROM janelas j
LEFT JOIN poco_mes p
  ON j.escopo_id = p.escopo_id
 AND j.poco_id = p.poco_id
 AND j.competencia = p.competencia
"""

detail = spark.sql(detail_sql)
expected_events = spark.table(CONTEXT_VIEW).count()

if detail.count() != expected_events * 6:
    raise RuntimeError("Esperadas seis observações mensais por evento.")

if (
    detail.groupBy("fato_id", "fase", "competencia")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Observações mensais repetidas para o mesmo evento.")

invalid_detail = detail.filter(
    F.col("oleo_reportado_m3").isNull()
    | (F.col("oleo_reportado_m3") < 0)
    | F.col("fontes_distintas").isNull()
    | (F.col("fontes_distintas") != 1)
    | F.col("registros_com_medidas_producao").isNull()
    | (F.col("registros_com_medidas_producao") <= 0)
    | (
        F.col("registros_com_oleo")
        != F.col("registros_com_medidas_producao")
    )
)

if invalid_detail.limit(1).count():
    display(invalid_detail.limit(10))
    raise RuntimeError(
        "Há meses sem óleo, com preenchimento parcial "
        "ou com mais de uma fonte. Investigar antes da comparação."
    )

spark.sql(f"""
    CREATE OR REPLACE VIEW {DETAIL_VIEW}
    COMMENT 'Óleo reportado por evento, fase e mês.
    Agregação dos contextos de campo e instalação da fonte.
    Um mesmo mês de um poço pode aparecer em janelas de eventos diferentes.'
    AS
    {detail_sql}
""")

comparison_sql = f"""
WITH totais AS (
    SELECT
        escopo_id,
        fato_id,

        SUM(CASE WHEN fase = 'ANTES'
            THEN oleo_reportado_m3 END) AS oleo_antes_m3,

        SUM(CASE WHEN fase = 'DEPOIS'
            THEN oleo_reportado_m3 END) AS oleo_depois_m3,

        SUM(CASE WHEN fase = 'ANTES'
            THEN dias_calendario_mes END) AS dias_antes,

        SUM(CASE WHEN fase = 'DEPOIS'
            THEN dias_calendario_mes END) AS dias_depois,

        SUM(CASE WHEN fase = 'ANTES'
            THEN 1 ELSE 0 END) AS meses_antes,

        SUM(CASE WHEN fase = 'DEPOIS'
            THEN 1 ELSE 0 END) AS meses_depois

    FROM {DETAIL_VIEW}
    GROUP BY escopo_id, fato_id
),

taxas AS (
    SELECT
        *,
        CAST(
            CAST(oleo_antes_m3 AS DECIMAL(28,9))
            / NULLIF(dias_antes, 0)
            AS DECIMAL(28,9)
        ) AS taxa_antes_m3_dia_calendario,

        CAST(
            CAST(oleo_depois_m3 AS DECIMAL(28,9))
            / NULLIF(dias_depois, 0)
            AS DECIMAL(28,9)
        ) AS taxa_depois_m3_dia_calendario
    FROM totais
),

variacoes AS (
    SELECT
        *,
        taxa_depois_m3_dia_calendario
            - taxa_antes_m3_dia_calendario
            AS diferenca_taxa_m3_dia_calendario,

        CASE WHEN taxa_antes_m3_dia_calendario > 0 THEN
            CAST(
                100.0 * (
                    taxa_depois_m3_dia_calendario
                    - taxa_antes_m3_dia_calendario
                ) / taxa_antes_m3_dia_calendario
                AS DECIMAL(28,6)
            )
        END AS variacao_percentual
    FROM taxas
)

SELECT
    e.*,
    v.oleo_antes_m3,
    v.oleo_depois_m3,
    v.dias_antes,
    v.dias_depois,
    v.meses_antes,
    v.meses_depois,
    v.taxa_antes_m3_dia_calendario,
    v.taxa_depois_m3_dia_calendario,
    v.diferenca_taxa_m3_dia_calendario,
    v.variacao_percentual,

    CASE
        WHEN v.oleo_antes_m3 = 0 AND v.oleo_depois_m3 = 0
            THEN 'ZERO_NAS_DUAS_JANELAS'
        WHEN v.oleo_antes_m3 = 0 AND v.oleo_depois_m3 > 0
            THEN 'DE_ZERO_PARA_POSITIVO'
        WHEN v.oleo_antes_m3 > 0 AND v.oleo_depois_m3 = 0
            THEN 'DE_POSITIVO_PARA_ZERO'
        WHEN v.diferenca_taxa_m3_dia_calendario > 0
            THEN 'TAXA_POSTERIOR_MAIOR'
        WHEN v.diferenca_taxa_m3_dia_calendario < 0
            THEN 'TAXA_POSTERIOR_MENOR'
        ELSE 'TAXAS_IGUAIS_NA_PRECISAO_ADOTADA'
    END AS comportamento_observado

FROM {CONTEXT_VIEW} e
JOIN variacoes v
  ON e.escopo_id = v.escopo_id
 AND e.fato_id = v.fato_id
"""

comparison = spark.sql(comparison_sql)

summary = comparison.agg(
    F.count("*").alias("registros"),
    F.countDistinct("fato_id").alias("eventos_distintos"),
).first()

if any(summary[c] != expected_events for c in summary.asDict()):
    raise RuntimeError("A comparação perdeu ou multiplicou eventos.")

if comparison.filter(
    (F.col("meses_antes") != 3)
    | (F.col("meses_depois") != 3)
    | F.col("taxa_antes_m3_dia_calendario").isNull()
    | F.col("taxa_depois_m3_dia_calendario").isNull()
    | (
        (F.col("oleo_antes_m3") == 0)
        & F.col("variacao_percentual").isNotNull()
    )
).limit(1).count():
    raise RuntimeError("Indicadores de janela inconsistentes.")

spark.sql(f"""
    CREATE OR REPLACE VIEW {COMPARISON_VIEW}
    COMMENT 'Comparação descritiva de óleo reportado antes/depois.
    Taxas por dia calendário, sem ajuste por dias efetivamente operados.
    Percentual ausente quando a base anterior é zero.
    Não estima efeito causal, produção recuperada ou perda evitada.'
    AS
    {comparison_sql}
""")

print(f"Detalhamento mensal: {DETAIL_VIEW}")
print(f"Comparação publicada: {COMPARISON_VIEW}")
print(f"Eventos comparados: {expected_events} | PASS")

print("\\nRESUMO POR GRUPO:")
display(
    comparison
    .groupBy("grupo_analise", "comportamento_observado")
    .count()
    .orderBy("grupo_analise", "comportamento_observado")
)

print("\\nCOMPARAÇÃO POR EVENTO:")
display(
    comparison.alias("e")
    .join(
        spark.table("workspace.oil_gas_gold.dim_poco").alias("p"),
        F.col("e.poco_id") == F.col("p.poco_id"),
        "left",
    )
    .select(
        F.col("p.nome_poco_anp"),
        F.col("e.fato_id"),
        F.col("e.grupo_analise"),
        F.col("e.oleo_antes_m3"),
        F.col("e.oleo_depois_m3"),
        F.col("e.dias_antes"),
        F.col("e.dias_depois"),
        F.col("e.taxa_antes_m3_dia_calendario"),
        F.col("e.taxa_depois_m3_dia_calendario"),
        F.col("e.diferenca_taxa_m3_dia_calendario"),
        F.col("e.variacao_percentual"),
        F.col("e.comportamento_observado"),
    )
    .orderBy("grupo_analise", "nome_poco_anp", "fato_id")
)

Detalhamento mensal: workspace.oil_gas_gold.vw_oleo_janelas_intervencoes_piloto
Comparação publicada: workspace.oil_gas_gold.vw_comparacao_oleo_intervencoes_piloto
Eventos comparados: 26 | PASS
\nRESUMO POR GRUPO:


grupo_analise,comportamento_observado,count
OUTRO_OBJETIVO,ZERO_NAS_DUAS_JANELAS,1
RESTAURACAO_COM_OUTROS_EVENTOS,DE_POSITIVO_PARA_ZERO,1
RESTAURACAO_COM_OUTROS_EVENTOS,DE_ZERO_PARA_POSITIVO,4
RESTAURACAO_COM_OUTROS_EVENTOS,TAXA_POSTERIOR_MENOR,1
RESTAURACAO_COM_OUTROS_EVENTOS,ZERO_NAS_DUAS_JANELAS,1
RESTAURACAO_SEM_OUTRO_EVENTO_OBSERVADO,DE_POSITIVO_PARA_ZERO,1
RESTAURACAO_SEM_OUTRO_EVENTO_OBSERVADO,DE_ZERO_PARA_POSITIVO,11
RESTAURACAO_SEM_OUTRO_EVENTO_OBSERVADO,TAXA_POSTERIOR_MAIOR,4
RESTAURACAO_SEM_OUTRO_EVENTO_OBSERVADO,TAXA_POSTERIOR_MENOR,1
RESTAURACAO_SEM_OUTRO_EVENTO_OBSERVADO,ZERO_NAS_DUAS_JANELAS,1


\nCOMPARAÇÃO POR EVENTO:


nome_poco_anp,fato_id,grupo_analise,oleo_antes_m3,oleo_depois_m3,dias_antes,dias_depois,taxa_antes_m3_dia_calendario,taxa_depois_m3_dia_calendario,diferenca_taxa_m3_dia_calendario,variacao_percentual,comportamento_observado
7-PM-43HA-RJS,b4d06b57d6e6386a56c6944e260d4781a4c66fb5f89734b898bbef56d703cb1b,OUTRO_OBJETIVO,0E-9,0E-9,92,91,0E-9,0E-9,0E-9,null,ZERO_NAS_DUAS_JANELAS
7-EN-37D-RJS,f31b3955f906ca3a04c4bf4d33635afc50150218bd968e52bc1a306e8495d11e,RESTAURACAO_COM_OUTROS_EVENTOS,0E-9,0E-9,90,92,0E-9,0E-9,0E-9,null,ZERO_NAS_DUAS_JANELAS
7-PM-9D-RJS,8c52a90503549a2514a24937de496f0bd6a672e48d43a241284a9467a85dd6ed,RESTAURACAO_COM_OUTROS_EVENTOS,5110.893000000,1806.517000000,92,92,55.553184783,19.636054348,-35.917130435,-64.653594,TAXA_POSTERIOR_MENOR
7-TBMT-10H-RJS,4e4aedc4e37e3d972f2fc8e8cc63077606967569b67869bc958c33639b5886db,RESTAURACAO_COM_OUTROS_EVENTOS,0E-9,32854.269000000,89,92,0E-9,357.111619565,357.111619565,null,DE_ZERO_PARA_POSITIVO
7-TBMT-10H-RJS,a2d56346225334cab27323f7c5e5fed02be8d2293c97b4b53308a10bbda39723,RESTAURACAO_COM_OUTROS_EVENTOS,0E-9,37002.663000000,89,92,0E-9,402.202858696,402.202858696,null,DE_ZERO_PARA_POSITIVO
7-VM-39D-RJS,546517a29e2aa7eef1f265bfdc3b94d203be9a1a00ced0f8109e88dfbf164463,RESTAURACAO_COM_OUTROS_EVENTOS,0E-9,3238.267000000,89,92,0E-9,35.198554348,35.198554348,null,DE_ZERO_PARA_POSITIVO
7-VM-39D-RJS,f0547a3bbd9d723046e4fadbb0cad9bde28d4646bf13f84e5d8e2a74719d93e5,RESTAURACAO_COM_OUTROS_EVENTOS,0E-9,4030.045000000,89,92,0E-9,43.804836957,43.804836957,null,DE_ZERO_PARA_POSITIVO
7-VM-49D-RJS,3ed64df466eec7f7c3e7257c7a2808e4cf175fdb8737a6a426a4a2ed71e20a81,RESTAURACAO_COM_OUTROS_EVENTOS,3527.280000000,0E-9,89,92,39.632359551,0E-9,-39.632359551,-100.000000,DE_POSITIVO_PARA_ZERO
7-CRP-12D-RJS,58c367383c1f174b979c35903b42e8da2ddac854b586e5e517a44566742bbb24,RESTAURACAO_SEM_OUTRO_EVENTO_OBSERVADO,0E-9,4466.202000000,91,92,0E-9,48.545673913,48.545673913,null,DE_ZERO_PARA_POSITIVO
7-CRP-18D-RJS,354f550b07ecde7224aa5609aec941e9b69968eb32c89edea4b93d0afe043acd,RESTAURACAO_SEM_OUTRO_EVENTO_OBSERVADO,0E-9,2473.235000000,91,91,0E-9,27.178406593,27.178406593,null,DE_ZERO_PARA_POSITIVO


In [0]:
import re
import uuid
import hashlib
import requests

from html.parser import HTMLParser
from pathlib import PurePosixPath
from urllib.parse import urljoin, urlsplit, unquote
from pyspark.sql import functions as F

ANP_SOURCE_URL = (
    "https://www.gov.br/anp/pt-br/centrais-de-conteudo/"
    "dados-abertos/fase-de-desenvolvimento-e-producao"
)
TARGET_YEARS = {2024, 2025, 2026}
DISCOVERY_RUN_ID = str(uuid.uuid4())
INVENTORY_TABLE = "workspace.oil_gas_ops.source_inventory_runs"

class AnchorCollector(HTMLParser):
    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.hrefs = []

    def handle_starttag(self, tag, attrs):
        if tag == "a":
            href = dict(attrs).get("href")
            if href:
                self.hrefs.append(href)

response = requests.get(ANP_SOURCE_URL, timeout=(10, 30))
response.raise_for_status()

if urlsplit(response.url).hostname != "www.gov.br":
    raise RuntimeError("Redirecionamento inesperado.")

if "text/html" not in response.headers.get("Content-Type", "").lower():
    raise RuntimeError("A página não retornou HTML.")

page_hash = hashlib.sha256(response.content).hexdigest()

parser = AnchorCollector()
parser.feed(response.text)

records = []
seen_urls = set()

for href in parser.hrefs:
    absolute_url = urljoin(response.url, href)
    parsed = urlsplit(absolute_url)
    path = unquote(parsed.path).lower()

    if (
        parsed.scheme not in {"http", "https"}
        or parsed.hostname != "www.gov.br"
        or not path.endswith(".csv")
    ):
        continue

    if "/arquivos-fase-de-desenvolvimento-e-producao/" not in path:
        continue

    if (
        "/arquivos-producao-de-petroleo-e-gas-natural-nacional/pm/"
        in path
    ):
        dataset = "producao_mar"

    elif (
        "/arquivos-producao-de-petroleo-e-gas-natural-nacional/pt/"
        in path
    ):
        dataset = "producao_terra"

    elif (
        "/informacoes-sobre-pocos/" in path
        and "intervencao" in path
    ):
        dataset = "intervencoes"

    else:
        continue

    years = sorted(set(
        int(value)
        for value in re.findall(r"(?<!\d)(20\d{2})(?!\d)", path)
    ))

    if not TARGET_YEARS.intersection(years):
        continue

    if absolute_url in seen_urls:
        continue
    seen_urls.add(absolute_url)

    filename = PurePosixPath(unquote(parsed.path)).name

    quarter = None
    if dataset == "producao_terra":
        match = re.search(
            r"(?:[_-]([1-4])[_-]trim|trim[_-]([1-4]))(?:[._-]|$)",
            filename.lower(),
        )

        if not match:
            raise RuntimeError(
                f"Trimestre não identificado no endereço: {filename}"
            )

        quarter = int(match.group(1) or match.group(2))

    records.append((
        hashlib.sha256(absolute_url.encode("utf-8")).hexdigest(),
        dataset,
        absolute_url,
        filename,
        years,
        quarter,
    ))

if not records:
    raise RuntimeError("Nenhum arquivo candidato encontrado.")

inventory = (
    spark.createDataFrame(
        records,
        schema="""
            source_file_id STRING,
            dataset STRING,
            source_url STRING,
            nome_arquivo STRING,
            anos_mencionados_no_endereco ARRAY<INT>,
            trimestre_indicado INT
        """,
    )
    .withColumn("discovery_run_id", F.lit(DISCOVERY_RUN_ID))
    .withColumn("descoberto_em", F.current_timestamp())
    .withColumn("pagina_sha256", F.lit(page_hash))
)

(
    inventory.write
    .format("delta")
    .mode("append")
    .saveAsTable(INVENTORY_TABLE)
)

previous_urls = (
    spark.table("workspace.oil_gas_ops.ingestion_files")
    .filter(F.col("status").isin("STORED", "REUSED"))
    .select("source_url")
    .distinct()
    .withColumn("url_ja_coletada", F.lit(True))
)

expansion_inventory = (
    inventory
    .join(previous_urls, "source_url", "left")
    .fillna(False, subset=["url_ja_coletada"])
)

print(f"Execução de descoberta: {DISCOVERY_RUN_ID}")
print("Os anos e trimestres abaixo são indícios do endereço.")
print("A cobertura real será conferida dentro de cada arquivo.")

display(
    expansion_inventory
    .select(
        "dataset",
        "nome_arquivo",
        "anos_mencionados_no_endereco",
        "trimestre_indicado",
        "url_ja_coletada",
        "source_url",
    )
    .orderBy(
        "dataset",
        "anos_mencionados_no_endereco",
        "trimestre_indicado",
        "nome_arquivo",
    )
)

display(
    expansion_inventory
    .groupBy("dataset", "url_ja_coletada")
    .count()
    .orderBy("dataset", "url_ja_coletada")
)

Execução de descoberta: b9383b3a-e9cb-43b8-a489-f781bd7f1c2f
Os anos e trimestres abaixo são indícios do endereço.
A cobertura real será conferida dentro de cada arquivo.


dataset,nome_arquivo,anos_mencionados_no_endereco,trimestre_indicado,url_ja_coletada,source_url
intervencoes,intervencao_em_pocos_2023-2024.csv,"List(2023, 2024)",null,false,https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/arquivos-fase-de-desenvolvimento-e-producao/informacoes-sobre-pocos/intervencao_em_pocos_2023-2024.csv
intervencoes,intervencao_em_pocos_2025-2026.csv,"List(2025, 2026)",null,true,https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/arquivos-fase-de-desenvolvimento-e-producao/informacoes-sobre-pocos/intervencao_em_pocos_2025-2026.csv
producao_mar,producao_por_poco_2024.csv,List(2024),null,false,https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/arquivos-fase-de-desenvolvimento-e-producao/arquivos-producao-de-petroleo-e-gas-natural-nacional/pm/producao_por_poco_2024.csv
producao_mar,producao-mar-2025.csv,List(2025),null,true,https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/arquivos-fase-de-desenvolvimento-e-producao/arquivos-producao-de-petroleo-e-gas-natural-nacional/pm/producao-mar-2025.csv
producao_mar,producao-mar-2026.csv,List(2026),null,false,https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/arquivos-fase-de-desenvolvimento-e-producao/arquivos-producao-de-petroleo-e-gas-natural-nacional/pm/producao-mar-2026.csv
producao_terra,producao-por-poco-terra-trim-1.csv,List(2024),1,false,https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/arquivos-fase-de-desenvolvimento-e-producao/arquivos-producao-de-petroleo-e-gas-natural-nacional/pt/2024/producao-por-poco-terra-trim-1.csv
producao_terra,producao-por-poco-terra-trim-2.csv,List(2024),2,false,https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/arquivos-fase-de-desenvolvimento-e-producao/arquivos-producao-de-petroleo-e-gas-natural-nacional/pt/2024/producao-por-poco-terra-trim-2.csv
producao_terra,producao-por-poco-terra-trim-3.csv,List(2024),3,false,https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/arquivos-fase-de-desenvolvimento-e-producao/arquivos-producao-de-petroleo-e-gas-natural-nacional/pt/2024/producao-por-poco-terra-trim-3.csv
producao_terra,producao_por_poco_terra_trim_4.csv,List(2024),4,false,https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/arquivos-fase-de-desenvolvimento-e-producao/arquivos-producao-de-petroleo-e-gas-natural-nacional/pt/2024/producao_por_poco_terra_trim_4.csv
producao_terra,producao_por_poco_terra_2025_1_trim.csv,List(2025),1,true,https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/arquivos-fase-de-desenvolvimento-e-producao/arquivos-producao-de-petroleo-e-gas-natural-nacional/pt/2025/producao_por_poco_terra_2025_1_trim.csv


dataset,url_ja_coletada,count
intervencoes,false,1
intervencoes,true,1
producao_mar,false,2
producao_mar,true,1
producao_terra,false,9
producao_terra,true,1
